# Platelet R2 — VALIDATED integrated reproducibility notebook v3

**49 cells.** Direct Google Colab execution: **Runtime → Run all**.

**Correctness fixes:** seasonal naïve source day (target `d-7`), no future-test demand
in the exploratory core inventory initialization, isolated per-run outputs,
full-precision S8 mean. The separate manuscript Table 3 / Table 4 inventory
engine, SARIMA information-matched rolling comparison, statistical tests and
figures are retained.

**Data:** `/content/drive/MyDrive/Platelets/platelet_data_english_260529.xlsx`.

**Output:** `/content/drive/MyDrive/Platelets/Reproducibility_Results/R2_VALIDATED_V3_RUNS/<run_id>/`

**DO NOT overwrite** the old `Platelet_Manuscript_Reproducibility.ipynb` or confuse
`analysis_r2_s18s19/11/12_test_inventory_*` (exploratory core simulator) with
manuscript Table 3/4 CSVs. Chronos is disabled by default and must be run
separately to regenerate S7.


## 0. Setup

In [ ]:
# Install required libraries for the full pipeline and S1.8/S1.9 comparisons.
# Colab already provides numpy/pandas/scikit-learn on most runtimes.
%pip install -q "lightgbm>=4.6,<5" "statsmodels>=0.14.6,<0.16" "scikit-learn>=1.3" "scipy>=1.11" openpyxl python-docx
print("Dependencies installed / available.")

In [ ]:
import os, sys, json, math, importlib.util, pathlib, uuid
from datetime import datetime, timezone

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
RUN_ID = os.environ.get("PLATELET_R2_RUN_ID") or (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
WORKDIR = (pathlib.Path("/content/platelet_r2_validated_runs") / RUN_ID
           if IN_COLAB else pathlib.Path.cwd() / "platelet_r2_validated_runs" / RUN_ID)
if WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise FileExistsError(f"Refusing to reuse a nonempty output directory: {WORKDIR}")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print("R2 run ID:", RUN_ID)
print("Working directory:", WORKDIR.resolve())


### Upload the raw data file

Upload `platelet_data_english_260529.xlsx`. If it is already sitting in the working
directory from a previous run of this notebook, this cell will just reuse it.


In [ ]:
DATA_FILENAME = "platelet_data_english_260529.xlsx"
from pathlib import Path
import shutil
if not Path(DATA_FILENAME).exists():
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_path = Path('/content/drive/MyDrive/Platelets') / DATA_FILENAME
        if drive_path.exists():
            shutil.copy2(drive_path, Path(DATA_FILENAME))
            print('Copied dataset from Google Drive:', drive_path)
        else:
            from google.colab import files
            print(f'Please upload {DATA_FILENAME}')
            uploaded = files.upload()
            if DATA_FILENAME not in uploaded:
                options = [n for n in uploaded if n.lower().endswith('.xlsx')]
                if options:
                    Path(options[0]).rename(DATA_FILENAME)
    else:
        raise FileNotFoundError(
            f'Place {DATA_FILENAME} in {WORKDIR.resolve()} and re-run this cell.'
        )
assert Path(DATA_FILENAME).exists(), f'Data file not found: {DATA_FILENAME}'
print('Input data:', Path(DATA_FILENAME).resolve())


## 1. Complete analysis pipeline embedded in this notebook

The next cell writes `platelet_analysis_v2.py` **inside the Colab workspace**, preserving the original full workflow and the minimal 181-day predicted-date boundary correction. LightGBM parameters are aligned with S1.8. SARIMA is run independently in Section 2 as a documented benchmark, rather than introduced into the Table 4 candidate-selection grid.

You do not need to download or upload a separate Python script.

In [ ]:
%%writefile platelet_analysis_v2.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
platelet_sparse_hurdle_analysis_v2.py

Sparse-aware platelet demand forecasting, inventory-policy simulation, and
manuscript figure/table generation.

This is the consolidated, GitHub-ready v2 pipeline for the platelet-demand
forecasting revision. It merges and cleans up two Colab notebook exports:

  - platelet_colab_sparse_hurdle_plus_models.py
      Sparse/intermittent diagnostics, hurdle/two-stage models, LightGBM,
      Prophet, LSTM, validation-based model selection, and inventory-policy
      simulation (Part 1), plus the manuscript Figure 1/2 and Table 1/2/4/5
      generation code that consumes Part 1's outputs (Part 2).
  - platelet_figures_20260702.py
      An earlier, self-contained draft pipeline (simpler LASSO/Ridge/SARIMA
      models with its own figures/tables). Its methodology was superseded by
      the sparse-hurdle pipeline above, so it is not reproduced here; only
      the current (Part 1 + Part 2) methodology is kept.

What changed relative to the raw Colab exports
------------------------------------------------
  - All Google Colab-only code (drive.mount, "!pip install ...", display())
    was removed. The script now runs as a plain, local/CLI Python program.
  - Duplicate function definitions introduced by re-pasted notebook cells
    (e.g. two identical copies of safe_filename(), two near-identical copies
    of ensure_dir()) were de-duplicated, keeping the version that actually
    executed last in the notebook.
  - Two different constants were both named META_COLS in the original
    notebook (one for the modeling data loader, one for the figure data
    loader). The figure-side constant was renamed to FIGURE_META_COLS to
    avoid a silent collision; no logic was changed.
  - The many repeated "for each model, plot ... and save" notebook cells
    (Figure 2 per model, ABO-trend per model, product-ABO trend per model)
    were consolidated into one function, generate_supplementary_figures(),
    kept as an optional step rather than always executed.
  - A single argparse-driven main() now drives the whole pipeline: run the
    forecasting/inventory analysis, then (optionally) regenerate Figure 1,
    Figure 2, and Tables 1/2/4/5 from its outputs.

Usage
-----
    python platelet_sparse_hurdle_analysis_v2.py --data /path/to/platelet_data_english_260529.xlsx

Optional flags (see `python platelet_sparse_hurdle_analysis_v2.py --help`):
    --output-dir DIR        Directory for the analysis CSV/XLSX outputs.
    --figures-dir DIR       Directory for Figure 1 / Figure 2 PNGs.
    --tables-dir DIR        Directory for the manuscript Excel/Word tables.
    --skip-analysis         Skip Part 1 and reuse existing CSVs in --output-dir.
    --skip-figures          Skip Figure 1 / Figure 2 generation.
    --skip-tables           Skip Table 1/2/4/5 generation.
    --extra-figures         Also generate the supplementary per-model figures
                             (Figure 2, ABO-trend, and product-ABO-trend for
                             every model found in the predictions file).
    --no-lightgbm / --no-prophet / --no-lstm
                             Skip the corresponding optional model family.

Requirements
------------
    numpy, pandas, scipy, scikit-learn, matplotlib, openpyxl
Optional (a model family or export is skipped with a warning if missing):
    lightgbm, prophet, torch, python-docx
"""

from __future__ import annotations

import argparse
import json
import math
import os
import re
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # safe default for headless/CLI runs; ignored if a GUI backend is already active
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Optional models (LightGBM, Prophet, LSTM/PyTorch) and the optional Word
# export (python-docx) are imported lazily inside the functions that need
# them, so the core analysis still runs in a minimal environment.

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


# ============================================================
# PART 1. Sparse-aware forecasting and inventory-policy pipeline
# ============================================================

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------
@dataclass


class SparseHurdleConfig:
    data_path: str = "platelet_data_english_260529.xlsx"
    output_dir: str = "results_sparse_hurdle_plus_models"

    train_start: str = "2019-03-01"
    train_end: str = "2024-02-29"
    val_start: str = "2024-03-01"
    val_end: str = "2024-08-31"
    test_start: str = "2024-09-01"
    test_end: str = "2025-02-28"

    target_prefixes: Tuple[str, str] = ("plt", "aph")
    abo_types: Tuple[str, str, str, str] = ("a", "b", "o", "ab")

    # Sparse/intermittent diagnostics.
    # A target is forced to include hurdle/two-stage candidates when either condition is met.
    sparse_zero_rate_threshold: float = 0.20
    intermittent_adi_threshold: float = 1.32
    intermittent_cv2_threshold: float = 0.49
    min_positive_days_for_hurdle: int = 30

    # Forecasting setup.
    max_horizon: int = 3
    moving_windows: Tuple[int, int] = (7, 14)
    seasonal_period: int = 7
    high_demand_quantile: float = 0.90
    underprediction_weight: float = 3.0
    random_state: int = 42

    # Optional extended models requested for manuscript comparison.
    # Set these to False to skip a model family and reduce run time.
    run_lightgbm: bool = True
    run_prophet: bool = True
    run_lstm: bool = True

    # LightGBM settings.
    lightgbm_n_estimators: int = 300
    lightgbm_learning_rate: float = 0.03
    lightgbm_num_leaves: int = 31
    lightgbm_min_child_samples: int = 20

    # Prophet settings.  Prophet is used as an interpretable time-series model with
    # selected known/lagged regressors. To keep the model stable, only a small number
    # of leakage-safe regressors are used.
    prophet_max_regressors: int = 12
    prophet_include_yearly: bool = True
    prophet_include_weekly: bool = True

    # LSTM settings. The LSTM is treated as a high-capacity benchmark; results should
    # be interpreted cautiously when the demand distribution shifts or sparse targets dominate.
    lstm_sequence_length: int = 14
    lstm_hidden_size: int = 32
    lstm_num_layers: int = 1
    lstm_dropout: float = 0.10
    lstm_max_epochs: int = 30
    lstm_batch_size: int = 32
    lstm_learning_rate: float = 1e-3
    lstm_patience: int = 5

    # Hurdle threshold tuning.
    occurrence_threshold_grid: Tuple[float, ...] = (0.20, 0.30, 0.40, 0.50, 0.60)
    threshold_selection_metric: str = "weighted_mae"  # or "cost_sensitive"

    # Inventory objective.
    shelf_life_days: int = 4
    lead_time_days: int = 1
    initial_days_supply: float = 2.0
    target_days_supply_grid: Tuple[float, ...] = (2.0, 2.5, 3.0, 3.5)
    safety_factor_grid: Tuple[float, ...] = (0.90, 1.00, 1.10, 1.20)
    shortage_cost: float = 10.0
    wastage_cost: float = 1.0
    procurement_cost: float = 0.0


def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def in_window(index: pd.DatetimeIndex, start: str, end: str) -> np.ndarray:
    return (index >= pd.Timestamp(start)) & (index <= pd.Timestamp(end))


# ============================================================
# 2. Data loading
# ============================================================

META_COLS = [
    "Category",
    "Revised variable name",
    "Condition 1 (inpatient/outpatient)",
    "Condition 2 (date)",
    "Description",
    "Condition 3 (detail)",
]


def clean_variable_name(x: object) -> str:
    s = str(x).strip().replace(" ", "_").replace("-", "_")
    s = re.sub(r"[^0-9A-Za-z_]+", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if s.startswith("PLT_"):
        s = "plt_" + s[len("PLT_"):]
    if s.endswith("_O"):
        s = s[:-2] + "_o"
    return s


def load_platelet_excel(xlsx_path: str | Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    xlsx_path = Path(xlsx_path)
    if not xlsx_path.exists():
        raise FileNotFoundError(f"File not found: {xlsx_path}")
    raw = pd.read_excel(xlsx_path, sheet_name=0)
    if raw.shape[1] <= len(META_COLS):
        raise ValueError("Expected metadata columns followed by date columns.")

    raw = raw.rename(columns=dict(zip(raw.columns[: len(META_COLS)], META_COLS)))
    metadata = raw[META_COLS].copy()
    metadata["variable"] = metadata["Revised variable name"].map(clean_variable_name)

    date_cols = list(raw.columns[len(META_COLS):])
    dates = pd.to_datetime(date_cols, errors="coerce")
    if dates.isna().any():
        bad = [str(c) for c, d in zip(date_cols, dates) if pd.isna(d)][:10]
        raise ValueError(f"Could not parse some date columns: {bad}")

    values = raw[date_cols].copy()
    values.index = metadata["variable"].values
    panel = values.T
    panel.index = pd.DatetimeIndex(dates, name="date")
    panel.columns = metadata["variable"].values
    panel = panel.sort_index().apply(pd.to_numeric, errors="coerce").fillna(0.0)

    if panel.columns.duplicated().any():
        panel = panel.T.groupby(level=0).sum().T
        metadata = metadata.drop_duplicates(subset=["variable"], keep="first").reset_index(drop=True)

    return panel, metadata


def get_target_columns(config: SparseHurdleConfig, panel: pd.DataFrame) -> List[str]:
    targets: List[str] = []
    for product in config.target_prefixes:
        for abo in config.abo_types:
            c = f"{product}_transf_{abo}"
            if c in panel.columns:
                targets.append(c)
    if not targets:
        raise ValueError("No target columns found. Expected columns such as plt_transf_a and aph_transf_a.")
    return targets


# ============================================================
# 3. Sparse / intermittent diagnostics
# ============================================================

def average_demand_interval(y: pd.Series) -> float:
    y = pd.Series(y).astype(float)
    positive_positions = np.flatnonzero(y.values > 0)
    if len(positive_positions) <= 1:
        return float("inf")
    intervals = np.diff(positive_positions)
    return float(np.mean(intervals))


def squared_cv_nonzero(y: pd.Series) -> float:
    nz = pd.Series(y).astype(float)
    nz = nz[nz > 0]
    if len(nz) <= 1 or np.isclose(nz.mean(), 0.0):
        return float("inf")
    return float((nz.std(ddof=1) / nz.mean()) ** 2)


def classify_intermittent_demand(adi: float, cv2: float, adi_thr: float = 1.32, cv2_thr: float = 0.49) -> str:
    # Syntetos-Boylan style classification.
    if adi < adi_thr and cv2 < cv2_thr:
        return "smooth"
    if adi >= adi_thr and cv2 < cv2_thr:
        return "intermittent"
    if adi < adi_thr and cv2 >= cv2_thr:
        return "erratic"
    return "lumpy"


def sparse_diagnostics(panel: pd.DataFrame, targets: Sequence[str], config: SparseHurdleConfig) -> pd.DataFrame:
    rows = []
    for t in targets:
        y = panel[t].astype(float)
        zero_rate = float((y == 0).mean())
        adi = average_demand_interval(y)
        cv2 = squared_cv_nonzero(y)
        cls = classify_intermittent_demand(
            adi,
            cv2,
            config.intermittent_adi_threshold,
            config.intermittent_cv2_threshold,
        )
        forced_hurdle = bool(
            zero_rate >= config.sparse_zero_rate_threshold
            or adi >= config.intermittent_adi_threshold
            or cls in {"intermittent", "lumpy"}
        )
        rows.append({
            "target": t,
            "product": "APC" if t.startswith("aph_") else "PC",
            "mean_demand": float(y.mean()),
            "median_demand": float(y.median()),
            "zero_rate": zero_rate,
            "nonzero_days": int((y > 0).sum()),
            "ADI_average_demand_interval": adi,
            "CV2_nonzero_demand": cv2,
            "sbc_classification": cls,
            "force_hurdle_candidates": forced_hurdle,
            "method_recommendation": (
                "Use hurdle/two-stage candidates; general regression alone is unstable"
                if forced_hurdle else
                "General regression acceptable; still compare against adaptive baselines"
            ),
        })
    return pd.DataFrame(rows)


# ============================================================
# 4. Feature engineering
# ============================================================

def make_calendar_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    cal = pd.DataFrame(index=index)
    cal["dow"] = index.dayofweek
    cal["month"] = index.month
    cal["is_weekend"] = (index.dayofweek >= 5).astype(int)
    cal["dow_mon"] = (index.dayofweek == 0).astype(int)
    cal["dow_tue"] = (index.dayofweek == 1).astype(int)
    cal["dow_wed"] = (index.dayofweek == 2).astype(int)
    cal["dow_thu"] = (index.dayofweek == 3).astype(int)
    cal["dow_fri"] = (index.dayofweek == 4).astype(int)
    cal["dow_sat"] = (index.dayofweek == 5).astype(int)
    cal["dow_sun"] = (index.dayofweek == 6).astype(int)
    return cal


def build_feature_frame(panel: pd.DataFrame, targets: Sequence[str], horizon: int, windows: Sequence[int]) -> pd.DataFrame:
    """Create leakage-safe features for direct horizon forecasting.

    Target y(t+h) is predicted at time t.  Calendar is known at target date t+h.
    All observed clinical/product variables are lagged so they are available at time t.
    """
    features = pd.DataFrame(index=panel.index)

    # Calendar features for target date.
    future_index = panel.index + pd.Timedelta(days=horizon)
    cal = make_calendar_features(pd.DatetimeIndex(future_index))
    cal.index = panel.index
    features = pd.concat([features, cal], axis=1)

    # Lagged target history and rolling history.
    for t in targets:
        features[f"{t}_lag1"] = panel[t].shift(1)
        features[f"{t}_lag7"] = panel[t].shift(7)
        for w in windows:
            features[f"{t}_roll{w}_mean"] = panel[t].shift(1).rolling(w, min_periods=max(2, min(w, 3))).mean()
            features[f"{t}_roll{w}_max"] = panel[t].shift(1).rolling(w, min_periods=max(2, min(w, 3))).max()
            features[f"{t}_roll{w}_nonzero_rate"] = (panel[t].shift(1) > 0).rolling(w, min_periods=max(2, min(w, 3))).mean()

    # Lagged non-target variables.  To keep Colab execution fast and prevent noisy high-dimensional
    # fitting, retain clinically and operationally relevant predictors only.  Target history above
    # already captures all 8 target series.
    relevant_patterns = (
        "income", "expire", "transfused", "past_1wk",
        "dept_pt_IMH", "ward_pt_MICU", "ward_pt_NICU",
        "surg_general", "surg_",
        "plt_1_", "plt_2_", "plt_3_", "plt_4_", "plt_5_", "plt_6_", "plt_7_", "plt_8_",
        "wbc_", "hb_", "inr_", "aptt_",
    )
    non_targets = [c for c in panel.columns if c not in targets and any(p in c for p in relevant_patterns)]
    lagged = panel[non_targets].shift(1)
    lagged.columns = [f"{c}_lag1" for c in non_targets]
    features = pd.concat([features, lagged], axis=1)

    features = features.replace([np.inf, -np.inf], np.nan)
    return features


def target_series_for_horizon(panel: pd.DataFrame, target: str, horizon: int) -> pd.Series:
    return panel[target].shift(-horizon).rename(target)


# ============================================================
# 5. Models
# ============================================================

def make_lasso(alpha: float = 0.05, random_state: int = 42) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, max_iter=1000, tol=1e-3, selection="random", random_state=random_state)),
    ])


def make_ridge(alpha: float = 5.0) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha)),
    ])


def make_logistic(random_state: int = 42) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=0.5,
            penalty="l2",
            class_weight="balanced",
            solver="liblinear",
            max_iter=3000,
            random_state=random_state,
        )),
    ])



def make_lightgbm_regressor(config: SparseHurdleConfig):
    """Return the S1.8-specified LightGBM regressor; None only if unavailable.

    LightGBM is useful as a nonlinear tabular benchmark. It is evaluated with the
    same leakage-safe features and validation/test splits as LASSO and Ridge.
    """
    try:
        from lightgbm import LGBMRegressor
    except Exception as e:
        print("[skip] LightGBM is not installed. In Colab, run: !pip -q install lightgbm")
        return None
    return LGBMRegressor(
        n_estimators=config.lightgbm_n_estimators,
        learning_rate=config.lightgbm_learning_rate,
        num_leaves=config.lightgbm_num_leaves,
        min_child_samples=config.lightgbm_min_child_samples,
        max_depth=-1,
        subsample=0.8,
        subsample_freq=1,  # nonzero frequency is required to actually use 80% row sampling
        colsample_bytree=0.8,
        objective="regression",  # L2 loss per Supplementary S1.8
        reg_alpha=0.0,
        reg_lambda=0.0,
        random_state=config.random_state,
        n_jobs=1,
        verbosity=-1,
    )


def fit_predict_lightgbm(X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, X_test: pd.DataFrame, config: SparseHurdleConfig) -> Optional[Tuple[pd.Series, pd.Series]]:
    model = make_lightgbm_regressor(config)
    if model is None:
        return None
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    val_pred = pd.Series(np.maximum(pipe.predict(X_val), 0.0), index=X_val.index)
    test_pred = pd.Series(np.maximum(pipe.predict(X_test), 0.0), index=X_test.index)
    return val_pred, test_pred


def _select_prophet_regressors(X_train: pd.DataFrame, target: str, max_regressors: int) -> List[str]:
    """Select a small set of stable, leakage-safe regressors for Prophet."""
    priority_patterns = [
        f"{target}_lag1", f"{target}_lag7", f"{target}_roll7_mean", f"{target}_roll14_mean",
        "dow", "month", "is_weekend", "dow_fri",
        "plt_3_inpt", "plt_4_inpt", "plt_5_inpt", "dept_pt_IMH", "surg_general",
    ]
    cols = []
    for pat in priority_patterns:
        matches = [c for c in X_train.columns if c == pat or pat in c]
        for c in matches:
            if c not in cols:
                cols.append(c)
    # Fill remaining slots with features most correlated with y will be handled outside if needed;
    # here we keep deterministic selection to avoid target leakage through validation/test.
    numeric_cols = [c for c in X_train.columns if c not in cols]
    for c in numeric_cols:
        if len(cols) >= max_regressors:
            break
        if any(k in c for k in ["lag1", "lag7", "roll7_mean", "roll14_mean", "dow", "month"]):
            cols.append(c)
    return cols[:max_regressors]


def fit_predict_prophet(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    target: str,
    horizon: int,
    config: SparseHurdleConfig,
) -> Optional[Tuple[pd.Series, pd.Series]]:
    """Fit Prophet with a small set of known/lagged regressors.

    The ds column is the target date (origin date + horizon). Regressors are values
    available at the forecast origin, so this remains leakage-safe.
    """
    try:
        from prophet import Prophet
    except Exception:
        print("[skip] Prophet is not installed. In Colab, run: !pip -q install prophet")
        return None

    regs = _select_prophet_regressors(X_train, target, config.prophet_max_regressors)
    imputer = SimpleImputer(strategy="median")
    Xtr = pd.DataFrame(imputer.fit_transform(X_train[regs]), index=X_train.index, columns=regs) if regs else pd.DataFrame(index=X_train.index)
    Xv = pd.DataFrame(imputer.transform(X_val[regs]), index=X_val.index, columns=regs) if regs else pd.DataFrame(index=X_val.index)
    Xte = pd.DataFrame(imputer.transform(X_test[regs]), index=X_test.index, columns=regs) if regs else pd.DataFrame(index=X_test.index)

    train_df = pd.DataFrame({
        "ds": X_train.index + pd.Timedelta(days=horizon),
        "y": y_train.values,
    })
    for c in regs:
        train_df[c] = Xtr[c].values

    try:
        m = Prophet(
            weekly_seasonality=config.prophet_include_weekly,
            yearly_seasonality=config.prophet_include_yearly,
            daily_seasonality=False,
            seasonality_mode="additive",
            changepoint_prior_scale=0.05,
        )
        for c in regs:
            m.add_regressor(c, standardize=True)
        m.fit(train_df)

        def _predict(X_eval: pd.DataFrame) -> pd.Series:
            future = pd.DataFrame({"ds": X_eval.index + pd.Timedelta(days=horizon)})
            X_eval_imp = Xv if X_eval is X_val else Xte
            for c in regs:
                future[c] = X_eval_imp[c].values
            fc = m.predict(future)
            return pd.Series(np.maximum(fc["yhat"].values, 0.0), index=X_eval.index)

        return _predict(X_val), _predict(X_test)
    except Exception as e:
        print(f"[skip] Prophet failed for {target}, horizon={horizon}: {e}")
        return None


class _TorchLSTMRegressor:
    """Small PyTorch LSTM benchmark for tabular time-series features."""

    def __init__(self, config: SparseHurdleConfig):
        self.config = config
        self.fitted_ = False

    def _make_sequences(self, X: pd.DataFrame, y: pd.Series, fit: bool = False):
        import torch
        if fit:
            self.imputer_ = SimpleImputer(strategy="median")
            self.scaler_ = StandardScaler()
            X_imp = self.imputer_.fit_transform(X)
            X_scaled = self.scaler_.fit_transform(X_imp)
        else:
            X_imp = self.imputer_.transform(X)
            X_scaled = self.scaler_.transform(X_imp)
        L = self.config.lstm_sequence_length
        xs, ys, idx = [], [], []
        for i in range(L - 1, len(X_scaled)):
            xs.append(X_scaled[i - L + 1:i + 1, :])
            ys.append(float(y.iloc[i]))
            idx.append(y.index[i])
        if not xs:
            return None, None, []
        return torch.tensor(np.asarray(xs), dtype=torch.float32), torch.tensor(np.asarray(ys), dtype=torch.float32).view(-1, 1), idx

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series):
        try:
            import torch
            from torch import nn
            from torch.utils.data import DataLoader, TensorDataset
        except Exception:
            print("[skip] PyTorch is not installed; LSTM skipped.")
            return False

        torch.manual_seed(self.config.random_state)
        Xtr, ytr, _ = self._make_sequences(X_train, y_train, fit=True)
        Xv, yv, _ = self._make_sequences(X_val, y_val, fit=False)
        if Xtr is None or Xv is None:
            return False
        n_features = Xtr.shape[-1]
        cfg = self.config

        class Net(nn.Module):
            def __init__(self):
                super().__init__()
                dropout = cfg.lstm_dropout if cfg.lstm_num_layers > 1 else 0.0
                self.lstm = nn.LSTM(n_features, cfg.lstm_hidden_size, cfg.lstm_num_layers, batch_first=True, dropout=dropout)
                self.head = nn.Linear(cfg.lstm_hidden_size, 1)
            def forward(self, x):
                out, _ = self.lstm(x)
                return self.head(out[:, -1, :])

        self.model_ = Net()
        opt = torch.optim.Adam(self.model_.parameters(), lr=cfg.lstm_learning_rate)
        loss_fn = nn.L1Loss()
        loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=cfg.lstm_batch_size, shuffle=False)
        best_val = float("inf")
        best_state = None
        patience_left = cfg.lstm_patience
        for epoch in range(cfg.lstm_max_epochs):
            self.model_.train()
            for xb, yb in loader:
                opt.zero_grad()
                loss = loss_fn(self.model_(xb), yb)
                loss.backward()
                opt.step()
            self.model_.eval()
            with torch.no_grad():
                val_loss = loss_fn(self.model_(Xv), yv).item()
            if val_loss + 1e-6 < best_val:
                best_val = val_loss
                best_state = {k: v.detach().clone() for k, v in self.model_.state_dict().items()}
                patience_left = cfg.lstm_patience
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break
        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.fitted_ = True
        return True

    def predict(self, X: pd.DataFrame, y_placeholder: pd.Series) -> pd.Series:
        import torch
        Xseq, _, idx = self._make_sequences(X, y_placeholder, fit=False)
        if Xseq is None:
            return pd.Series(np.nan, index=X.index)
        self.model_.eval()
        with torch.no_grad():
            pred = self.model_(Xseq).cpu().numpy().ravel()
        out = pd.Series(np.nan, index=X.index, dtype=float)
        out.loc[idx] = np.maximum(pred, 0.0)
        # Fill the first sequence_length-1 dates with a simple median of predictions to keep metrics defined.
        fill_value = float(np.nanmedian(out.values)) if np.isfinite(np.nanmedian(out.values)) else 0.0
        return out.fillna(fill_value)


def fit_predict_lstm(X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series, X_test: pd.DataFrame, y_test: pd.Series, config: SparseHurdleConfig) -> Optional[Tuple[pd.Series, pd.Series]]:
    try:
        model = _TorchLSTMRegressor(config)
        ok = model.fit(X_train, y_train, X_val, y_val)
        if not ok:
            return None
        return model.predict(X_val, y_val), model.predict(X_test, y_test)
    except Exception as e:
        print(f"[skip] LSTM failed: {e}")
        return None


class HurdleModel:
    """Two-stage occurrence × positive-quantity model.

    The model can return a soft expected-demand forecast: P(y>0) × E(y | y>0),
    and thresholded forecasts where low occurrence probability maps to zero.
    Thresholds are tuned on validation, not test.
    """

    def __init__(
        self,
        occurrence_model: Optional[Pipeline] = None,
        quantity_model: Optional[Pipeline] = None,
        threshold: float = 0.50,
        min_positive_days: int = 30,
        random_state: int = 42,
    ):
        self.occurrence_model = occurrence_model or make_logistic(random_state=random_state)
        self.quantity_model = quantity_model or make_lasso(alpha=0.05, random_state=random_state)
        self.threshold = threshold
        self.min_positive_days = min_positive_days
        self.random_state = random_state
        self.fitted_ = False
        self.fallback_positive_mean_: float = 0.0
        self.has_two_classes_: bool = True
        self.has_enough_positives_: bool = True

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "HurdleModel":
        y = pd.Series(y).astype(float)
        occ = (y > 0).astype(int)
        self.has_two_classes_ = occ.nunique() == 2
        self.fallback_positive_mean_ = float(y[y > 0].mean()) if (y > 0).any() else 0.0
        self.has_enough_positives_ = int((y > 0).sum()) >= self.min_positive_days

        if self.has_two_classes_:
            self.occurrence_model.fit(X, occ)
        else:
            self.constant_occurrence_ = int(occ.iloc[0]) if len(occ) else 0

        if self.has_enough_positives_:
            self.quantity_model.fit(X.loc[y > 0], y.loc[y > 0])
        self.fitted_ = True
        return self

    def occurrence_probability(self, X: pd.DataFrame) -> np.ndarray:
        if not self.fitted_:
            raise RuntimeError("Model is not fitted.")
        if self.has_two_classes_:
            try:
                return self.occurrence_model.predict_proba(X)[:, 1]
            except Exception:
                pred = self.occurrence_model.predict(X)
                return np.asarray(pred, dtype=float)
        return np.full(len(X), float(getattr(self, "constant_occurrence_", 0)))

    def positive_quantity(self, X: pd.DataFrame) -> np.ndarray:
        if self.has_enough_positives_:
            q = self.quantity_model.predict(X)
        else:
            q = np.full(len(X), self.fallback_positive_mean_)
        return np.maximum(np.asarray(q, dtype=float), 0.0)

    def predict_soft(self, X: pd.DataFrame) -> np.ndarray:
        p = self.occurrence_probability(X)
        q = self.positive_quantity(X)
        return np.maximum(p * q, 0.0)

    def predict_thresholded(self, X: pd.DataFrame, threshold: Optional[float] = None) -> np.ndarray:
        threshold = self.threshold if threshold is None else threshold
        p = self.occurrence_probability(X)
        q = self.positive_quantity(X)
        return np.where(p >= threshold, q, 0.0).astype(float)


def rolling_forecast_series(y_full: pd.Series, eval_index: pd.DatetimeIndex, horizon: int, window: int, kind: str = "mean") -> pd.Series:
    """Forecast y(t+h) using information up to t = target_date - h."""
    preds = []
    for target_date in eval_index:
        cutoff = target_date - pd.Timedelta(days=horizon)
        hist = y_full.loc[y_full.index <= cutoff].tail(window)
        if len(hist) == 0:
            val = 0.0
        elif kind == "nonzero_rate":
            val = float((hist > 0).mean())
        elif kind == "positive_mean":
            pos = hist[hist > 0]
            val = float(pos.mean()) if len(pos) else 0.0
        else:
            val = float(hist.mean())
        preds.append(max(val, 0.0))
    return pd.Series(preds, index=eval_index)


def seasonal_naive_forecast(y_full: pd.Series, eval_index: pd.DatetimeIndex, horizon: int, seasonal_period: int = 7) -> pd.Series:
    """Weekly naïve forecast indexed by forecast origin t, predicting demand at t+h.

    For h=1 the source is y[t-6], i.e., the date (t+1)-7.
    """
    if horizon < 1 or seasonal_period < 1:
        raise ValueError("horizon and seasonal_period must be positive")
    preds = []
    for origin_date in eval_index:
        # eval_index denotes forecast ORIGIN t; target is t+h.
        # Use the same weekday one complete seasonal cycle before t+h.
        # For horizons longer than one cycle, step back additional full cycles
        # so the source cannot exceed the information cutoff at origin t.
        cycles_back = max(1, math.ceil(horizon / seasonal_period))
        source_date = origin_date + pd.Timedelta(days=horizon - cycles_back * seasonal_period)
        if source_date > origin_date:
            raise AssertionError("Seasonal naive would use future information")
        if source_date in y_full.index:
            val = float(y_full.loc[source_date])
        else:
            # Backup only when the requested historical date is unavailable.
            # Do not use observations later than the forecast origin.
            hist = y_full.loc[y_full.index <= origin_date]
            val = float(hist.tail(seasonal_period).mean()) if len(hist) else 0.0
        preds.append(max(val, 0.0))
    return pd.Series(preds, index=eval_index)


# ============================================================
# 6. Metrics and threshold selection
# ============================================================

def weighted_mae(y_true: np.ndarray, y_pred: np.ndarray, under_weight: float = 3.0) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = np.abs(y_true - y_pred)
    weights = np.where(y_pred < y_true, under_weight, 1.0)
    return float(np.mean(err * weights))


def safe_mase(y_true: np.ndarray, y_pred: np.ndarray, y_train: np.ndarray, seasonal_period: int = 7) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    if len(y_train) <= seasonal_period:
        denom = np.mean(np.abs(np.diff(y_train))) if len(y_train) > 1 else np.nan
    else:
        denom = np.mean(np.abs(y_train[seasonal_period:] - y_train[:-seasonal_period]))
    if not np.isfinite(denom) or denom <= 1e-12:
        return np.nan
    return float(np.mean(np.abs(y_true - y_pred)) / denom)


def occurrence_metrics(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.50) -> Dict[str, float]:
    y_bin = (np.asarray(y_true) > 0).astype(int)
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    out = {
        "occurrence_accuracy": float(accuracy_score(y_bin, y_pred)),
        "occurrence_precision": float(precision_score(y_bin, y_pred, zero_division=0)),
        "occurrence_recall": float(recall_score(y_bin, y_pred, zero_division=0)),
        "occurrence_f1": float(f1_score(y_bin, y_pred, zero_division=0)),
    }
    if len(np.unique(y_bin)) == 2 and len(np.unique(y_score)) > 1:
        try:
            out["occurrence_auc"] = float(roc_auc_score(y_bin, y_score))
        except Exception:
            out["occurrence_auc"] = np.nan
    else:
        out["occurrence_auc"] = np.nan
    return out


def forecast_metrics(
    y_true: pd.Series,
    y_pred: pd.Series,
    y_train: pd.Series,
    under_weight: float,
    high_threshold: float,
    occurrence_score: Optional[pd.Series] = None,
    occurrence_threshold: float = 0.50,
) -> Dict[str, float]:
    y_true = y_true.astype(float)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(float).clip(lower=0)
    out = {
        "n": int(len(y_true)),
        "mean_actual": float(y_true.mean()),
        "mean_pred": float(y_pred.mean()),
        "zero_rate_actual": float((y_true == 0).mean()),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "weighted_mae": weighted_mae(y_true.values, y_pred.values, under_weight),
        "mase": safe_mase(y_true.values, y_pred.values, y_train.values),
        "bias": float(np.mean(y_pred.values - y_true.values)),
        "underprediction_rate": float((y_pred.values < y_true.values).mean()),
        "overprediction_rate": float((y_pred.values > y_true.values).mean()),
    }
    # High-demand event detection: score can be forecast quantity.
    true_high = (y_true.values >= high_threshold).astype(int)
    pred_high = (y_pred.values >= high_threshold).astype(int)
    out.update({
        "high_precision": float(precision_score(true_high, pred_high, zero_division=0)),
        "high_recall": float(recall_score(true_high, pred_high, zero_division=0)),
        "high_f1": float(f1_score(true_high, pred_high, zero_division=0)),
    })
    # Occurrence detection for hurdle-like interpretation.
    occ_score = occurrence_score if occurrence_score is not None else y_pred
    out.update(occurrence_metrics(y_true.values, np.asarray(occ_score), occurrence_threshold))
    return out


def tune_hurdle_threshold(
    model: HurdleModel,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    threshold_grid: Sequence[float],
    under_weight: float,
) -> Tuple[float, pd.DataFrame]:
    rows = []
    for th in threshold_grid:
        pred = pd.Series(model.predict_thresholded(X_val, threshold=th), index=y_val.index)
        p = pd.Series(model.occurrence_probability(X_val), index=y_val.index)
        row = {"threshold": th}
        row.update(forecast_metrics(
            y_true=y_val,
            y_pred=pred,
            y_train=y_val,  # unused for threshold choice except MASE placeholder
            under_weight=under_weight,
            high_threshold=float(y_val.quantile(0.90)),
            occurrence_score=p,
            occurrence_threshold=th,
        ))
        rows.append(row)
    df = pd.DataFrame(rows)
    best = df.sort_values(["weighted_mae", "mae", "occurrence_f1"], ascending=[True, True, False]).iloc[0]
    return float(best["threshold"]), df


# ============================================================
# 7. Prediction pipeline
# ============================================================

_FEATURE_CACHE: Dict[int, pd.DataFrame] = {}


def split_X_y_from_all(
    X_all: pd.DataFrame,
    y_all: pd.Series,
    start: str,
    end: str,
    horizon: int = 1,
) -> Tuple[pd.DataFrame, pd.Series]:
    """Split by the DATE BEING PREDICTED, leaving all features unchanged.

    y_all indexed at origin t actually contains demand at t+horizon.
    Does not use any outcome of the validation/test period in prior fitting.
    """
    if not X_all.index.equals(y_all.index):
        raise ValueError('The feature/target origin indexes do not match')
    target_dates = pd.DatetimeIndex(X_all.index) + pd.Timedelta(days=horizon)
    idx_mask = in_window(target_dates, start, end)
    valid_mask = idx_mask & y_all.notna().values
    X = X_all.loc[valid_mask]
    y = y_all.loc[valid_mask].astype(float)
    return X, y


def fit_predict_models_for_target_horizon(
    panel: pd.DataFrame,
    targets: Sequence[str],
    target: str,
    horizon: int,
    config: SparseHurdleConfig,
    diag_row: Dict[str, object],
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fit general and sparse-aware candidates, return predictions and metrics for val/test."""
    # Build leakage-safe feature matrix once per horizon, then reuse for all targets.
    if horizon not in _FEATURE_CACHE:
        _FEATURE_CACHE[horizon] = build_feature_frame(panel, targets, horizon=horizon, windows=config.moving_windows)
    X_all = _FEATURE_CACHE[horizon]
    y_all = target_series_for_horizon(panel, target, horizon=horizon)
    X_train, y_train = split_X_y_from_all(X_all, y_all, config.train_start, config.train_end, horizon=horizon)
    X_val, y_val = split_X_y_from_all(X_all, y_all, config.val_start, config.val_end, horizon=horizon)
    X_test, y_test = split_X_y_from_all(X_all, y_all, config.test_start, config.test_end, horizon=horizon)
    # Reproducibility guard: the outcome-date test interval has exactly 181 days.
    expected_n = len(pd.date_range(config.test_start, config.test_end, freq='D'))
    if len(y_test) != expected_n:
        raise AssertionError(f'{target}, h={horizon}: expected {expected_n} test days, got {len(y_test)}')
    if (X_train.index + pd.Timedelta(days=horizon)).max() >= pd.Timestamp(config.val_start):
        raise AssertionError(f'{target}, h={horizon}: train includes validation outcomes')
    if (X_val.index + pd.Timedelta(days=horizon)).max() >= pd.Timestamp(config.test_start):
        raise AssertionError(f'{target}, h={horizon}: validation includes test outcomes')

    if len(y_train) == 0 or len(y_val) == 0 or len(y_test) == 0:
        raise ValueError(f"Empty split for {target}, horizon={horizon}")

    high_threshold = float(y_train.quantile(config.high_demand_quantile))
    if high_threshold <= 0:
        high_threshold = float(max(1.0, y_train[y_train > 0].quantile(config.high_demand_quantile) if (y_train > 0).any() else 1.0))

    pred_rows: List[pd.DataFrame] = []
    metric_rows: List[Dict[str, object]] = []
    threshold_rows: List[pd.DataFrame] = []

    def add_predictions(split: str, model_name: str, y: pd.Series, pred: pd.Series, occ_score: Optional[pd.Series] = None, occ_threshold: float = 0.50):
        nonlocal pred_rows, metric_rows
        pred = pd.Series(pred, index=y.index).astype(float).clip(lower=0)
        rowdf = pd.DataFrame({
            "date": y.index,  # forecast-origin date (kept for backward compatibility)
            "target_date": y.index + pd.Timedelta(days=horizon),
            "split": split,
            "target": target,
            "horizon": horizon,
            "model": model_name,
            "actual": y.values,
            "pred": pred.values,
            "is_sparse_target": bool(diag_row.get("force_hurdle_candidates", False)),
        })
        if occ_score is not None:
            rowdf["occurrence_score"] = pd.Series(occ_score, index=y.index).values
            rowdf["occurrence_threshold"] = occ_threshold
        pred_rows.append(rowdf)
        m = forecast_metrics(
            y_true=y,
            y_pred=pred,
            y_train=y_train,
            under_weight=config.underprediction_weight,
            high_threshold=high_threshold,
            occurrence_score=occ_score,
            occurrence_threshold=occ_threshold,
        )
        metric_rows.append({
            "split": split,
            "target": target,
            "horizon": horizon,
            "model": model_name,
            "is_sparse_target": bool(diag_row.get("force_hurdle_candidates", False)),
            **m,
        })

    # Baselines and ordinary regression models.
    y_full = panel[target].astype(float)
    for split, y_eval in [("validation", y_val), ("test", y_test)]:
        # historical mean uses training mean only.
        add_predictions(split, "historical_mean", y_eval, pd.Series(y_train.mean(), index=y_eval.index))
        add_predictions(split, "seasonal_naive", y_eval, seasonal_naive_forecast(y_full, y_eval.index, horizon, config.seasonal_period))
        for w in config.moving_windows:
            add_predictions(split, f"moving_average_{w}d", y_eval, rolling_forecast_series(y_full, y_eval.index, horizon, w, kind="mean"))

    regression_models = {
        "lasso": make_lasso(alpha=0.05, random_state=config.random_state),
        "ridge": make_ridge(alpha=5.0),
    }
    reg_val_preds = {}
    reg_test_preds = {}
    for name, model in regression_models.items():
        fitted = clone(model).fit(X_train, y_train)
        val_pred = pd.Series(np.maximum(fitted.predict(X_val), 0.0), index=y_val.index)
        test_pred = pd.Series(np.maximum(fitted.predict(X_test), 0.0), index=y_test.index)
        reg_val_preds[name] = val_pred
        reg_test_preds[name] = test_pred
        add_predictions("validation", name, y_val, val_pred)
        add_predictions("test", name, y_test, test_pred)

    # Optional extended manuscript-comparison models: LightGBM, Prophet, and LSTM.
    # These are evaluated under exactly the same splits and metrics. If a package is
    # unavailable or a model fails for a sparse target, it is skipped and documented by the console log.
    if config.run_lightgbm:
        out = fit_predict_lightgbm(X_train, y_train, X_val, X_test, config)
        if out is not None:
            val_pred, test_pred = out
            add_predictions("validation", "lightgbm", y_val, val_pred)
            add_predictions("test", "lightgbm", y_test, test_pred)

    if config.run_prophet:
        out = fit_predict_prophet(X_train, y_train, X_val, X_test, target, horizon, config)
        if out is not None:
            val_pred, test_pred = out
            add_predictions("validation", "prophet_augmented", y_val, val_pred)
            add_predictions("test", "prophet_augmented", y_test, test_pred)

    if config.run_lstm:
        out = fit_predict_lstm(X_train, y_train, X_val, y_val, X_test, y_test, config)
        if out is not None:
            val_pred, test_pred = out
            add_predictions("validation", "lstm", y_val, val_pred)
            add_predictions("test", "lstm", y_test, test_pred)

    # Blends combine feature-based model with adaptive rolling behavior.
    for base_name in ["lasso"]:
        for w in [7, 14]:
            val_ma = rolling_forecast_series(y_full, y_val.index, horizon, w, kind="mean")
            test_ma = rolling_forecast_series(y_full, y_test.index, horizon, w, kind="mean")
            val_pred = 0.5 * reg_val_preds[base_name] + 0.5 * val_ma
            test_pred = 0.5 * reg_test_preds[base_name] + 0.5 * test_ma
            model_name = f"blend_{base_name}_ma{w}_50_50"
            add_predictions("validation", model_name, y_val, val_pred)
            add_predictions("test", model_name, y_test, test_pred)

    # Hurdle candidates are required for sparse/intermittent targets, but also evaluated for all targets
    # so that the validation process can decide whether they help.
    positive_days = int((y_train > 0).sum())
    force_hurdle = bool(diag_row.get("force_hurdle_candidates", False))
    if force_hurdle:
        # Hurdle LASSO: logistic occurrence + LASSO positive quantity.
        hurdle_lasso = HurdleModel(
            occurrence_model=make_logistic(random_state=config.random_state),
            quantity_model=make_lasso(alpha=0.05, random_state=config.random_state),
            min_positive_days=config.min_positive_days_for_hurdle,
            random_state=config.random_state,
        ).fit(X_train, y_train)
        best_th, th_df = tune_hurdle_threshold(
            hurdle_lasso, X_val, y_val,
            config.occurrence_threshold_grid,
            config.underprediction_weight,
        )
        hurdle_lasso.threshold = best_th
        th_df.insert(0, "target", target)
        th_df.insert(1, "horizon", horizon)
        th_df.insert(2, "model", "hurdle_lasso")
        threshold_rows.append(th_df)

        for mode in ["soft", "thresholded"]:
            model_name = f"hurdle_lasso_{mode}"
            if mode == "soft":
                val_pred = pd.Series(hurdle_lasso.predict_soft(X_val), index=y_val.index)
                test_pred = pd.Series(hurdle_lasso.predict_soft(X_test), index=y_test.index)
            else:
                val_pred = pd.Series(hurdle_lasso.predict_thresholded(X_val), index=y_val.index)
                test_pred = pd.Series(hurdle_lasso.predict_thresholded(X_test), index=y_test.index)
            val_prob = pd.Series(hurdle_lasso.occurrence_probability(X_val), index=y_val.index)
            test_prob = pd.Series(hurdle_lasso.occurrence_probability(X_test), index=y_test.index)
            add_predictions("validation", model_name, y_val, val_pred, occ_score=val_prob, occ_threshold=best_th)
            add_predictions("test", model_name, y_test, test_pred, occ_score=test_prob, occ_threshold=best_th)

        # Hurdle rolling: rolling nonzero probability × rolling positive mean.
        for w in [14]:
            for split, y_eval in [("validation", y_val), ("test", y_test)]:
                p = rolling_forecast_series(y_full, y_eval.index, horizon, w, kind="nonzero_rate")
                q = rolling_forecast_series(y_full, y_eval.index, horizon, w, kind="positive_mean")
                # Soft expected demand and thresholded version.  Use validation-tuned threshold based on grid.
                soft_pred = p * q
                add_predictions(split, f"hurdle_rolling{w}_soft", y_eval, soft_pred, occ_score=p, occ_threshold=0.50)
            # Tune rolling threshold on validation.
            p_val = rolling_forecast_series(y_full, y_val.index, horizon, w, kind="nonzero_rate")
            q_val = rolling_forecast_series(y_full, y_val.index, horizon, w, kind="positive_mean")
            rows = []
            for th in config.occurrence_threshold_grid:
                pred = pd.Series(np.where(p_val >= th, q_val, 0.0), index=y_val.index)
                m = forecast_metrics(y_val, pred, y_train, config.underprediction_weight, high_threshold, p_val, th)
                rows.append({"threshold": th, **m})
            roll_th_df = pd.DataFrame(rows)
            best_roll_th = float(roll_th_df.sort_values(["weighted_mae", "mae", "occurrence_f1"], ascending=[True, True, False]).iloc[0]["threshold"])
            roll_th_df.insert(0, "target", target)
            roll_th_df.insert(1, "horizon", horizon)
            roll_th_df.insert(2, "model", f"hurdle_rolling{w}_thresholded")
            threshold_rows.append(roll_th_df)
            for split, y_eval in [("validation", y_val), ("test", y_test)]:
                p = rolling_forecast_series(y_full, y_eval.index, horizon, w, kind="nonzero_rate")
                q = rolling_forecast_series(y_full, y_eval.index, horizon, w, kind="positive_mean")
                pred = pd.Series(np.where(p >= best_roll_th, q, 0.0), index=y_eval.index)
                add_predictions(split, f"hurdle_rolling{w}_thresholded", y_eval, pred, occ_score=p, occ_threshold=best_roll_th)

    preds = pd.concat(pred_rows, ignore_index=True)
    metrics = pd.DataFrame(metric_rows)
    thresholds = pd.concat(threshold_rows, ignore_index=True) if threshold_rows else pd.DataFrame()
    return preds, metrics, thresholds


def run_forecasting(panel: pd.DataFrame, targets: Sequence[str], diag: pd.DataFrame, config: SparseHurdleConfig) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    all_preds = []
    all_metrics = []
    all_thresholds = []
    diag_map = diag.set_index("target").to_dict(orient="index")
    for h in range(1, config.max_horizon + 1):
        print(f"[forecast] horizon={h}")
        for target in targets:
            preds, metrics, thresholds = fit_predict_models_for_target_horizon(
                panel=panel,
                targets=targets,
                target=target,
                horizon=h,
                config=config,
                diag_row=diag_map[target],
            )
            all_preds.append(preds)
            all_metrics.append(metrics)
            if len(thresholds):
                all_thresholds.append(thresholds)
    pred_df = pd.concat(all_preds, ignore_index=True)
    metric_df = pd.concat(all_metrics, ignore_index=True)
    threshold_df = pd.concat(all_thresholds, ignore_index=True) if all_thresholds else pd.DataFrame()
    return pred_df, metric_df, threshold_df


# ============================================================
# 8. Validation model selection and inventory simulation
# ============================================================

def select_models_by_validation(metrics: pd.DataFrame, config: SparseHurdleConfig) -> pd.DataFrame:
    """Select forecasting model per target/horizon using validation weighted MAE.

    For sparse targets, hurdle models are not automatically forced to win; they must improve validation.
    But the selection table keeps a flag showing whether hurdle/two-stage candidates were required.
    """
    val = metrics[metrics["split"] == "validation"].copy()
    rows = []
    for (target, horizon), g in val.groupby(["target", "horizon"]):
        g2 = g.sort_values(["weighted_mae", "mae", "occurrence_f1"], ascending=[True, True, False])
        best = g2.iloc[0].to_dict()
        best["selection_metric"] = "validation_weighted_mae"
        best["ranked_candidate_count"] = len(g2)
        rows.append(best)
    return pd.DataFrame(rows)


def build_selected_prediction_table(preds: pd.DataFrame, selected: pd.DataFrame, split: str) -> pd.DataFrame:
    sel = selected[["target", "horizon", "model"]].rename(columns={"model": "selected_model"})
    df = preds[preds["split"] == split].merge(sel, on=["target", "horizon"], how="inner")
    df = df[df["model"] == df["selected_model"]].copy()
    return df.drop(columns=["selected_model"])


def prediction_lookup(selected_preds: pd.DataFrame) -> Dict[Tuple[str, int], pd.Series]:
    out: Dict[Tuple[str, int], pd.Series] = {}
    for (target, h), g in selected_preds.groupby(["target", "horizon"]):
        s = pd.Series(g["pred"].values, index=pd.to_datetime(g["date"])).sort_index()
        out[(target, int(h))] = s
    return out


def expected_demand_window(preds_by_horizon: Dict[Tuple[str, int], pd.Series], target: str, date: pd.Timestamp, target_days_supply: float) -> float:
    full_days = int(math.floor(target_days_supply))
    frac = target_days_supply - full_days
    total = 0.0
    for h in range(1, full_days + 1):
        s = preds_by_horizon.get((target, h))
        if s is not None and date in s.index:
            total += float(s.loc[date])
    if frac > 1e-9:
        h = full_days + 1
        s = preds_by_horizon.get((target, h))
        if s is not None and date in s.index:
            total += frac * float(s.loc[date])
    return max(total, 0.0)


def _as_int_units(x: float) -> int:
    """Round to the nearest whole unit, never negative. Used for inventory states
    and demand (integer platelet units), matching the discrete-unit, age-structured
    engine described in Supplementary Methods S1.15."""
    if x is None or not np.isfinite(float(x)):
        return 0
    return max(0, int(round(float(x))))


def simulate_inventory_for_target(
    actual: pd.Series,
    preds_by_horizon: Dict[Tuple[str, int], pd.Series],
    target: str,
    target_days_supply: float,
    safety_factor: float,
    config: SparseHurdleConfig,
    initial_reference_demand: Optional[float] = None,
) -> Tuple[Dict[str, float], pd.DataFrame]:
    """Integer, age-structured inventory simulation (discrete units, ceiling-based
    ordering). This mirrors the engine described in Supplementary Methods S1.15:
    on-hand inventory, demand, arrivals, and order quantities are all whole units;
    the initial inventory seed is rounded to the nearest integer, and every
    subsequent order quantity is rounded UP using the ceiling function so that the
    order-up-to target is never undershot by rounding. Exact daily mass balance
    (opening inventory + arrivals = fulfilled + wasted + closing inventory) is
    verified for every simulated day.
    """
    actual = actual.astype(float).sort_index()
    inv_by_age = [0 for _ in range(config.shelf_life_days)]
    first_forecast = expected_demand_window(preds_by_horizon, target, actual.index[0], target_days_supply)
    # IMPORTANT: No test/validation outcome may be used to initialize stock.
    # The calling code supplies a demand mean computed STRICTLY BEFORE the
    # evaluation interval (training for validation; train+validation for test).
    if initial_reference_demand is None or not np.isfinite(float(initial_reference_demand)):
        raise ValueError("Supply pre-period initial_reference_demand; using actual.mean() leaks future data")
    if initial_reference_demand < 0:
        raise ValueError("Historical reference demand cannot be negative")
    init_units = max(
        config.initial_days_supply * (first_forecast / max(target_days_supply, 1e-6)),
        float(initial_reference_demand) * config.initial_days_supply,
    )
    init_units = _as_int_units(init_units)
    inv_by_age[0] = init_units
    pending_orders: List[Tuple[pd.Timestamp, int]] = []
    rows = []

    for date, demand in actual.items():
        opening_inventory = sum(inv_by_age)

        # Receive orders arriving today (integer units).
        arrivals = 0
        remaining_pending = []
        for arrival_date, qty in pending_orders:
            if arrival_date <= date:
                arrivals += qty
            else:
                remaining_pending.append((arrival_date, qty))
        pending_orders = remaining_pending

        # Age inventory and add arrivals as age 0.
        expired = inv_by_age[-1]
        inv_by_age = [arrivals] + inv_by_age[:-1]

        # Fulfill oldest first (integer demand).
        need = _as_int_units(demand)
        fulfilled = 0
        for age in reversed(range(len(inv_by_age))):
            take = min(inv_by_age[age], need)
            inv_by_age[age] -= take
            need -= take
            fulfilled += take
            if need <= 0:
                break
        unmet = max(need, 0)

        on_hand = sum(inv_by_age)
        already_ordered = sum(qty for _, qty in pending_orders)
        desired_position = safety_factor * expected_demand_window(preds_by_horizon, target, date, target_days_supply)
        # Ceiling is applied to the whole gap between the desired inventory
        # position and what is already on hand or on order, not to each term
        # separately, so the order-up-to target is met or exceeded exactly.
        order_qty = max(0, math.ceil(desired_position - (on_hand + already_ordered)))
        if order_qty > 0:
            pending_orders.append((date + pd.Timedelta(days=config.lead_time_days), order_qty))

        closing_inventory = sum(inv_by_age)
        # Exact integer mass balance for the day: opening + arrivals = fulfilled + wasted + closing.
        assert opening_inventory + arrivals == fulfilled + expired + closing_inventory, (
            f"Mass-balance violation for {target} on {date}: "
            f"opening={opening_inventory} arrivals={arrivals} fulfilled={fulfilled} "
            f"wasted={expired} closing={closing_inventory}"
        )

        rows.append({
            "date": date,
            "target": target,
            "initial_reference_demand": float(initial_reference_demand),
            "initial_inventory_units": int(init_units),
            "actual_demand": float(demand),
            "fulfilled": int(fulfilled),
            "unmet": int(unmet),
            "wasted": int(expired),
            "procured_ordered": int(order_qty),
            "on_hand_end": int(closing_inventory),
            "target_days_supply": target_days_supply,
            "safety_factor": safety_factor,
        })

    trace = pd.DataFrame(rows)
    demand_sum = trace["actual_demand"].sum()
    procured = trace["procured_ordered"].sum()
    fulfilled = trace["fulfilled"].sum()
    unmet = trace["unmet"].sum()
    wasted = trace["wasted"].sum()
    summary = {
        "target": target,
        "initial_reference_demand": float(initial_reference_demand),
        "initial_inventory_units": int(init_units),
        "demand": float(demand_sum),
        "procured": float(procured),
        "fulfilled": float(fulfilled),
        "unmet": float(unmet),
        "wasted": float(wasted),
        "service_level": float(fulfilled / demand_sum) if demand_sum > 0 else np.nan,
        "wastage_rate": float(wasted / procured) if procured > 0 else 0.0,
        "cost": float(config.shortage_cost * unmet + config.wastage_cost * wasted + config.procurement_cost * procured),
        "target_days_supply": target_days_supply,
        "safety_factor": safety_factor,
    }
    return summary, trace


def run_inventory_grid(
    panel: pd.DataFrame,
    targets: Sequence[str],
    selected_preds: pd.DataFrame,
    config: SparseHurdleConfig,
    split: str,
    start: str,
    end: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    preds_by_h = prediction_lookup(selected_preds[selected_preds["split"] == split])
    actual_period = panel.loc[in_window(panel.index, start, end), list(targets)]
    summaries = []
    traces = []
    prior_mask = (panel.index >= pd.Timestamp(config.train_start)) & (panel.index < pd.Timestamp(start))
    if not prior_mask.any():
        raise ValueError("No pre-period history for inventory grid initialization")
    for t in targets:
        actual = actual_period[t]
        reference_mean = float(panel.loc[prior_mask, t].mean())
        for days in config.target_days_supply_grid:
            for sf in config.safety_factor_grid:
                summary, trace = simulate_inventory_for_target(
                    actual, preds_by_h, t, days, sf, config,
                    initial_reference_demand=reference_mean,
                )
                summaries.append(summary)
                trace = trace.copy()
                trace["split"] = split
                traces.append(trace)
    return pd.DataFrame(summaries), pd.concat(traces, ignore_index=True)


def select_inventory_policy(validation_grid: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for target, g in validation_grid.groupby("target"):
        best = g.sort_values(["cost", "unmet", "wasted"], ascending=[True, True, True]).iloc[0].to_dict()
        best["selection_basis"] = "validation_min_cost"
        rows.append(best)
    return pd.DataFrame(rows)


def evaluate_selected_inventory(
    panel: pd.DataFrame,
    targets: Sequence[str],
    selected_preds: pd.DataFrame,
    selected_policy: pd.DataFrame,
    config: SparseHurdleConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    preds_by_h = prediction_lookup(selected_preds[selected_preds["split"] == "test"])
    actual_test = panel.loc[in_window(panel.index, config.test_start, config.test_end), list(targets)]
    policy_map = selected_policy.set_index("target").to_dict(orient="index")
    summaries = []
    traces = []
    pretest_mask = ((panel.index >= pd.Timestamp(config.train_start)) &
                    (panel.index < pd.Timestamp(config.test_start)))
    if not pretest_mask.any():
        raise ValueError("No pretest history for inventory initialization")
    for t in targets:
        pol = policy_map[t]
        reference_mean = float(panel.loc[pretest_mask, t].mean())
        summary, trace = simulate_inventory_for_target(
            actual=actual_test[t],
            preds_by_horizon=preds_by_h,
            target=t,
            target_days_supply=float(pol["target_days_supply"]),
            safety_factor=float(pol["safety_factor"]),
            config=config,
            initial_reference_demand=reference_mean,
        )
        summary["selected_on_validation"] = True
        summaries.append(summary)
        trace["split"] = "test"
        traces.append(trace)
    target_summary = pd.DataFrame(summaries)
    target_summary["product"] = np.where(target_summary["target"].str.startswith("aph_"), "APC", "PC")
    product_summary = target_summary.groupby("product", as_index=False).agg({
        "demand": "sum",
        "procured": "sum",
        "fulfilled": "sum",
        "unmet": "sum",
        "wasted": "sum",
        "cost": "sum",
    })
    product_summary["service_level"] = product_summary["fulfilled"] / product_summary["demand"]
    product_summary["wastage_rate"] = np.where(product_summary["procured"] > 0, product_summary["wasted"] / product_summary["procured"], 0.0)
    return target_summary, product_summary, pd.concat(traces, ignore_index=True)


# ============================================================
# 9. Reporting
# ============================================================

def summarize_models(metrics: pd.DataFrame) -> pd.DataFrame:
    return metrics.groupby(["split", "horizon", "model"], as_index=False).agg(
        mean_mae=("mae", "mean"),
        mean_weighted_mae=("weighted_mae", "mean"),
        mean_mase=("mase", "mean"),
        mean_bias=("bias", "mean"),
        mean_occurrence_f1=("occurrence_f1", "mean"),
        mean_high_f1=("high_f1", "mean"),
        n_targets=("target", "nunique"),
    ).sort_values(["split", "horizon", "mean_weighted_mae", "mean_mae"])


def paired_tests_vs_lasso(metrics: pd.DataFrame, split: str = "test", horizon: int = 1) -> pd.DataFrame:
    g = metrics[(metrics["split"] == split) & (metrics["horizon"] == horizon)].copy()
    rows = []
    lasso = g[g["model"] == "lasso"]
    if lasso.empty:
        return pd.DataFrame()
    for metric in ["mae", "weighted_mae"]:
        lasso_map = lasso.set_index("target")[metric]
        for model, gm in g.groupby("model"):
            if model == "lasso":
                continue
            comp_map = gm.set_index("target")[metric]
            common = sorted(set(lasso_map.index) & set(comp_map.index))
            if len(common) < 3:
                pval = np.nan
                diff = np.nan
            else:
                d = lasso_map.loc[common].values - comp_map.loc[common].values
                diff = float(np.mean(d))
                try:
                    pval = float(wilcoxon(lasso_map.loc[common], comp_map.loc[common]).pvalue)
                except Exception:
                    pval = np.nan
            rows.append({
                "split": split,
                "horizon": horizon,
                "metric": metric,
                "comparison": f"lasso vs {model}",
                "n_common_targets": len(common),
                "lasso_mean": float(lasso_map.loc[common].mean()) if common else np.nan,
                "comparator_mean": float(comp_map.loc[common].mean()) if common else np.nan,
                "mean_difference_lasso_minus_comparator": diff,
                "p_value": pval,
                "interpretation": "LASSO significantly better" if np.isfinite(pval) and pval < 0.05 and diff < 0 else "No significant LASSO advantage",
            })
    return pd.DataFrame(rows)


def make_decision_table(diag: pd.DataFrame, selected_models: pd.DataFrame, selected_inventory: pd.DataFrame, test_metrics: pd.DataFrame) -> pd.DataFrame:
    h1_sel = selected_models[selected_models["horizon"] == 1][["target", "model", "weighted_mae", "mae", "occurrence_f1"]].rename(
        columns={"model": "selected_h1_model", "weighted_mae": "val_weighted_mae", "mae": "val_mae", "occurrence_f1": "val_occurrence_f1"}
    )
    test_best = test_metrics[(test_metrics["split"] == "test") & (test_metrics["horizon"] == 1)].copy()
    test_best = test_best.sort_values(["target", "weighted_mae", "mae"]).groupby("target", as_index=False).first()
    test_best = test_best[["target", "model", "weighted_mae", "mae", "occurrence_f1"]].rename(
        columns={"model": "best_test_h1_model", "weighted_mae": "best_test_weighted_mae", "mae": "best_test_mae", "occurrence_f1": "best_test_occurrence_f1"}
    )
    inv = selected_inventory[["target", "target_days_supply", "safety_factor", "cost", "service_level", "wastage_rate"]].rename(
        columns={"cost": "val_inventory_cost", "service_level": "val_service_level", "wastage_rate": "val_wastage_rate"}
    )
    out = diag.merge(h1_sel, on="target", how="left").merge(test_best, on="target", how="left").merge(inv, on="target", how="left")
    out["final_method_comment"] = np.where(
        out["force_hurdle_candidates"],
        "Sparse/intermittent: interpret general regression cautiously; prefer validated hurdle/blend/adaptive policy.",
        "Non-sparse: general regression acceptable, but adaptive baselines remain required comparators.",
    )
    return out


def write_caveats(path: Path, config: SparseHurdleConfig) -> None:
    txt = f"""Sparse-aware platelet forecasting methodological notes
=========================================================

1. Demand definition
   The outcome is observed issued-unit demand. It is not unobserved true clinical demand.
   Service level therefore means fulfillment of observed operational demand.

2. Why sparse targets require hurdle/two-stage candidates
   Targets with high zero-demand rate or intermittent/lumpy demand patterns are unstable under
   ordinary regression alone. A single regression model mixes occurrence and quantity. This script
   therefore compares general regression with hurdle/two-stage candidates:
      occurrence model: P(y > 0)
      quantity model: E(y | y > 0)
      final forecast: soft P × quantity or thresholded occurrence × quantity

3. Sparse classification
   Sparse/intermittent targets are flagged by zero-rate >= {config.sparse_zero_rate_threshold},
   ADI >= {config.intermittent_adi_threshold}, or Syntetos-Boylan class of intermittent/lumpy.

4. Validation-based selection
   Hurdle models are not automatically declared superior. They must improve validation performance.
   The final test period is used only for evaluation.

5. Optional extended models
   LightGBM, Prophet, and LSTM are included as candidate comparators when enabled. They are evaluated
   under the same train/validation/test split, weighted MAE, occurrence metrics, and inventory-policy
   selection rules as the sparse-aware baseline models.

6. Cost-sensitive objective
   Underprediction receives {config.underprediction_weight} times the error weight of overprediction.
   Inventory cost is {config.shortage_cost} × unmet + {config.wastage_cost} × wasted + {config.procurement_cost} × procured.

7. Manuscript interpretation
   If hurdle/two-stage models improve sparse targets, report this as evidence that product-ABO-specific
   modeling is required. If they do not improve a sparse target, still state that general regression alone
   is unstable and must be benchmarked against sparse-aware alternatives.
"""
    path.write_text(txt, encoding="utf-8")


# ============================================================
# 10. Main runner
# ============================================================

def run_analysis(config: SparseHurdleConfig) -> Dict[str, pd.DataFrame]:
    outdir = ensure_dir(config.output_dir)
    print("=" * 80)
    print("Sparse-aware platelet forecasting analysis with LightGBM, Prophet, and LSTM")
    print("Data:", config.data_path)
    print("Output:", outdir)
    print("=" * 80)

    panel, metadata = load_platelet_excel(config.data_path)
    targets = get_target_columns(config, panel)
    print(f"Panel shape: {panel.shape}; targets: {targets}")

    audit = []
    for c in panel.columns:
        s = panel[c]
        audit.append({
            "variable": c,
            "is_target": c in targets,
            "n_days": len(s),
            "missing_n": int(s.isna().sum()),
            "zero_rate": float((s == 0).mean()),
            "mean": float(s.mean()),
            "std": float(s.std()),
            "min": float(s.min()),
            "max": float(s.max()),
        })
    audit_df = pd.DataFrame(audit)
    diag = sparse_diagnostics(panel, targets, config)

    audit_df.to_csv(outdir / "00_data_audit.csv", index=False)
    diag.to_csv(outdir / "01_sparse_intermittent_diagnostics.csv", index=False)

    preds, metrics, thresholds = run_forecasting(panel, targets, diag, config)
    preds.to_csv(outdir / "02_all_predictions_long.csv", index=False)
    metrics.to_csv(outdir / "03_all_metrics_by_target.csv", index=False)
    thresholds.to_csv(outdir / "04_hurdle_threshold_tuning.csv", index=False)

    summary = summarize_models(metrics)
    summary.to_csv(outdir / "05_model_summary_by_split_horizon.csv", index=False)

    selected_models = select_models_by_validation(metrics, config)
    selected_models.to_csv(outdir / "06_selected_forecast_model_by_validation.csv", index=False)

    selected_preds = build_selected_prediction_table(preds, selected_models, split="validation")
    selected_preds_test = build_selected_prediction_table(preds, selected_models, split="test")
    selected_all = pd.concat([selected_preds, selected_preds_test], ignore_index=True)
    selected_all.to_csv(outdir / "07_selected_predictions_long.csv", index=False)

    paired = paired_tests_vs_lasso(metrics, split="test", horizon=1)
    paired.to_csv(outdir / "08_paired_tests_vs_lasso_h1.csv", index=False)

    # Inventory grid based on selected forecasting model per target/horizon.
    val_grid, val_trace = run_inventory_grid(panel, targets, selected_all, config, split="validation", start=config.val_start, end=config.val_end)
    val_grid.to_csv(outdir / "09_validation_inventory_grid_selected_forecasts.csv", index=False)
    selected_inventory = select_inventory_policy(val_grid)
    selected_inventory.to_csv(outdir / "10_selected_inventory_policy_by_validation.csv", index=False)

    test_target_inv, test_product_inv, test_trace = evaluate_selected_inventory(panel, targets, selected_all, selected_inventory, config)
    test_target_inv.to_csv(outdir / "11_test_inventory_target_level.csv", index=False)
    test_product_inv.to_csv(outdir / "12_test_inventory_product_level.csv", index=False)
    test_trace.to_csv(outdir / "13_test_inventory_daily_trace.csv", index=False)
    (outdir / "CORE_INVENTORY_SCOPE_README.txt").write_text(
        "EXPLORATORY CORE PIPELINE INVENTORY OUTPUTS - NOT MANUSCRIPT TABLES 3/4.\n"
        "Initialization uses only demand before the evaluation period; no test future actual.mean().\n"
        "The core engine expires stock BEFORE allocation, selects a model by validation wMAE,\n"
        "then grid-searches the buffer. The manuscript Table 3/4 engine is a SEPARATE policy\n"
        "simulation (fulfillment before expiry; Table 4 joint model+buffer validation selection).\n"
        "Never substitute 11/12_test_inventory_* values for manuscript Table 4.\n",
        encoding="utf-8",
    )

    decision = make_decision_table(diag, selected_models, selected_inventory, metrics)
    decision.to_csv(outdir / "14_sparse_aware_target_decision_table.csv", index=False)

    # Excel workbook for convenient review.
    with pd.ExcelWriter(outdir / "sparse_hurdle_analysis_summary.xlsx", engine="openpyxl") as writer:
        audit_df.to_excel(writer, sheet_name="data_audit", index=False)
        diag.to_excel(writer, sheet_name="sparse_diagnostics", index=False)
        summary.to_excel(writer, sheet_name="model_summary", index=False)
        selected_models.to_excel(writer, sheet_name="selected_models", index=False)
        paired.to_excel(writer, sheet_name="paired_tests", index=False)
        thresholds.to_excel(writer, sheet_name="hurdle_thresholds", index=False)
        selected_inventory.to_excel(writer, sheet_name="selected_inventory", index=False)
        test_target_inv.to_excel(writer, sheet_name="test_inventory_target", index=False)
        test_product_inv.to_excel(writer, sheet_name="test_inventory_product", index=False)
        decision.to_excel(writer, sheet_name="decision_table", index=False)

    write_caveats(outdir / "methodological_caveats_sparse_hurdle.txt", config)
    (outdir / "analysis_config.json").write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")

    print("\nSaved outputs to:", outdir)
    print("Key result files:")
    for name in [
        "01_sparse_intermittent_diagnostics.csv",
        "05_model_summary_by_split_horizon.csv",
        "06_selected_forecast_model_by_validation.csv",
        "10_selected_inventory_policy_by_validation.csv",
        "11_test_inventory_target_level.csv",
        "12_test_inventory_product_level.csv",
        "14_sparse_aware_target_decision_table.csv",
        "sparse_hurdle_analysis_summary.xlsx",
    ]:
        print(" -", name)

    return {
        "panel": panel,
        "metadata": metadata,
        "audit": audit_df,
        "sparse_diagnostics": diag,
        "predictions": preds,
        "metrics": metrics,
        "thresholds": thresholds,
        "summary": summary,
        "selected_models": selected_models,
        "paired_tests": paired,
        "selected_inventory": selected_inventory,
        "test_inventory_target": test_target_inv,
        "test_inventory_product": test_product_inv,
        "decision_table": decision,
    }



# ============================================================
# PART 2. Manuscript Figure 1 and Figure 2 generation
#
# These functions read the raw panel directly (via
# load_daily_panel_and_metadata, independent from Part 1's
# load_platelet_excel) and the Part 1 forecasting outputs, then
# reproduce the manuscript figures.
# ============================================================


# =========================================================
# 1. Config
# =========================================================
@dataclass
class FigureConfig:
    data_path: str
    output_dir: str

    # 최신 sparse/hurdle-aware(+extended models) 결과를 반영하려면 selected_predictions 사용
    selected_predictions_csv: str | None = None

    # 특정 모델(LASSO, Prophet 등)을 직접 Figure 2에 반영하고 싶을 때 사용
    all_predictions_csv: str | None = None
    figure2_model_name: str | None = None   # 예: "lasso", "prophet_augmented"

    # split dates
    train_start: str = "2019-03-01"
    train_end: str = "2024-02-29"
    val_start: str = "2024-03-01"
    val_end: str = "2024-08-31"
    test_start: str = "2024-09-01"
    test_end: str = "2025-02-28"

    # Figure 1 baseline for panel (C)
    baseline_start: str = "2019-03-01"
    baseline_end: str = "2019-06-30"

    # Figure 2 visible window
    recent_start: str = "2023-01-01"

    rolling_window: int = 30
    forecast_label: str = "Validation-selected forecast"

    # 자동 탐색이 실패하면 직접 넣을 수 있음
    hem_onc_cols_manual: list | None = None
    ga_cols_manual: list | None = None


# Note: renamed from META_COLS to avoid colliding with the differently-shaped
# META_COLS used by load_platelet_excel() in the forecasting pipeline above.
FIGURE_META_COLS = ["category", "variable", "cond1", "cond2", "description", "cond3"]

ABO_COLORS = {
    "A": "#1f77b4",   # blue
    "B": "#2ca02c",   # green
    "O": "#d62728",   # red
    "AB": "#9467bd",  # purple
}

PC_TARGETS = ["plt_transf_a", "plt_transf_b", "plt_transf_o", "plt_transf_ab"]
APC_TARGETS = ["aph_transf_a", "aph_transf_b", "aph_transf_o", "aph_transf_ab"]


# =========================================================
# 2. Data loader
# =========================================================
def _normalize_variable_name(name: str) -> str:
    name = str(name)
    if name.startswith("PLT_"):
        name = "plt_" + name[len("PLT_"):]
    if name.endswith("_O"):
        name = name[:-2] + "_o"
    return name


def _merge_aph_products(panel: pd.DataFrame) -> pd.DataFrame:
    """
    aph22_* + aph23_* -> aph_* 통합 컬럼 생성
    (원본은 유지하고 aph_*만 추가)
    """
    panel = panel.copy()
    flows = ["income", "transf", "transfused", "return", "expire"]
    abos = ["a", "b", "o", "ab"]

    for flow in flows:
        for abo in abos:
            c22 = f"aph22_{flow}_{abo}"
            c23 = f"aph23_{flow}_{abo}"
            new = f"aph_{flow}_{abo}"
            if new not in panel.columns and c22 in panel.columns and c23 in panel.columns:
                panel[new] = panel[c22].fillna(0) + panel[c23].fillna(0)

    for abo in abos:
        c22 = f"past_1wk_trans_aph22_{abo}"
        c23 = f"past_1wk_trans_aph23_{abo}"
        new = f"past_1wk_trans_aph_{abo}"
        if new not in panel.columns and c22 in panel.columns and c23 in panel.columns:
            panel[new] = panel[c22].fillna(0) + panel[c23].fillna(0)

    return panel


def load_daily_panel_and_metadata(xlsx_path: str):
    xlsx_path = Path(xlsx_path)
    df = pd.read_excel(xlsx_path, sheet_name=0)

    if len(df.columns) < 6:
        raise ValueError("Expected at least 6 metadata columns in the Excel file.")

    rename_map = dict(zip(df.columns[:6], FIGURE_META_COLS))
    df = df.rename(columns=rename_map)

    meta = df[FIGURE_META_COLS].copy()
    meta["variable"] = meta["variable"].astype(str).map(_normalize_variable_name)

    date_cols = [c for c in df.columns if c not in FIGURE_META_COLS]
    panel = df[date_cols].T
    panel.columns = meta["variable"].tolist()
    panel.index = pd.to_datetime(panel.index)
    panel.index.name = "date"
    panel = panel.sort_index()

    for c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

    panel = _merge_aph_products(panel)
    return panel, meta


# =========================================================
# 3. Metadata search helpers
# =========================================================
def _meta_text(meta: pd.DataFrame) -> pd.Series:
    txt = (
        meta[["category", "variable", "cond1", "cond2", "description", "cond3"]]
        .fillna("")
        .astype(str)
        .agg(" | ".join, axis=1)
        .str.lower()
    )
    return txt


def find_columns_by_keyword_groups(meta: pd.DataFrame, keyword_groups: list[list[str]]) -> list[str]:
    """
    keyword_groups = [
        ["hematology", "oncology", "inpatient"],
        ["hemato", "oncology", "ward"],
    ]
    각 group 안의 키워드는 모두 포함되어야 함.
    여러 group에서 찾은 결과를 union.
    """
    txt = _meta_text(meta)
    selected = set()

    for group in keyword_groups:
        mask = pd.Series(True, index=meta.index)
        for kw in group:
            mask &= txt.str.contains(str(kw).lower(), regex=False)
        selected.update(meta.loc[mask, "variable"].tolist())

    return sorted(selected)


def auto_detect_hem_onc_cols(meta: pd.DataFrame) -> list[str]:
    patterns = [
        ["hematology", "oncology", "inpatient"],
        ["hematology-oncology", "inpatient"],
        ["hemato", "oncology", "inpatient"],
        ["hematology", "oncology", "ward"],
        ["hemato", "oncology", "ward"],
    ]
    return find_columns_by_keyword_groups(meta, patterns)


def auto_detect_general_anesthesia_cols(meta: pd.DataFrame) -> list[str]:
    patterns = [
        ["general", "anesthesia"],
        ["general anaesthesia"],
        ["ga", "surgery"],
    ]
    return find_columns_by_keyword_groups(meta, patterns)


def pick_existing(panel: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in panel.columns]


def print_candidate_columns(meta: pd.DataFrame, keyword: str, top_n: int = 30):
    txt = _meta_text(meta)
    cand = meta.loc[txt.str.contains(keyword.lower(), regex=False), ["variable", "description", "category"]]
    print(f"\n[Candidates for keyword='{keyword}']")
    print(cand.head(top_n).to_string(index=False))


# =========================================================
# 4. Prediction loader
# =========================================================
def standardize_prediction_columns(pred: pd.DataFrame) -> pd.DataFrame:
    pred = pred.copy()
    pred.columns = [str(c).strip().lower() for c in pred.columns]

    alias_map = {
        "date": ["date", "ds"],
        "target": ["target", "series", "variable"],
        "actual": ["actual", "y_true", "truth", "y"],
        "pred": ["pred", "prediction", "y_pred", "forecast"],
        "model": ["model", "forecast_model", "selected_model"],
        "horizon": ["horizon", "fh"],
        "split": ["split"],
    }

    rename = {}
    for std_col, aliases in alias_map.items():
        found = [c for c in aliases if c in pred.columns]
        if found:
            rename[found[0]] = std_col

    pred = pred.rename(columns=rename)

    required = ["date", "target", "pred"]
    missing = [c for c in required if c not in pred.columns]
    if missing:
        raise ValueError(f"Prediction file is missing required columns: {missing}")

    pred["date"] = pd.to_datetime(pred["date"])
    if "target" in pred.columns:
        pred["target"] = pred["target"].astype(str)
    if "model" in pred.columns:
        pred["model"] = pred["model"].astype(str)
    return pred


def load_predictions(
    selected_predictions_csv: str | None = None,
    all_predictions_csv: str | None = None,
    model_name: str | None = None,
    horizon: int = 1,
) -> pd.DataFrame | None:
    """
    우선순위:
      1) selected_predictions_csv 사용
      2) all_predictions_csv + model_name 사용
    """
    if selected_predictions_csv:
        pred = pd.read_csv(selected_predictions_csv)
        pred = standardize_prediction_columns(pred)
        if "horizon" in pred.columns:
            pred = pred.loc[pred["horizon"] == horizon].copy()
        return pred

    if all_predictions_csv and model_name:
        pred = pd.read_csv(all_predictions_csv)
        pred = standardize_prediction_columns(pred)
        if "horizon" in pred.columns:
            pred = pred.loc[pred["horizon"] == horizon].copy()
        if "model" not in pred.columns:
            raise ValueError("all_predictions_csv exists, but 'model' column not found.")
        pred = pred.loc[pred["model"].str.lower() == model_name.lower()].copy()
        return pred

    return None


def aggregate_predictions_by_product(pred: pd.DataFrame, targets: list[str]) -> pd.DataFrame:
    """
    target별 예측을 product 단위로 합산
    반환 columns: [date, actual, pred]
    """
    sub = pred.loc[pred["target"].isin(targets)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["date", "actual", "pred"])

    agg_dict = {"pred": "sum"}
    if "actual" in sub.columns:
        agg_dict["actual"] = "sum"

    out = sub.groupby("date", as_index=False).agg(agg_dict)
    return out


# =========================================================
# 5. Figure 1
# =========================================================
def _plot_abo_panel(ax, panel, target_cols, panel_letter, ymax=None):
    for blood_type, col in zip(["A", "B", "O", "AB"], target_cols):
        if col not in panel.columns:
            warnings.warn(f"{col} not found. Skipping.")
            continue

        s = panel[col].fillna(0)
        rm = s.rolling(30, min_periods=1).mean()

        # faint daily series
        ax.vlines(s.index, 0, s.values, color=ABO_COLORS[blood_type], alpha=0.08, linewidth=0.6)
        # rolling mean
        ax.plot(rm.index, rm.values, color=ABO_COLORS[blood_type], lw=1.8, label=f"Type {blood_type}")

    ax.set_ylabel("Units/day")
    if ymax is not None:
        ax.set_ylim(0, ymax)
    ax.legend(loc="upper left", ncol=2, frameon=True, fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.text(-0.06, 0.98, f"({panel_letter})", transform=ax.transAxes,
            fontsize=22, va="top", ha="right")


def _relative_series(series: pd.Series, baseline_start: str, baseline_end: str, rolling_window: int = 30):
    rm = series.rolling(rolling_window, min_periods=1).mean()
    baseline = rm.loc[baseline_start:baseline_end].mean()
    if pd.isna(baseline) or baseline == 0:
        baseline = 1.0
    return rm / baseline


def plot_figure1_temporal_patterns(
    panel: pd.DataFrame,
    meta: pd.DataFrame,
    cfg: FigureConfig,
):
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    hem_onc_cols = cfg.hem_onc_cols_manual or auto_detect_hem_onc_cols(meta)

    hem_onc_cols = pick_existing(panel, hem_onc_cols)

    if len(hem_onc_cols) == 0:
        print("[WARN] Hematology-oncology inpatient columns were not auto-detected.")
        print_candidate_columns(meta, "hematology")
        print_candidate_columns(meta, "oncology")
        raise ValueError(
            "No hematology-oncology inpatient columns found automatically. "
            "Please set cfg.hem_onc_cols_manual."
        )

    total_plt = panel[pick_existing(panel, PC_TARGETS + APC_TARGETS)].fillna(0).sum(axis=1)
    hem_onc = panel[hem_onc_cols].fillna(0).sum(axis=1)

    rel_total = _relative_series(total_plt, cfg.baseline_start, cfg.baseline_end, cfg.rolling_window)
    rel_hem_onc = _relative_series(hem_onc, cfg.baseline_start, cfg.baseline_end, cfg.rolling_window)

    fig, axes = plt.subplots(
        3, 1, figsize=(11.5, 8.8), sharex=True,
        gridspec_kw={"height_ratios": [1.0, 1.0, 0.9], "hspace": 0.15}
    )

    _plot_abo_panel(axes[0], panel, PC_TARGETS, panel_letter="A")
    _plot_abo_panel(axes[1], panel, APC_TARGETS, panel_letter="B")

    axes[2].plot(rel_total.index, rel_total.values, color="#1f4aa8", lw=1.8, label="Total platelets issued")
    axes[2].plot(rel_hem_onc.index, rel_hem_onc.values, color="#8b1d1d", lw=1.8, label="Hematology-oncology inpatient")
    axes[2].axhline(1.0, color="gray", lw=1.0, ls=":")
    # Test start marker (the Figure 1 legend describes this vertical dashed line).
    for ax in axes:
        ax.axvline(pd.Timestamp(cfg.test_start), color="black", ls="--", lw=1.0, alpha=0.55)
    axes[2].set_ylabel("Relative")
    axes[2].set_xlabel("Date")
    axes[2].legend(loc="upper left", frameon=True, fontsize=9)
    axes[2].grid(True, alpha=0.25)
    axes[2].text(-0.06, 0.98, "(C)", transform=axes[2].transAxes,
                 fontsize=22, va="top", ha="right")

    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    outpath = output_dir / "Figure1_temporal_patterns.png"
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

    return outpath


# =========================================================
# 6. Figure 2
# =========================================================
def _plot_surge_panel(
    ax,
    product_name: str,
    actual_series: pd.Series,
    forecast_series: pd.Series | None,
    cfg: FigureConfig,
    panel_letter: str,
    color_bar: str,
    color_roll: str,
    forecast_color: str,
):
    recent = actual_series.loc[cfg.recent_start:cfg.test_end].copy()
    roll = recent.rolling(cfg.rolling_window, min_periods=1).mean()

    train_mean = actual_series.loc[cfg.train_start:cfg.train_end].mean()
    test_mean = actual_series.loc[cfg.test_start:cfg.test_end].mean()
    surge_ratio = (test_mean / train_mean) if train_mean > 0 else np.nan

    # background shading
    ax.axvspan(pd.to_datetime(cfg.recent_start), pd.to_datetime(cfg.test_start), color="#dbe6f3", alpha=0.35)
    ax.axvspan(pd.to_datetime(cfg.test_start), pd.to_datetime(cfg.test_end), color="#edd7d2", alpha=0.45)

    # actual daily demand
    ax.bar(recent.index, recent.values, width=1.0, color=color_bar, alpha=0.18, label="Actual daily demand", edgecolor="none")

    # 30-day rolling mean
    ax.plot(roll.index, roll.values, color=color_roll, lw=1.8, label=f"{cfg.rolling_window}-day rolling mean")

    # historical mean
    ax.axhline(train_mean, color="gray", lw=1.5, ls="--", label=f"Historical-mean ({train_mean:.1f} units/day)")

    # forecast (test period only)
    if forecast_series is not None and len(forecast_series) > 0:
        fc = forecast_series.loc[cfg.test_start:cfg.test_end].copy()
        ax.plot(fc.index, fc.values, color=forecast_color, lw=2.0, label=cfg.forecast_label)

    # test start marker
    ax.axvline(pd.to_datetime(cfg.test_start), color="black", lw=1.1, ls=":")

    # text
    ax.text(-0.06, 0.98, f"({panel_letter})", transform=ax.transAxes,
            fontsize=20, va="top", ha="right")

    if pd.notna(surge_ratio):
        y_min, y_max = ax.get_ylim()
        ax.text(
            pd.to_datetime(cfg.test_start) + pd.Timedelta(days=12),
            y_min + (y_max - y_min) * 0.08,
            f"×{surge_ratio:.1f} demand surge",
            fontsize=12,
            color="black",
        )

    ax.set_ylabel(f"Daily {product_name} issued")
    ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_figure2_demand_surge(
    panel: pd.DataFrame,
    cfg: FigureConfig,
    pred_df: pd.DataFrame | None = None,
):
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    actual_pc = panel[pick_existing(panel, PC_TARGETS)].fillna(0).sum(axis=1)
    actual_apc = panel[pick_existing(panel, APC_TARGETS)].fillna(0).sum(axis=1)

    pred_pc = None
    pred_apc = None

    if pred_df is not None:
        agg_pc = aggregate_predictions_by_product(pred_df, PC_TARGETS)
        agg_apc = aggregate_predictions_by_product(pred_df, APC_TARGETS)

        if not agg_pc.empty:
            pred_pc = agg_pc.set_index("date")["pred"].sort_index()
        if not agg_apc.empty:
            pred_apc = agg_apc.set_index("date")["pred"].sort_index()

    # 수정된 부분: hspace를 직접 넣지 않고 gridspec_kw 사용
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(11.5, 8.5),
        sharex=True,
        gridspec_kw={"hspace": 0.17}
    )

    _plot_surge_panel(
        ax=axes[0],
        product_name="PC",
        actual_series=actual_pc,
        forecast_series=pred_pc,
        cfg=cfg,
        panel_letter="A",
        color_bar="#86a9d7",
        color_roll="#2f6fab",
        forecast_color="#d62728",
    )

    _plot_surge_panel(
        ax=axes[1],
        product_name="APC",
        actual_series=actual_apc,
        forecast_series=pred_apc,
        cfg=cfg,
        panel_letter="B",
        color_bar="#e6a3a3",
        color_roll="#8b1d1d",
        forecast_color="#e83e8c",
    )

    # 각 panel별 legend를 따로 생성
    axes[0].legend(loc="upper left", fontsize=9, frameon=True)
    axes[1].legend(loc="upper left", fontsize=9, frameon=True)

    axes[1].set_xlabel("Date")

    outpath = output_dir / "Figure2_demand_surge.png"

    # tight_layout은 gridspec hspace와 충돌할 수 있으므로 rect만 가볍게 적용
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

    return outpath


def plot_figure2_product_abo_forecasts(
    panel: pd.DataFrame,
    cfg: FigureConfig,
    pred_df: pd.DataFrame | None = None,
    save_dpi: int = 300,
) -> Path:
    """Plot eight product-ABO panels using LASSO and MA7 one-day forecasts.

    IMPORTANT: 02_all_predictions_long.csv stores ORIGIN dates, *not* the dates
    of the outcomes. Its row at origin t forecasts y(t + horizon). Align with
    actual demand by target_date = date + horizon; then restrict target_date
    to the complete independent test interval. The prediction for the first
    test day originates on the final validation day, so retain both validation
    and test origin rows. This changes only figure display, NOT model metrics.
    """
    import matplotlib.dates as mdates
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    if pred_df is None:
        if not cfg.all_predictions_csv:
            raise ValueError("Figure 2 requires the all-predictions CSV file")
        pred_df = pd.read_csv(cfg.all_predictions_csv)
    pred_df = standardize_prediction_columns(pred_df)
    needed = {"date", "target", "model", "split", "horizon", "pred"}
    missing_columns = needed.difference(pred_df.columns)
    if missing_columns:
        raise ValueError(f"Figure 2 prediction table lacks: {sorted(missing_columns)}")

    pred_df["model"] = pred_df["model"].astype(str).str.lower().str.strip()
    pred_df["target"] = pred_df["target"].astype(str).str.lower().str.strip()
    pred_df["split"] = pred_df["split"].astype(str).str.lower().str.strip()
    pred_df["date"] = pd.to_datetime(pred_df["date"], errors="coerce").dt.normalize()
    pred_df["horizon"] = pd.to_numeric(pred_df["horizon"], errors="coerce")
    if pred_df["date"].isna().any():
        raise ValueError("Figure 2 predictions contain unparseable origin dates")

    # The central fix: move every prediction from its ORIGIN date to the
    # actual DATE BEING FORECAST. Do not filter split='test' before this.
    pred_df["target_date"] = pred_df["date"] + pd.to_timedelta(
        pred_df["horizon"], unit="D"
    )
    pred_df = pred_df.loc[
        pred_df["model"].isin(["lasso", "moving_average_7d"])
        & pred_df["split"].isin(["validation", "test"])
        & pred_df["horizon"].eq(1)
        & pred_df["target_date"].between(
            pd.Timestamp(cfg.test_start), pd.Timestamp(cfg.test_end)
        )
    ].copy()

    targets_order = [
        ("PC-A", "plt_transf_a"), ("PC-B", "plt_transf_b"),
        ("PC-O", "plt_transf_o"), ("PC-AB", "plt_transf_ab"),
        ("APC-A", "aph_transf_a"), ("APC-B", "aph_transf_b"),
        ("APC-O", "aph_transf_o"), ("APC-AB", "aph_transf_ab"),
    ]
    test_days = pd.date_range(cfg.test_start, cfg.test_end, freq="D")
    if test_days.empty:
        raise ValueError("Figure 2 test period is empty")
    if pred_df.duplicated(["target", "model", "target_date"]).any():
        bad = pred_df.loc[
            pred_df.duplicated(["target", "model", "target_date"], keep=False),
            ["date", "target_date", "target", "model", "split"],
        ].head(10)
        raise ValueError(f"Duplicated Figure 2 target-date predictions:\n{bad}")

    plot_data = []
    for label, target in targets_order:
        if target not in panel.columns:
            raise KeyError(f"Cannot plot {label}: missing {target!r} in the panel")
        actual = pd.to_numeric(panel[target], errors="coerce").sort_index()
        actual.index = pd.to_datetime(actual.index).normalize()
        test_actual = actual.reindex(test_days)
        if test_actual.isna().any():
            bad_days = test_actual.index[test_actual.isna()]
            raise ValueError(f"Actual {label} values missing on {list(bad_days[:5])}")
        train_mean = float(actual.loc[cfg.train_start:cfg.train_end].mean())
        if not np.isfinite(train_mean):
            raise ValueError(f"Training mean missing for {label}")
        rolling30 = actual.rolling(
            window=cfg.rolling_window, min_periods=cfg.rolling_window
        ).mean().reindex(test_days)
        if rolling30.isna().any():
            raise ValueError(f"Insufficient prior history for {label}'s rolling mean")

        forecasts = {}
        for model in ("lasso", "moving_average_7d"):
            part = pred_df.loc[
                pred_df["target"].eq(target) & pred_df["model"].eq(model),
                ["target_date", "date", "pred", "split"],
            ].copy()
            if part.empty:
                raise ValueError(f"No {model} predictions for {label}")
            forecast = pd.to_numeric(
                part.set_index("target_date")["pred"], errors="coerce"
            ).reindex(test_days)
            if forecast.isna().any():
                bad_days = test_days[forecast.isna()]
                raise ValueError(
                    f"{label}: {len(bad_days)} actual target dates lack {model} "
                    f"forecasts: {[str(x.date()) for x in bad_days[:8]]}. "
                    "Check that both validation and test origin rows are present."
                )
            # Cross-check prediction CSV alignment against the raw actuals.
            if "actual" in pred_df.columns:
                recorded = pd.to_numeric(
                    pred_df.loc[
                        pred_df["target"].eq(target) & pred_df["model"].eq(model),
                        ["target_date", "actual"],
                    ].set_index("target_date")["actual"], errors="coerce"
                ).reindex(test_days)
                mismatch = recorded.notna() & ~np.isclose(
                    recorded.to_numpy(dtype=float),
                    test_actual.to_numpy(dtype=float),
                    equal_nan=True,
                )
                if mismatch.any():
                    bad_days = test_days[mismatch]
                    raise ValueError(
                        f"{label} {model}: saved actual values do not match "
                        f"raw panel on {list(bad_days[:5])}. "
                        "Inspect date/target alignment before publishing."
                    )
            forecasts[model] = forecast
        plot_data.append((label, test_actual, rolling30, train_mean, forecasts))

    colors = dict(actual="#9fc4df", rolling="#2474a3", lasso="#ef8b24", ma7="#329b49")
    fig, axes = plt.subplots(4, 2, figsize=(15, 12.6), sharex=True)
    top_pc = max(max(float(a.max()), float(r.max()), float(f["lasso"].max()),
                     float(f["moving_average_7d"].max())) for _, a, r, _, f in plot_data[:4])
    top_apc = max(max(float(a.max()), float(r.max()), float(f["lasso"].max()),
                      float(f["moving_average_7d"].max())) for _, a, r, _, f in plot_data[4:])
    for i, (label, actual, rolling30, train_mean, forecasts) in enumerate(plot_data):
        ax = axes.flat[i]
        ax.bar(test_days, actual.values, width=0.92, color=colors["actual"],
               alpha=0.84, linewidth=0, zorder=1)
        ax.plot(test_days, rolling30.values, color=colors["rolling"], lw=1.9, zorder=3)
        ax.axhline(train_mean, color="black", lw=1.6, ls="--", zorder=2)
        ax.plot(test_days, forecasts["lasso"].values, color=colors["lasso"],
                lw=1.45, zorder=4)
        ax.plot(test_days, forecasts["moving_average_7d"].values,
                color=colors["ma7"], lw=1.45, zorder=5)
        ax.set_title(label, fontsize=12, pad=5)
        ax.set_ylabel("Units/day", fontsize=10)
        ax.set_ylim(bottom=0, top=max(1, (top_pc if i < 4 else top_apc) * 1.08))
        ax.set_xlim(pd.Timestamp(cfg.test_start) - pd.Timedelta(days=9),
                    pd.Timestamp(cfg.test_end) + pd.Timedelta(days=9))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        ax.tick_params(axis="both", labelsize=9)
        ax.tick_params(axis="x", labelbottom=True)
        ax.grid(False)

    legend_handles = [
        Line2D([0], [0], color=colors["rolling"], lw=2, label=f"{cfg.rolling_window}-day rolling mean"),
        Line2D([0], [0], color="black", lw=1.6, ls="--", label="Training mean"),
        Line2D([0], [0], color=colors["lasso"], lw=1.6, label="LASSO"),
        Line2D([0], [0], color=colors["ma7"], lw=1.6, label="MA7"),
        Patch(facecolor=colors["actual"], edgecolor="none", label="Actual"),
    ]
    fig.suptitle("Figure 2. Product–ABO–specific platelet demand and forecasts",
                 fontsize=15, y=0.995)
    fig.legend(handles=legend_handles, loc="lower center", ncol=5,
               frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.005))
    fig.subplots_adjust(left=0.07, right=0.985, top=0.945, bottom=0.075,
                        hspace=0.28, wspace=0.12)
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    outpath = output_dir / "Figure2_product_ABO_LASSO_MA7.png"
    fig.savefig(outpath, dpi=save_dpi, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"[Figure 2] Saved {len(test_days)} target-day forecasts per model/ABO target.")
    return outpath


def safe_filename(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[^a-zA-Z0-9가-힣_+-]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


ABO_INFO = {
    "a":  {"label": "Type A",  "color": "#1f77b4"},
    "b":  {"label": "Type B",  "color": "#2ca02c"},
    "o":  {"label": "Type O",  "color": "#d62728"},
    "ab": {"label": "Type AB", "color": "#9467bd"},
}


def target_to_product_abo(target: str):
    """
    Example:
      plt_transf_a  -> PC, a
      aph_transf_ab -> APC, ab
    """
    target = str(target).lower()

    if target.startswith("plt_"):
        product = "PC"
    elif target.startswith("aph_"):
        product = "APC"
    else:
        product = "Unknown"

    abo = target.split("_")[-1]
    return product, abo


def plot_model_trends_by_blood_type(
    panel: pd.DataFrame,
    pred_df_model: pd.DataFrame,
    model_name: str,
    output_dir: str,
    recent_start: str = "2023-01-01",
    train_start: str = "2019-03-01",
    train_end: str = "2024-02-29",
    test_start: str = "2024-09-01",
    test_end: str = "2025-02-28",
    rolling_window: int = 30,
    save_dpi: int = 300,
):
    """
    Create one 2 x 4 figure for one model.
    Top row: PC-A, PC-B, PC-O, PC-AB
    Bottom row: APC-A, APC-B, APC-O, APC-AB
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pred_df_model = pred_df_model.copy()
    pred_df_model["date"] = pd.to_datetime(pred_df_model["date"])
    pred_df_model["target"] = pred_df_model["target"].astype(str)

    targets_order = [
        "plt_transf_a", "plt_transf_b", "plt_transf_o", "plt_transf_ab",
        "aph_transf_a", "aph_transf_b", "aph_transf_o", "aph_transf_ab",
    ]

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(18, 7.8),
        sharex=True,
        gridspec_kw={"hspace": 0.28, "wspace": 0.22}
    )

    axes = axes.flatten()

    for ax, target in zip(axes, targets_order):
        product, abo = target_to_product_abo(target)
        abo_label = ABO_INFO.get(abo, {}).get("label", abo.upper())
        color = ABO_INFO.get(abo, {}).get("color", "tab:blue")

        if target not in panel.columns:
            ax.set_title(f"{product}-{abo_label}\nTarget missing", fontsize=10)
            ax.axis("off")
            continue

        actual = panel[target].fillna(0).copy()
        actual_recent = actual.loc[recent_start:test_end]
        actual_roll = actual_recent.rolling(rolling_window, min_periods=1).mean()
        train_mean = actual.loc[train_start:train_end].mean()

        pred_target = pred_df_model.loc[pred_df_model["target"] == target].copy()

        if pred_target.empty:
            forecast_series = None
        else:
            forecast_series = (
                pred_target
                .set_index("date")["pred"]
                .sort_index()
                .loc[test_start:test_end]
            )

        # background shading
        ax.axvspan(
            pd.to_datetime(recent_start),
            pd.to_datetime(test_start),
            color="#dbe6f3",
            alpha=0.35
        )
        ax.axvspan(
            pd.to_datetime(test_start),
            pd.to_datetime(test_end),
            color="#edd7d2",
            alpha=0.45
        )

        # actual daily demand
        ax.bar(
            actual_recent.index,
            actual_recent.values,
            width=1.0,
            color=color,
            alpha=0.16,
            edgecolor="none",
            label="Actual daily demand"
        )

        # actual rolling mean
        ax.plot(
            actual_roll.index,
            actual_roll.values,
            color=color,
            lw=1.8,
            label=f"Actual {rolling_window}-day rolling mean"
        )

        # train historical mean
        ax.axhline(
            train_mean,
            color="gray",
            lw=1.1,
            ls="--",
            label="Training historical mean"
        )

        # forecast
        if forecast_series is not None and len(forecast_series) > 0:
            ax.plot(
                forecast_series.index,
                forecast_series.values,
                color="black",
                lw=1.7,
                label=f"{model_name} forecast"
            )

        # test start marker
        ax.axvline(
            pd.to_datetime(test_start),
            color="black",
            lw=0.9,
            ls=":"
        )

        ax.set_title(f"{product}-{abo_label}", fontsize=11)
        ax.grid(True, alpha=0.25)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if ax in axes[0:4]:
            ax.tick_params(labelbottom=False)

    axes[0].set_ylabel("Daily PC issued\n(units/day)")
    axes[4].set_ylabel("Daily APC issued\n(units/day)")

    # Common legend from first axis
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=True,
        fontsize=9,
        bbox_to_anchor=(0.5, 1.02)
    )

    fig.suptitle(
        f"ABO-specific platelet demand and {model_name} forecast",
        fontsize=15,
        y=1.08
    )

    outpath = output_dir / f"ABO_trends_{safe_filename(model_name)}.png"
    plt.savefig(outpath, dpi=save_dpi, bbox_inches="tight")
    plt.show()

    return outpath


# =========================================================
# Function: Product-specific ABO trend figure
# =========================================================

def plot_product_abo_trends_for_model(
    panel: pd.DataFrame,
    pred_df_model: pd.DataFrame,
    model_name: str,
    product: str,
    output_dir: str,
    recent_start: str = "2023-01-01",
    train_start: str = "2019-03-01",
    train_end: str = "2024-02-29",
    test_start: str = "2024-09-01",
    test_end: str = "2025-02-28",
    rolling_window: int = 30,
    save_dpi: int = 300,
):
    """
    product = "PC" or "APC"
    Creates one 1 x 4 figure by ABO type.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    product = product.upper()

    if product == "PC":
        targets_order = ["plt_transf_a", "plt_transf_b", "plt_transf_o", "plt_transf_ab"]
        y_label = "Daily PC issued\n(units/day)"
    elif product == "APC":
        targets_order = ["aph_transf_a", "aph_transf_b", "aph_transf_o", "aph_transf_ab"]
        y_label = "Daily APC issued\n(units/day)"
    else:
        raise ValueError("product must be either 'PC' or 'APC'.")

    pred_df_model = pred_df_model.copy()
    pred_df_model["date"] = pd.to_datetime(pred_df_model["date"])
    pred_df_model["target"] = pred_df_model["target"].astype(str)

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(18, 3.8),
        sharex=True,
        gridspec_kw={"wspace": 0.22}
    )

    for ax, target in zip(axes, targets_order):
        _, abo = target_to_product_abo(target)
        abo_label = ABO_INFO.get(abo, {}).get("label", abo.upper())
        color = ABO_INFO.get(abo, {}).get("color", "tab:blue")

        if target not in panel.columns:
            ax.set_title(f"{product}-{abo_label}\nTarget missing", fontsize=10)
            ax.axis("off")
            continue

        actual = panel[target].fillna(0)
        actual_recent = actual.loc[recent_start:test_end]
        actual_roll = actual_recent.rolling(rolling_window, min_periods=1).mean()
        train_mean = actual.loc[train_start:train_end].mean()

        pred_target = pred_df_model.loc[pred_df_model["target"] == target].copy()

        if pred_target.empty:
            forecast_series = None
        else:
            forecast_series = (
                pred_target
                .set_index("date")["pred"]
                .sort_index()
                .loc[test_start:test_end]
            )

        ax.axvspan(pd.to_datetime(recent_start), pd.to_datetime(test_start),
                   color="#dbe6f3", alpha=0.35)
        ax.axvspan(pd.to_datetime(test_start), pd.to_datetime(test_end),
                   color="#edd7d2", alpha=0.45)

        ax.bar(
            actual_recent.index,
            actual_recent.values,
            width=1.0,
            color=color,
            alpha=0.16,
            edgecolor="none",
            label="Actual daily demand"
        )

        ax.plot(
            actual_roll.index,
            actual_roll.values,
            color=color,
            lw=1.8,
            label=f"Actual {rolling_window}-day rolling mean"
        )

        ax.axhline(
            train_mean,
            color="gray",
            lw=1.1,
            ls="--",
            label="Training historical mean"
        )

        if forecast_series is not None and len(forecast_series) > 0:
            ax.plot(
                forecast_series.index,
                forecast_series.values,
                color="black",
                lw=1.7,
                label=f"{model_name} forecast"
            )

        ax.axvline(pd.to_datetime(test_start), color="black", lw=0.9, ls=":")
        ax.set_title(f"{product}-{abo_label}", fontsize=11)
        ax.grid(True, alpha=0.25)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel(y_label)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=True,
        fontsize=9,
        bbox_to_anchor=(0.5, 1.08)
    )

    fig.suptitle(
        f"{product} ABO-specific demand and {model_name} forecast",
        fontsize=14,
        y=1.18
    )

    outpath = output_dir / f"{product}_ABO_trends_{safe_filename(model_name)}.png"
    plt.savefig(outpath, dpi=save_dpi, bbox_inches="tight")
    plt.show()

    return outpath



# ============================================================
# PART 3. Manuscript Table 1, 2, 4, and 5 generation
# ============================================================


try:
    from docx import Document
    from docx.shared import Pt, Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
except ImportError:
    Document = None


def read_csv_if_exists(path):
    path = Path(path)
    if not path.exists():
        print(f"[WARNING] File not found: {path}")
        return None
    return pd.read_csv(path)


def clean_model_name(model):
    """
    Convert internal model names into manuscript-friendly names.
    """
    mapping = {
        "historical_mean": "Historical mean",
        "seasonal_naive": "Seasonal naïve",
        "moving_average_7d": "7-day moving average",
        "moving_average_14d": "14-day moving average",
        "lasso": "LASSO",
        "ridge": "Ridge",
        "lightgbm": "LightGBM",
        "prophet_augmented": "Augmented Prophet",
        "lstm": "LSTM",
        "blend_lasso_ma7_50_50": "LASSO–MA7 blend",
        "blend_lasso_ma14_50_50": "LASSO–MA14 blend",
        "hurdle_lasso_soft": "Hurdle LASSO, soft",
        "hurdle_lasso_thresholded": "Hurdle LASSO, thresholded",
        "hurdle_rolling14_soft": "Hurdle rolling-14, soft",
        "hurdle_rolling14_thresholded": "Hurdle rolling-14, thresholded",
    }
    return mapping.get(str(model), str(model))


def clean_target_name(target):
    """
    Convert internal target names into manuscript-friendly target names.
    """
    target = str(target).lower()

    product = "APC" if target.startswith("aph_") else "PC"
    abo = target.split("_")[-1].upper()

    return f"{product}-{abo}"


def model_family(model):
    """
    Assign model family for manuscript table.
    """
    model = str(model).lower()

    if model in ["historical_mean", "seasonal_naive", "moving_average_7d", "moving_average_14d"]:
        return "Baseline / time-series"
    elif model in ["lasso", "ridge", "lightgbm", "prophet_augmented", "lstm"]:
        return "Regression / ML / DL"
    elif model.startswith("blend_"):
        return "Adaptive blend"
    elif model.startswith("hurdle_"):
        return "Sparse-aware hurdle"
    else:
        return "Other"


def fmt_num(x, digits=2):
    """
    Format numeric values for manuscript tables.
    """
    if pd.isna(x):
        return "-"
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def fmt_pct(x, digits=1):
    """
    Format proportion as percentage.
    """
    if pd.isna(x):
        return "-"
    try:
        return f"{100 * float(x):.{digits}f}"
    except Exception:
        return str(x)


def reorder_target_columns(df):
    """
    Reorder product–ABO target columns if present.
    """
    target_order = [
        "PC-A", "PC-B", "PC-O", "PC-AB",
        "APC-A", "APC-B", "APC-O", "APC-AB"
    ]

    existing_targets = [c for c in target_order if c in df.columns]
    other_cols = [c for c in df.columns if c not in target_order]

    return df[other_cols[:2] + existing_targets + other_cols[2:]]


# ============================================================
# 2. Table 1: Expanded model performance table
# ============================================================

def create_table1_model_performance(
    metrics_csv,
    split="test",
    horizon=1,
    metric_for_target="mae",
    rank_metric="weighted_mae",
):
    """
    Create Table 1.
    Expanded performance comparison of forecasting models across
    eight product–ABO target series.

    Required input:
        03_all_metrics_by_target.csv

    Expected columns:
        split, target, horizon, model, mae, weighted_mae, mase

    Returns:
        pd.DataFrame
    """

    metrics = pd.read_csv(metrics_csv)
    metrics.columns = [str(c).strip().lower() for c in metrics.columns]

    required = {"split", "target", "horizon", "model", metric_for_target}
    missing = required - set(metrics.columns)

    if missing:
        raise ValueError(f"Missing required columns in metrics file: {missing}")

    df = metrics[
        (metrics["split"].astype(str).str.lower() == split.lower()) &
        (metrics["horizon"].astype(int) == int(horizon))
    ].copy()

    if df.empty:
        raise ValueError("No rows found for the selected split and horizon.")

    df["Target"] = df["target"].map(clean_target_name)
    df["Model"] = df["model"].map(clean_model_name)
    df["Model family"] = df["model"].map(model_family)

    # Target-level metric pivot
    pivot = (
        df.pivot_table(
            index=["Model family", "Model"],
            columns="Target",
            values=metric_for_target,
            aggfunc="mean"
        )
        .reset_index()
    )

    # Summary metrics
    summary_cols = {}

    if "mae" in df.columns:
        summary_cols["Mean MAE"] = ("mae", "mean")

    if "weighted_mae" in df.columns:
        summary_cols["Mean weighted MAE"] = ("weighted_mae", "mean")

    if "mase" in df.columns:
        summary_cols["Mean MASE"] = ("mase", "mean")

    if "occurrence_f1" in df.columns:
        summary_cols["Mean occurrence F1"] = ("occurrence_f1", "mean")

    if "high_f1" in df.columns:
        summary_cols["Mean high-demand F1"] = ("high_f1", "mean")

    summary = (
        df.groupby(["Model family", "Model"], as_index=False)
        .agg(**summary_cols)
    )

    # Mean rank across targets
    if rank_metric in df.columns:
        df["_rank"] = df.groupby("Target")[rank_metric].rank(method="average", ascending=True)
        rank_df = (
            df.groupby(["Model family", "Model"], as_index=False)
            .agg(**{"Mean rank": ("_rank", "mean")})
        )
        summary = summary.merge(rank_df, on=["Model family", "Model"], how="left")

    table = pivot.merge(summary, on=["Model family", "Model"], how="left")

    # Reorder target columns
    target_order = [
        "PC-A", "PC-B", "PC-O", "PC-AB",
        "APC-A", "APC-B", "APC-O", "APC-AB"
    ]

    ordered_cols = ["Model family", "Model"]
    ordered_cols += [c for c in target_order if c in table.columns]
    ordered_cols += [c for c in table.columns if c not in ordered_cols]

    table = table[ordered_cols]

    # Sort models by family and mean weighted MAE or mean MAE
    sort_col = "Mean weighted MAE" if "Mean weighted MAE" in table.columns else "Mean MAE"

    if sort_col in table.columns:
        table = table.sort_values(["Model family", sort_col], ascending=[True, True])

    # Format numeric columns
    formatted = table.copy()

    for col in formatted.columns:
        if col not in ["Model family", "Model"]:
            formatted[col] = formatted[col].map(lambda x: fmt_num(x, 2))

    return formatted


# ============================================================
# 3. Table 2: Sparse and intermittent demand diagnostics
# ============================================================

def create_table2_sparse_diagnostics(diagnostics_csv):
    """
    Create Table 2.
    Sparse and intermittent demand diagnostics for product–ABO targets.

    Required input:
        01_sparse_intermittent_diagnostics.csv

    Expected columns:
        target, product, mean_demand, median_demand, zero_rate,
        nonzero_days, ADI_average_demand_interval,
        CV2_nonzero_demand, sbc_classification,
        force_hurdle_candidates, method_recommendation

    Returns:
        pd.DataFrame
    """

    diag = pd.read_csv(diagnostics_csv)
    diag.columns = [str(c).strip() for c in diag.columns]

    lower_map = {c.lower(): c for c in diag.columns}

    def get_col(possible_names):
        for name in possible_names:
            if name.lower() in lower_map:
                return lower_map[name.lower()]
        return None

    target_col = get_col(["target"])
    mean_col = get_col(["mean_demand"])
    median_col = get_col(["median_demand"])
    zero_col = get_col(["zero_rate"])
    nonzero_col = get_col(["nonzero_days"])
    adi_col = get_col(["ADI_average_demand_interval", "adi_average_demand_interval", "ADI"])
    cv2_col = get_col(["CV2_nonzero_demand", "cv2_nonzero_demand", "CV2"])
    class_col = get_col(["sbc_classification", "classification", "demand_class"])
    hurdle_col = get_col(["force_hurdle_candidates", "sparse_aware_candidate"])
    rec_col = get_col(["method_recommendation", "recommendation"])

    if target_col is None:
        raise ValueError("Diagnostics file must contain a target column.")

    out = pd.DataFrame()
    out["Target"] = diag[target_col].map(clean_target_name)

    if mean_col:
        out["Mean demand, units/day"] = diag[mean_col].map(lambda x: fmt_num(x, 2))

    if median_col:
        out["Median demand, units/day"] = diag[median_col].map(lambda x: fmt_num(x, 2))

    if zero_col:
        out["Zero-demand days, %"] = diag[zero_col].map(lambda x: fmt_pct(x, 1))

    if nonzero_col:
        out["Nonzero days, n"] = diag[nonzero_col].astype("Int64").astype(str)

    if adi_col:
        out["Average demand interval"] = diag[adi_col].map(lambda x: fmt_num(x, 3))

    if cv2_col:
        out["CV² among nonzero demand"] = diag[cv2_col].map(lambda x: fmt_num(x, 3))

    if class_col:
        out["Demand class"] = diag[class_col].astype(str)

    if hurdle_col:
        out["Sparse-aware candidate"] = diag[hurdle_col].map(
            lambda x: "Yes" if bool(x) else "No"
        )

    if rec_col:
        out["Recommended modeling approach"] = diag[rec_col].astype(str)

    target_order = [
        "PC-A", "PC-B", "PC-O", "PC-AB",
        "APC-A", "APC-B", "APC-O", "APC-AB"
    ]

    out["_order"] = out["Target"].map({t: i for i, t in enumerate(target_order)})
    out = out.sort_values("_order").drop(columns="_order")

    return out


# ============================================================
# 4. Table 4: Simulated inventory outcomes
# ============================================================

def create_table4_inventory_outcomes(
    product_inventory_csv=None,
    target_inventory_csv=None,
    policy_label="Validation-selected policy",
):
    """
    Create Table 4.
    Simulated inventory outcomes at product level.

    Preferred input:
        12_test_inventory_product_level.csv

    Alternative input:
        11_test_inventory_target_level.csv

    Expected columns:
        product, demand, procured, fulfilled, unmet, wasted,
        service_level, wastage_rate, cost

    Returns:
        pd.DataFrame
    """

    product_df = None

    if product_inventory_csv is not None and Path(product_inventory_csv).exists():
        product_df = pd.read_csv(product_inventory_csv)
        product_df.columns = [str(c).strip().lower() for c in product_df.columns]

    elif target_inventory_csv is not None and Path(target_inventory_csv).exists():
        target_df = pd.read_csv(target_inventory_csv)
        target_df.columns = [str(c).strip().lower() for c in target_df.columns]

        if "product" not in target_df.columns and "target" in target_df.columns:
            target_df["product"] = np.where(
                target_df["target"].astype(str).str.startswith("aph_"),
                "APC",
                "PC"
            )

        agg_dict = {}

        for col in ["demand", "procured", "fulfilled", "unmet", "wasted", "cost"]:
            if col in target_df.columns:
                agg_dict[col] = "sum"

        product_df = (
            target_df.groupby("product", as_index=False)
            .agg(agg_dict)
        )

        if "fulfilled" in product_df.columns and "demand" in product_df.columns:
            product_df["service_level"] = product_df["fulfilled"] / product_df["demand"]

        if "wasted" in product_df.columns and "procured" in product_df.columns:
            product_df["wastage_rate"] = np.where(
                product_df["procured"] > 0,
                product_df["wasted"] / product_df["procured"],
                0
            )

    else:
        raise FileNotFoundError(
            "Either product_inventory_csv or target_inventory_csv must exist."
        )

    if product_df is None or product_df.empty:
        raise ValueError("Inventory data are empty.")

    out = pd.DataFrame()
    out["Product"] = product_df["product"].astype(str)
    out["Ordering policy"] = policy_label

    if "demand" in product_df.columns:
        out["Demand, units"] = product_df["demand"].map(lambda x: fmt_num(x, 0))

    if "procured" in product_df.columns:
        out["Procured, units"] = product_df["procured"].map(lambda x: fmt_num(x, 0))

    if "service_level" in product_df.columns:
        out["Service level, %"] = product_df["service_level"].map(lambda x: fmt_pct(x, 1))

    if "unmet" in product_df.columns:
        out["Unmet demand, units"] = product_df["unmet"].map(lambda x: fmt_num(x, 0))

    if "wastage_rate" in product_df.columns:
        out["Wastage rate, %"] = product_df["wastage_rate"].map(lambda x: fmt_pct(x, 1))

    if "wasted" in product_df.columns:
        out["Wasted units"] = product_df["wasted"].map(lambda x: fmt_num(x, 0))

    if "cost" in product_df.columns:
        out["Operational cost"] = product_df["cost"].map(lambda x: fmt_num(x, 1))

    product_order = {"PC": 1, "APC": 2}
    out["_order"] = out["Product"].map(product_order).fillna(99)
    out = out.sort_values("_order").drop(columns="_order")

    return out


# ============================================================
# 5. Table 5: Validation-selected model and inventory policy
# ============================================================

def create_table5_selected_model_policy(
    selected_models_csv,
    selected_policy_csv,
    test_inventory_target_csv=None,
    horizon=1,
):
    """
    Create Table 5.
    Validation-selected forecast model and inventory policy by product–ABO target.

    Required input:
        06_selected_forecast_model_by_validation.csv
        10_selected_inventory_policy_by_validation.csv

    Optional input:
        11_test_inventory_target_level.csv

    Expected selected model columns:
        target, horizon, model, weighted_mae, mae, occurrence_f1

    Expected selected policy columns:
        target, target_days_supply, safety_factor, cost,
        service_level, wastage_rate

    Returns:
        pd.DataFrame
    """

    sel_model = pd.read_csv(selected_models_csv)
    sel_policy = pd.read_csv(selected_policy_csv)

    sel_model.columns = [str(c).strip().lower() for c in sel_model.columns]
    sel_policy.columns = [str(c).strip().lower() for c in sel_policy.columns]

    if "horizon" in sel_model.columns:
        sel_model = sel_model[sel_model["horizon"].astype(int) == int(horizon)].copy()

    required_model = {"target", "model"}
    missing_model = required_model - set(sel_model.columns)

    if missing_model:
        raise ValueError(f"Missing required columns in selected model file: {missing_model}")

    if "target" not in sel_policy.columns:
        raise ValueError("Selected policy file must contain target column.")

    # Keep useful model columns
    model_cols = ["target", "model"]

    for c in ["weighted_mae", "mae", "occurrence_f1", "high_f1", "bias"]:
        if c in sel_model.columns:
            model_cols.append(c)

    sel_model2 = sel_model[model_cols].copy()

    # Keep useful policy columns
    policy_cols = ["target"]

    for c in [
        "target_days_supply",
        "safety_factor",
        "cost",
        "service_level",
        "wastage_rate",
    ]:
        if c in sel_policy.columns:
            policy_cols.append(c)

    sel_policy2 = sel_policy[policy_cols].copy()

    table = sel_model2.merge(sel_policy2, on="target", how="left")

    # Add test inventory outcomes if available
    if test_inventory_target_csv is not None and Path(test_inventory_target_csv).exists():
        test_inv = pd.read_csv(test_inventory_target_csv)
        test_inv.columns = [str(c).strip().lower() for c in test_inv.columns]

        inv_cols = ["target"]

        rename_map = {}

        for c in ["demand", "procured", "unmet", "wasted", "service_level", "wastage_rate", "cost"]:
            if c in test_inv.columns:
                inv_cols.append(c)
                rename_map[c] = f"test_{c}"

        test_inv2 = test_inv[inv_cols].rename(columns=rename_map)
        table = table.merge(test_inv2, on="target", how="left")

    out = pd.DataFrame()
    out["Target"] = table["target"].map(clean_target_name)
    out["Selected forecast model"] = table["model"].map(clean_model_name)

    if "weighted_mae" in table.columns:
        out["Validation weighted MAE"] = table["weighted_mae"].map(lambda x: fmt_num(x, 2))

    if "mae" in table.columns:
        out["Validation MAE"] = table["mae"].map(lambda x: fmt_num(x, 2))

    if "occurrence_f1" in table.columns:
        out["Validation occurrence F1"] = table["occurrence_f1"].map(lambda x: fmt_num(x, 2))

    if "target_days_supply" in table.columns:
        out["Selected days of supply"] = table["target_days_supply"].map(lambda x: fmt_num(x, 1))

    if "safety_factor" in table.columns:
        out["Selected safety factor"] = table["safety_factor"].map(lambda x: fmt_num(x, 2))

    if "cost" in table.columns:
        out["Validation inventory cost"] = table["cost"].map(lambda x: fmt_num(x, 1))

    if "service_level" in table.columns:
        out["Validation service level, %"] = table["service_level"].map(lambda x: fmt_pct(x, 1))

    if "wastage_rate" in table.columns:
        out["Validation wastage rate, %"] = table["wastage_rate"].map(lambda x: fmt_pct(x, 1))

    if "test_service_level" in table.columns:
        out["Test service level, %"] = table["test_service_level"].map(lambda x: fmt_pct(x, 1))

    if "test_wastage_rate" in table.columns:
        out["Test wastage rate, %"] = table["test_wastage_rate"].map(lambda x: fmt_pct(x, 1))

    if "test_unmet" in table.columns:
        out["Test unmet demand, units"] = table["test_unmet"].map(lambda x: fmt_num(x, 0))

    if "test_wasted" in table.columns:
        out["Test wasted units"] = table["test_wasted"].map(lambda x: fmt_num(x, 0))

    target_order = [
        "PC-A", "PC-B", "PC-O", "PC-AB",
        "APC-A", "APC-B", "APC-O", "APC-AB"
    ]

    out["_order"] = out["Target"].map({t: i for i, t in enumerate(target_order)})
    out = out.sort_values("_order").drop(columns="_order")

    return out


# ============================================================
# 6. Save tables to Excel
# ============================================================

def save_tables_to_excel(tables, output_xlsx):
    """
    Save all manuscript tables to one Excel workbook.

    Parameters
    ----------
    tables : dict
        Dictionary of table name -> pd.DataFrame

    output_xlsx : str
        Output Excel path.
    """

    output_xlsx = Path(output_xlsx)
    ensure_dir(output_xlsx.parent)

    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        for sheet_name, df in tables.items():
            clean_sheet = re.sub(r"[\[\]\:\*\?\/\\]", "_", sheet_name)[:31]
            df.to_excel(writer, sheet_name=clean_sheet, index=False)

    print(f"[Saved Excel] {output_xlsx}")


# ============================================================
# 7. Save tables to Word
# ============================================================

def add_dataframe_to_docx(document, df, title, note=None):
    """
    Add a pandas DataFrame as a Word table.
    """

    document.add_heading(title, level=1)

    if note:
        p = document.add_paragraph(note)
        p.style = document.styles["Normal"]

    table = document.add_table(rows=1, cols=len(df.columns))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = "Table Grid"

    # Header
    hdr_cells = table.rows[0].cells

    for i, col in enumerate(df.columns):
        hdr_cells[i].text = str(col)

    # Body
    for _, row in df.iterrows():
        row_cells = table.add_row().cells

        for i, value in enumerate(row):
            row_cells[i].text = str(value)

    # Formatting
    for row in table.rows:
        for cell in row.cells:
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            for paragraph in cell.paragraphs:
                paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
                for run in paragraph.runs:
                    run.font.size = Pt(8)

    document.add_paragraph("")


def save_tables_to_word(tables, output_docx):
    """
    Save all manuscript tables to a Word document.
    """

    if Document is None:
        raise ImportError(
            "python-docx is not installed. Run: !pip -q install python-docx"
        )

    output_docx = Path(output_docx)
    ensure_dir(output_docx.parent)

    document = Document()

    section = document.sections[0]
    section.top_margin = Inches(0.6)
    section.bottom_margin = Inches(0.6)
    section.left_margin = Inches(0.5)
    section.right_margin = Inches(0.5)

    document.add_heading(
        "Insert-ready Tables for Platelet Demand Forecasting Manuscript",
        level=0
    )

    for title, df in tables.items():
        add_dataframe_to_docx(document, df, title)

    document.save(output_docx)

    print(f"[Saved Word] {output_docx}")


# ============================================================
# 8. Master function
# ============================================================

def create_all_manuscript_tables(
    result_dir,
    output_dir,
    split="test",
    horizon=1,
):
    """
    Create Tables 1, 2, 4, and 5 from forecasting result files.

    Parameters
    ----------
    result_dir : str
        Directory containing model-output CSV files.

    output_dir : str
        Directory where table files will be saved.

    split : str
        Evaluation split for Table 1.

    horizon : int
        Forecast horizon for Table 1 and Table 5.

    Returns
    -------
    dict
        Dictionary of table name -> pd.DataFrame
    """

    result_dir = Path(result_dir)
    output_dir = ensure_dir(output_dir)

    paths = {
        "diagnostics": result_dir / "01_sparse_intermittent_diagnostics.csv",
        "metrics": result_dir / "03_all_metrics_by_target.csv",
        "selected_models": result_dir / "06_selected_forecast_model_by_validation.csv",
        "selected_policy": result_dir / "10_selected_inventory_policy_by_validation.csv",
        "test_inventory_target": result_dir / "11_test_inventory_target_level.csv",
        "test_inventory_product": result_dir / "12_test_inventory_product_level.csv",
    }

    # Table 1
    table1 = create_table1_model_performance(
        metrics_csv=paths["metrics"],
        split=split,
        horizon=horizon,
        metric_for_target="mae",
        rank_metric="weighted_mae",
    )

    # Table 2
    table2 = create_table2_sparse_diagnostics(
        diagnostics_csv=paths["diagnostics"]
    )

    # Table 4
    table4 = create_table4_inventory_outcomes(
        product_inventory_csv=paths["test_inventory_product"],
        target_inventory_csv=paths["test_inventory_target"],
        policy_label="Validation-selected policy",
    )

    # Table 5
    table5 = create_table5_selected_model_policy(
        selected_models_csv=paths["selected_models"],
        selected_policy_csv=paths["selected_policy"],
        test_inventory_target_csv=paths["test_inventory_target"],
        horizon=horizon,
    )

    tables = {
        "Table 1. Expanded forecasting model performance": table1,
        "Table 2. Sparse and intermittent demand diagnostics": table2,
        "Table 4. Simulated inventory outcomes": table4,
        "Table 5. Validation-selected model and policy": table5,
    }

    # Save
    save_tables_to_excel(
        tables=tables,
        output_xlsx=output_dir / "platelet_manuscript_tables.xlsx",
    )

    try:
        save_tables_to_word(
            tables=tables,
            output_docx=output_dir / "platelet_manuscript_tables.docx",
        )
    except Exception as e:
        print(f"[WARNING] Word export failed: {e}")
        print("Excel export was completed successfully.")

    return tables



# ============================================================
# PART 4. Optional supplementary per-model figures
#
# The original notebook re-pasted this same "loop over every model and
# plot X" pattern four separate times (once for Figure 2 per model, once
# more for a "selected models" subset, once for ABO-specific trends per
# model, and once for PC/APC-split ABO trends per model). They are
# consolidated here into one function, kept optional (--extra-figures)
# since the manuscript only uses Figure 1 and Figure 2.
# ============================================================

def generate_supplementary_figures(
    panel: pd.DataFrame,
    base_cfg: "FigureConfig",
    all_predictions_csv: str,
    output_dir: str,
    models: Optional[List[str]] = None,
    horizon: int = 1,
) -> Dict[str, List[Path]]:
    """Generate per-model Figure 2, ABO-trend, and product-ABO-trend plots.

    Parameters
    ----------
    panel : output of load_daily_panel_and_metadata().
    base_cfg : a FigureConfig used as the template for split dates, the
        rolling window, and the recent-window start; its output_dir and
        prediction-file fields are overridden per model below.
    all_predictions_csv : path to Part 1's "02_all_predictions_long.csv".
    output_dir : base directory; three subfolders are created under it.
    models : optional explicit list of model names to plot. If None, every
        model found in `all_predictions_csv` is used.

    Returns
    -------
    dict with keys "figure2", "abo_trends", "product_abo_trends", each a
    list of saved file paths.
    """
    all_predictions_csv = str(all_predictions_csv)
    pred_all = pd.read_csv(all_predictions_csv)
    pred_all.columns = [str(c).strip().lower() for c in pred_all.columns]
    if "model" not in pred_all.columns:
        raise ValueError("all_predictions_csv must contain a 'model' column.")

    if models is None:
        models = sorted(pred_all["model"].dropna().astype(str).unique())
    print(f"[supplementary figures] {len(models)} model(s): {models}")

    fig2_dir = ensure_dir(Path(output_dir) / "Figure2_by_model")
    abo_dir = ensure_dir(Path(output_dir) / "ABO_trends_by_model")
    product_abo_dir = ensure_dir(Path(output_dir) / "Product_ABO_trends_by_model")

    saved: Dict[str, List[Path]] = {"figure2": [], "abo_trends": [], "product_abo_trends": []}

    for model_name in models:
        pred_df_model = load_predictions(
            selected_predictions_csv=None,
            all_predictions_csv=all_predictions_csv,
            model_name=model_name,
            horizon=horizon,
        )
        if pred_df_model is None or pred_df_model.empty:
            print(f"  [skip] No horizon-{horizon} prediction found for model: {model_name}")
            continue

        # --- Figure 2 for this model ---
        cfg_model = FigureConfig(
            data_path=base_cfg.data_path,
            output_dir=str(fig2_dir),
            selected_predictions_csv=None,
            all_predictions_csv=all_predictions_csv,
            figure2_model_name=model_name,
            forecast_label=f"{model_name} forecast",
            train_start=base_cfg.train_start,
            train_end=base_cfg.train_end,
            val_start=base_cfg.val_start,
            val_end=base_cfg.val_end,
            test_start=base_cfg.test_start,
            test_end=base_cfg.test_end,
            recent_start=base_cfg.recent_start,
            rolling_window=base_cfg.rolling_window,
        )
        fig2_path = Path(plot_figure2_demand_surge(panel=panel, cfg=cfg_model, pred_df=pred_df_model))
        new_fig2_path = fig2_dir / f"Figure2_{safe_filename(model_name)}.png"
        if new_fig2_path.exists():
            new_fig2_path.unlink()
        fig2_path.rename(new_fig2_path)
        saved["figure2"].append(new_fig2_path)

        # --- ABO-specific trend (one 2x4 figure) ---
        abo_path = plot_model_trends_by_blood_type(
            panel=panel,
            pred_df_model=pred_df_model,
            model_name=model_name,
            output_dir=str(abo_dir),
            recent_start=base_cfg.recent_start,
            train_start=base_cfg.train_start,
            train_end=base_cfg.train_end,
            test_start=base_cfg.test_start,
            test_end=base_cfg.test_end,
            rolling_window=base_cfg.rolling_window,
        )
        saved["abo_trends"].append(Path(abo_path))

        # --- PC / APC split ABO trend ---
        for product in ("PC", "APC"):
            product_path = plot_product_abo_trends_for_model(
                panel=panel,
                pred_df_model=pred_df_model,
                model_name=model_name,
                product=product,
                output_dir=str(product_abo_dir),
                recent_start=base_cfg.recent_start,
                train_start=base_cfg.train_start,
                train_end=base_cfg.train_end,
                test_start=base_cfg.test_start,
                test_end=base_cfg.test_end,
                rolling_window=base_cfg.rolling_window,
            )
            saved["product_abo_trends"].append(Path(product_path))

        print(f"  [ok] {model_name}")

    print(f"[supplementary figures] Saved {len(saved['figure2'])} Figure 2, "
          f"{len(saved['abo_trends'])} ABO-trend, and "
          f"{len(saved['product_abo_trends'])} product-ABO-trend plot(s).")
    return saved



# ============================================================
# PART 5. Command-line entry point
# ============================================================

def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Sparse-aware platelet demand forecasting, inventory-policy "
                     "simulation, and manuscript figure/table generation.",
    )
    parser.add_argument(
        "--data", default="platelet_data_english_260529.xlsx",
        help="Path to the source Excel file (default: %(default)s).",
    )
    parser.add_argument(
        "--output-dir", default="results_sparse_hurdle_plus_models",
        help="Directory for Part 1 analysis CSV/XLSX outputs (default: %(default)s).",
    )
    parser.add_argument(
        "--figures-dir", default="paper_figures",
        help="Directory for Figure 1 / Figure 2 PNGs (default: %(default)s).",
    )
    parser.add_argument(
        "--tables-dir", default="paper_tables",
        help="Directory for the manuscript Excel/Word tables (default: %(default)s).",
    )
    parser.add_argument(
        "--skip-analysis", action="store_true",
        help="Skip Part 1 (forecasting + inventory simulation) and reuse the "
             "existing CSVs already in --output-dir.",
    )
    parser.add_argument(
        "--skip-figures", action="store_true",
        help="Skip Figure 1 / Figure 2 generation.",
    )
    parser.add_argument(
        "--skip-tables", action="store_true",
        help="Skip Table 1/2/4/5 generation.",
    )
    parser.add_argument(
        "--extra-figures", action="store_true",
        help="Also generate the supplementary per-model figures (Figure 2, "
             "ABO-trend, and product-ABO-trend for every model found in the "
             "predictions file). Off by default because the manuscript only "
             "uses Figure 1 and Figure 2.",
    )
    parser.add_argument("--no-lightgbm", action="store_true", help="Skip the LightGBM model.")
    parser.add_argument("--no-prophet", action="store_true", help="Skip the Prophet model.")
    parser.add_argument("--no-lstm", action="store_true", help="Skip the LSTM model.")
    return parser


def main(argv: Optional[Sequence[str]] = None) -> None:
    args = build_arg_parser().parse_args(argv)

    cfg = SparseHurdleConfig(
        data_path=args.data,
        output_dir=args.output_dir,
        run_lightgbm=not args.no_lightgbm,
        run_prophet=not args.no_prophet,
        run_lstm=not args.no_lstm,
    )
    result_dir = Path(cfg.output_dir)

    # --- Part 1: forecasting + inventory-policy analysis ---
    if not args.skip_analysis:
        run_analysis(cfg)
    else:
        print(f"[skip] Reusing existing analysis outputs in: {result_dir}")

    # --- Part 2: Figure 1 / Figure 2 ---
    if not args.skip_figures:
        fig_cfg = FigureConfig(
            data_path=cfg.data_path,
            output_dir=args.figures_dir,
            all_predictions_csv=str(result_dir / "02_all_predictions_long.csv"),
            forecast_label="LASSO and MA7 forecasts",
            train_start=cfg.train_start,
            train_end=cfg.train_end,
            val_start=cfg.val_start,
            val_end=cfg.val_end,
            test_start=cfg.test_start,
            test_end=cfg.test_end,
        )
        panel, meta = load_daily_panel_and_metadata(fig_cfg.data_path)
        # Use the complete prediction table to show both LASSO and MA7.
        pred_df = pd.read_csv(fig_cfg.all_predictions_csv)

        fig1_path = plot_figure1_temporal_patterns(panel=panel, meta=meta, cfg=fig_cfg)
        print(f"Figure 1 saved to: {fig1_path}")

        fig2_path = plot_figure2_product_abo_forecasts(panel=panel, cfg=fig_cfg, pred_df=pred_df)
        print(f"Figure 2 (PC/APC by ABO; LASSO and MA7) saved to: {fig2_path}")

        if args.extra_figures:
            generate_supplementary_figures(
                panel=panel,
                base_cfg=fig_cfg,
                all_predictions_csv=str(result_dir / "02_all_predictions_long.csv"),
                output_dir=args.figures_dir,
            )
    else:
        print("[skip] Figure generation skipped.")

    # --- Part 3: Table 1/2/4/5 ---
    if not args.skip_tables:
        create_all_manuscript_tables(
            result_dir=result_dir,
            output_dir=args.tables_dir,
            split="test",
            horizon=1,
        )
    else:
        print("[skip] Table generation skipped.")


if __name__ == "__main__":
    main()


In [ ]:

# Import it as a module too (used directly by the custom analyses below: Table 3,
# Supplementary S4/S5/S8), in addition to running it as a script (for Table 2/4 and
# Figures 1-2, exactly reproducing the published pipeline run).
spec = importlib.util.spec_from_file_location("pv2", "platelet_analysis_v2.py")
pv2 = importlib.util.module_from_spec(spec)
sys.modules["pv2"] = pv2  # must be registered before exec_module for @dataclass to work
spec.loader.exec_module(pv2)
print("platelet_analysis_v2 loaded OK.")


## 2. Run the core forecast pipeline (exploratory core inventory is NOT manuscript Tables 3/4)

This runs the forecast/diagnostic core pipeline. Its exploratory inventory outputs use a different engine and must NOT be confused with the separate manuscript Tables 3/4 produced below. This runs the main forecasting code with the established primary models; outputs are written to `analysis_r2_s18s19`, `tables_r2_s18s19`, and `figures_r2_s18s19`, leaving the older folders untouched. The subprocess must exit successfully; otherwise execution stops and the error is visible.

In [ ]:
# Stop immediately if the core command fails; do not proceed with stale CSVs.
import subprocess
core_cmd = [
    sys.executable, "platelet_analysis_v2.py",
    "--data", DATA_FILENAME,
    "--output-dir", "analysis_r2_s18s19",
    "--tables-dir", "tables_r2_s18s19",
    "--figures-dir", "figures_r2_s18s19",
    "--no-prophet", "--no-lstm",
]
print("Running:", " ".join(core_cmd), flush=True)
subprocess.run(core_cmd, check=True)
print("Core analysis completed successfully.")

### S1.8 and S1.9: 181-day **information-matched** SARIMA comparison

The study's LASSO/MA inputs at forecast origin t=d−1 are shifted by one day, so their latest observed demand is d−2. **Main Table 2 SARIMA uses this SAME cutoff**: for each target day d, model parameters are learned on training only, Kalman states are updated with prior observations, and the d forecast is issued without using d−1 or d. Thus each forecast is made for the next target day at origin d−1, with the same data availability assumptions as the primary models. This is rolling, not a fixed 181-step forecast.

For transparency, a separate **more recent-data sensitivity** uses demand through d−1; that conventional rolling forecast cannot be presented as having an identical information set. The two forecast series, MAE, MASE, convergence diagnostics, and comparison metadata are all saved. LightGBM uses the published S1.8 settings.

In [ ]:
%%writefile platelet_S18_S19_benchmarks.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Supplementary Methods S1.8-S1.9 reproducibility benchmarks.
SARIMA is evaluated separately from validation-based target-model selection so
that Tables 3/4 and the primary LASSO/MA/blend results are not re-optimized.
"""
from __future__ import annotations
import json, platform, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 7)
TARGET_LABELS = {
    "plt_transf_a":"PC-A", "plt_transf_b":"PC-B",
    "plt_transf_o":"PC-O", "plt_transf_ab":"PC-AB",
    "aph_transf_a":"APC-A", "aph_transf_b":"APC-B",
    "aph_transf_o":"APC-O", "aph_transf_ab":"APC-AB",
}


def run_sarima_s19(panel: pd.DataFrame, cfg, output_dir: str | Path,
                   target_columns: list[str] | None = None,
                   maxiter: int = 100):
    """Rolling SARIMA forecasts with IDENTICAL information cutoff to LASSO/MA7.

    For forecast date d, all manuscript primary model observed-demand predictors
    use observations no later than d-2 (because origin t=d-1 has shift(1)).
    The fair SARIMA point forecast is thus y(d)|data_through(d-2), obtained
    by a fixed-parameter two-step Kalman prediction from d-2. Operationally
    it is next-day forecast at the decision/origin day d-1, when only data
    through d-2 are available. This is not an 181-step fixed test forecast.

    An extra sensitivity reports the conventional rolling one-step filter
    using observations through d-1. It uses a MORE RECENT information set
    than the primary LASSO/MA/blend predictors, so it must not replace the
    information-matched SARIMA row in Table 2.
    """
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    import statsmodels
    target_columns=list(target_columns or TARGET_LABELS)
    if sorted(target_columns)!=sorted(TARGET_LABELS):
        raise ValueError("All eight product-ABO targets are required")
    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)
    expected_days=pd.date_range(cfg.test_start,cfg.test_end,freq='D')
    spec=dict(order=SARIMA_ORDER,seasonal_order=SARIMA_SEASONAL_ORDER,
              trend='n',enforce_stationarity=False,enforce_invertibility=False)
    rows=[];predictions=[];native_rows=[];native_predictions=[];notes=[]
    for target in target_columns:
        series=panel[target].astype(float).sort_index().asfreq('D')
        if series.isna().any():raise ValueError(f"{target}: missing observations")
        train=series.loc[cfg.train_start:cfg.train_end]
        valid=series.loc[cfg.val_start:cfg.val_end]
        test=series.loc[cfg.test_start:cfg.test_end]
        if len(train)!=1827 or len(valid)!=184 or len(test)!=181 or not test.index.equals(expected_days):
            raise AssertionError(f"{target}: train/validation/test date boundaries invalid")
        start=time.monotonic()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            fitted=SARIMAX(train,**spec).fit(disp=False,maxiter=maxiter)
        initially_converged=bool(fitted.mle_retvals.get('converged',False))
        if not initially_converged:
            with warnings.catch_warnings(record=True) as caught_retry:
                warnings.simplefilter('always')
                fitted=SARIMAX(train,**spec).fit(start_params=fitted.params,
                    disp=False,maxiter=max(300,maxiter))
            caught.extend(caught_retry)
        converged=bool(fitted.mle_retvals.get('converged',False))
        if not converged:
            raise RuntimeError(f"{target}: SARIMA optimizer did not converge")

        # This ONE filtering pass computes causal predictions for each date.
        # get_prediction(dynamic=d-1) omits day d-1 and therefore conditions
        # on values only through d-2. Never fit coefficients with val/test.
        filtered=SARIMAX(series,**spec).filter(fitted.params)
        aligned=np.maximum(0.0,np.asarray([
            filtered.get_prediction(start=d,end=d,dynamic=d-pd.Timedelta(days=1))
                    .predicted_mean.iloc[0] for d in expected_days],dtype=float))
        latest=np.maximum(0.0,np.asarray(filtered.get_prediction(
            start=expected_days[0],end=expected_days[-1],dynamic=False
        ).predicted_mean,dtype=float))
        if len(aligned)!=181 or len(latest)!=181 or not np.isfinite(aligned).all():
            raise AssertionError(f"{target}: incomplete/nonfinite forecasts")
        # Independent small-sample verification: literal 2-step forecast from
        # the last ALLOWED observed date must equal dynamic (d-1) prediction.
        verification_error=0.0
        for d,pred_value in zip(expected_days[:3],aligned[:3]):
            observed_until=d-pd.Timedelta(days=2)
            state=fitted.extend(series.loc[cfg.val_start:observed_until])
            literal_two_step=max(0.0,float(np.asarray(state.forecast(steps=2))[-1]))
            verification_error=max(verification_error,abs(literal_two_step-pred_value))
        if verification_error>1e-5:
            raise AssertionError(f"{target}: lag-matched forecast mismatch {verification_error}")
        # Check the additional conventional one-step rolling forecast.
        native_state=fitted.extend(valid)
        max_native_delta=0.0
        for d,pred_value in zip(expected_days[:3],latest[:3]):
            literal_one_step=max(0.0,float(np.asarray(native_state.forecast(steps=1))[0]))
            max_native_delta=max(max_native_delta,abs(literal_one_step-pred_value))
            native_state=native_state.extend(test.loc[[d]])
        if max_native_delta>1e-5:
            raise AssertionError(f"{target}: conventional one-step mismatch")
        truth=test.to_numpy(dtype=float)
        tr=train.to_numpy(dtype=float)
        scale=float(np.mean(np.abs(tr[7:]-tr[:-7])))
        if scale<=0:raise ValueError(f"{target}: invalid MASE denominator")
        mae=float(np.mean(np.abs(truth-aligned)))
        native_mae=float(np.mean(np.abs(truth-latest)))
        label=TARGET_LABELS[target]
        for d,a,yhat,yhat_native in zip(expected_days,truth,aligned,latest):
            predictions.append(dict(date=d.strftime('%Y-%m-%d'),target=label,
                target_raw=target,model='SARIMA',actual=float(a),pred=float(yhat),
                forecast_horizon_from_origin=1,latest_observed_lag_days=2,
                protocol='rolling_h1_availability_matched'))
            native_predictions.append(dict(date=d.strftime('%Y-%m-%d'),target=label,
                target_raw=target,model='SARIMA',actual=float(a),pred=float(yhat_native),
                forecast_horizon_from_origin=1,latest_observed_lag_days=1,
                protocol='rolling_h1_latest_prior_day'))
        w=' | '.join(str(x.message) for x in caught)
        rows.append(dict(target=target,Target=label,sarima_mae=mae,
            sarima_mase_s113=mae/scale,n_test_days=181,converged=converged,
            fit_seconds=round(time.monotonic()-start,3),
            optimizer_iterations=fitted.mle_retvals.get('iterations'),
            initial_fit_converged=initially_converged,warnings=w,
            max_abs_matched_vs_literal_error=verification_error,
            protocol='rolling_h1_availability_matched'))
        native_rows.append(dict(target=target,Target=label,sarima_mae=native_mae,
            sarima_mase_s113=native_mae/scale,n_test_days=181,converged=converged,
            protocol='rolling_h1_latest_prior_day'))
        print(f"  {label:6s} MAE lag-matched={mae:.3f} "
              f"MAE previous-day={native_mae:.3f} converged={converged}",flush=True)
    metrics=pd.DataFrame(rows);preds=pd.DataFrame(predictions)
    extra=pd.DataFrame(native_rows);extra_preds=pd.DataFrame(native_predictions)
    for df in (preds,extra_preds):
        if len(df)!=1448 or df.groupby('target')['date'].nunique().ne(181).any():
            raise AssertionError('Not 8 targets x 181 days')
    prefix='SARIMA_S19_rolling_h1_matched'
    metrics.to_csv(output_dir/(prefix+'_metrics_by_target.csv'),index=False)
    preds.to_csv(output_dir/(prefix+'_predictions_181days.csv'),index=False)
    native_prefix='SARIMA_S19_rolling_h1_latest_day'
    extra.to_csv(output_dir/(native_prefix+'_metrics_by_target.csv'),index=False)
    extra_preds.to_csv(output_dir/(native_prefix+'_predictions_181days.csv'),index=False)
    meta=dict(model='SARIMA',protocol='rolling_h1_availability_matched',
        order=SARIMA_ORDER,seasonal_order=SARIMA_SEASONAL_ORDER,
        parameter_fit='training only through 2024-02-29',
        information_cutoff='For target date d, observed data only through d-2, matching MA/LASSO feature construction',
        parameter_refit_during_validation=False,parameter_refit_during_test=False,
        forecast_strategy='one day ahead from forecast-origin d-1; two filter steps from last confirmed d-2',
        mean_mae=float(metrics['sarima_mae'].mean()),
        mean_mase_s113=float(metrics['sarima_mase_s113'].mean()),
        supplementary_mae_latest_prior_day=float(extra['sarima_mae'].mean()),
        supplementary_mase_latest_prior_day=float(extra['sarima_mase_s113'].mean()),
        n_test_days=181,n_targets=8,
        optimizer_all_converged=bool(metrics['converged'].all()),
        statsmodels_version=statsmodels.__version__,python_version=platform.python_version(),
        warning='Do not mix 181-step fixed or more recent-data forecasts into main Table 2')
    (output_dir/(prefix+'_run_metadata.json')).write_text(
        json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
    print('SARIMA fair h=1 MAE=',meta['mean_mae'],'MASE=',meta['mean_mase_s113'])
    print('SARIMA more recent-data sensitivity MAE=',meta['supplementary_mae_latest_prior_day'])
    return metrics,preds,meta


def run_lightgbm_s18(pv2, panel: pd.DataFrame, cfg, output_dir: str | Path,
                     target_columns: list[str] | None = None):
    """Training-only S1.8 LightGBM benchmark; use the existing h=1 feature frame."""
    import lightgbm
    output_dir=Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    verify_lightgbm_s18(pv2,cfg)
    target_columns=list(target_columns or TARGET_LABELS)
    X_all=pv2.build_feature_frame(panel,target_columns,horizon=1,windows=cfg.moving_windows)
    rows=[];preds=[]
    for target in target_columns:
        y_all=pv2.target_series_for_horizon(panel,target,horizon=1)
        X_train,y_train=pv2.split_X_y_from_all(X_all,y_all,cfg.train_start,cfg.train_end,horizon=1)
        X_val,y_val=pv2.split_X_y_from_all(X_all,y_all,cfg.val_start,cfg.val_end,horizon=1)
        X_test,y_test=pv2.split_X_y_from_all(X_all,y_all,cfg.test_start,cfg.test_end,horizon=1)
        if len(y_test)!=181 or len(y_val)!=184:
            raise AssertionError(f'{target}: invalid validation/test coverage')
        prediction=pv2.fit_predict_lightgbm(X_train,y_train,X_val,X_test,cfg)
        if prediction is None:raise RuntimeError('LightGBM was skipped; cannot claim reproduction')
        p_test=np.asarray(prediction[1],dtype=float)
        truth=y_test.to_numpy(dtype=float)
        mae=float(np.mean(np.abs(truth-p_test)))
        t_train=panel[target].loc[cfg.train_start:cfg.train_end].to_numpy(dtype=float)
        denom=float(np.mean(np.abs(t_train[7:]-t_train[:-7])))
        mase=mae/denom
        label=TARGET_LABELS[target]
        rows.append(dict(Target=label,target=target,lightgbm_mae=mae,
                         lightgbm_mase_s113=mase,n_test_days=len(y_test)))
        for dt,actual,pred in zip(y_test.index+pd.Timedelta(days=1),truth,p_test):
            preds.append(dict(date=dt.strftime('%Y-%m-%d'),target=label,model='LightGBM',
                              actual=float(actual),pred=float(pred)))
        print(f'  {label:6} LGB MAE={mae:.3f}, MASE={mase:.3f}, n=181',flush=True)
    metrics=pd.DataFrame(rows)
    predictions=pd.DataFrame(preds)
    assert len(metrics)==8 and len(predictions)==1448
    metrics.to_csv(output_dir/'LightGBM_S18_metrics_by_target.csv',index=False)
    predictions.to_csv(output_dir/'LightGBM_S18_predictions_181days.csv',index=False)
    meta={'protocol':'train-only fit, 181-day test', 'objective':'regression',
          'n_estimators':300,'num_leaves':31,'learning_rate':0.03,'subsample':0.8,
          'subsample_freq':1,'colsample_bytree':0.8,'min_child_samples':20,
          'random_state':42,'lightgbm_version':lightgbm.__version__,
          'mean_mae':float(metrics['lightgbm_mae'].mean()),
          'mean_mase_s113':float(metrics['lightgbm_mase_s113'].mean()),
          'published_mean_mae':6.445,
          'note':'Parameters match the documented reconstruction, not necessarily the separate original experiment.'}
    (output_dir/'LightGBM_S18_run_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
    print('LightGBM mean MAE:',round(meta['mean_mae'],3),'vs reported 6.445')
    return metrics,predictions,meta


def verify_lightgbm_s18(pv2, cfg):
    """Fail fast if the active LightGBM config differs from S1.8."""
    import lightgbm
    m=pv2.make_lightgbm_regressor(cfg)
    if m is None:
        raise ImportError('LightGBM is unavailable: install lightgbm first')
    p=m.get_params()
    expected={'objective':'regression','n_estimators':300,'num_leaves':31,
              'learning_rate':0.03,'min_child_samples':20,'max_depth':-1,
              'subsample':0.8,'subsample_freq':1,'colsample_bytree':0.8,
              'reg_alpha':0.0,'reg_lambda':0.0,'random_state':42}
    mismatches={k:(p.get(k),v) for k,v in expected.items() if p.get(k)!=v}
    if mismatches: raise AssertionError(f'LightGBM differs from S1.8: {mismatches}')
    print('S1.8 LightGBM parameters verified:',expected)
    print('LightGBM installed version:',lightgbm.__version__)
    return expected


if __name__=='__main__':
    import argparse,importlib.util,sys
    ap=argparse.ArgumentParser(description='Standalone S1.8/S1.9 check')
    ap.add_argument('--data',required=True)
    ap.add_argument('--core',default='platelet_r2_minimal_S18_S19.py')
    ap.add_argument('--output-dir',default='analysis_minimal181')
    ap.add_argument('--maxiter',type=int,default=100)
    ap.add_argument('--protocol',choices=['published_static','rolling_one_step'],default='published_static')
    args=ap.parse_args()
    spec=importlib.util.spec_from_file_location('pv2',args.core)
    pv2=importlib.util.module_from_spec(spec)
    sys.modules['pv2']=pv2
    spec.loader.exec_module(pv2)
    cfg=pv2.SparseHurdleConfig(data_path=args.data,run_prophet=False,run_lstm=False)
    verify_lightgbm_s18(pv2,cfg)
    panel,_=pv2.load_platelet_excel(args.data)
    run_sarima_s19(panel,cfg,args.output_dir,protocol=args.protocol)


In [ ]:
# Run SARIMA with strictly comparable one-day-ahead test forecasts.
import importlib.util, sys, json
from pathlib import Path
bench_spec = importlib.util.spec_from_file_location(
    "platelet_bench", "platelet_S18_S19_benchmarks.py")
platelet_bench = importlib.util.module_from_spec(bench_spec)
sys.modules["platelet_bench"] = platelet_bench
bench_spec.loader.exec_module(platelet_bench)

bench_cfg = pv2.SparseHurdleConfig(
    data_path="platelet_data_english_260529.xlsx",run_prophet=False,run_lstm=False)
lightgbm_params_S18 = platelet_bench.verify_lightgbm_s18(pv2,bench_cfg)
bench_panel, _ = pv2.load_platelet_excel(bench_cfg.data_path)

# Always run: the published static 181-step protocol is NOT a fair h=1 comparison.
sarima_metrics_s19, sarima_preds_s19, sarima_metadata_s19 = platelet_bench.run_sarima_s19(
    bench_panel,bench_cfg,"analysis_r2_s18s19",
    target_columns=list(platelet_bench.TARGET_LABELS))
assert sarima_metadata_s19["protocol"] == "rolling_h1_availability_matched"
assert sarima_metadata_s19["optimizer_all_converged"]
print("SARIMA information-matched h=1 MAE:",sarima_metadata_s19["mean_mae"])
print("SARIMA using more recent observation (sensitivity):",
      sarima_metadata_s19["supplementary_mae_latest_prior_day"])
print("SARIMA common S1.13 MASE:",sarima_metadata_s19["mean_mase_s113"])

lightgbm_metrics_s18, lightgbm_preds_s18, lightgbm_metadata_s18 = platelet_bench.run_lightgbm_s18(
    pv2,bench_panel,bench_cfg,"analysis_r2_s18s19",
    list(platelet_bench.TARGET_LABELS))
print("LightGBM mean MAE:",round(lightgbm_metadata_s18["mean_mae"],3))


### Table 2. Forecasting accuracy and independently reproduced benchmarks

This compares the full eight-target forecast table. The independent SARIMA row uses a **static 181-step** prediction protocol, so it is not a head-to-head rolling h=1 forecast; the notebook prints this limitation with the table. The historical LightGBM and SARIMA manuscript values are references for audit, **not pre-filled results**.

In [ ]:

import pandas as pd
import numpy as np

TARGET_LABELS = {
    "plt_transf_a": "PC-A", "plt_transf_b": "PC-B", "plt_transf_o": "PC-O", "plt_transf_ab": "PC-AB",
    "aph_transf_a": "APC-A", "aph_transf_b": "APC-B", "aph_transf_o": "APC-O", "aph_transf_ab": "APC-AB",
}
MODEL_LABELS = {
    "historical_mean": "Historical mean", "seasonal_naive": "Seasonal naive",
    "moving_average_7d": "MA7", "moving_average_14d": "MA14", "lasso": "LASSO",
    "ridge": "Ridge", "lightgbm": "LightGBM",
    "blend_lasso_ma7_50_50": "LASSO-MA7 blend", "blend_lasso_ma14_50_50": "LASSO-MA14 blend",
}

metrics = pd.read_csv("analysis_r2_s18s19/03_all_metrics_by_target.csv")
m = metrics[(metrics["split"] == "test") & (metrics["horizon"] == 1)].copy()
m = m[m["model"].isin(MODEL_LABELS)]
wide = m.pivot_table(index="model", columns="target", values="mae", aggfunc="first")
wide = wide.rename(columns=TARGET_LABELS).rename(index=MODEL_LABELS)
wide = wide[[TARGET_LABELS[t] for t in TARGET_LABELS]]
wide["Mean MAE"] = wide.mean(axis=1)

mase = m.pivot_table(index="model", columns="target", values="mase", aggfunc="first").rename(index=MODEL_LABELS)
wide["Mean MASE"] = mase.mean(axis=1)

# Add independent SARIMA benchmark to Table 2 only (not candidate models in Table 4).
sarima_file = Path("analysis_r2_s18s19/SARIMA_S19_rolling_h1_matched_metrics_by_target.csv")
if not sarima_file.exists():
    raise FileNotFoundError(f"SARIMA was not run: {sarima_file}")
sarima_out = pd.read_csv(sarima_file)
assert len(sarima_out) == 8 and sarima_out["n_test_days"].eq(181).all()
sarima_row = pd.Series({r["Target"]: r["sarima_mae"] for _, r in sarima_out.iterrows()})
sarima_row["Mean MAE"] = sarima_out["sarima_mae"].mean()
sarima_row["Mean MASE"] = sarima_out["sarima_mase_s113"].mean()
wide.loc["SARIMA", sarima_row.index] = sarima_row.values
# Recompute mean ranks within 8 targets after SARIMA is added.
target_cols = [TARGET_LABELS[t] for t in TARGET_LABELS]
wide["Mean rank"] = wide[target_cols].rank(axis=0,method="average",ascending=True).mean(axis=1)
table2 = wide.sort_values("Mean MAE").round(3)
table2.to_csv("analysis_r2_s18s19/Table2_S18_S19_corrected.csv")
print("Reproduced Table 2 (test split, horizon = 1):\n")
print(table2.to_string())

print("\nFINAL REPRODUCED TABLE 2 — no outdated manuscript numbers are printed.")
print("SARIMA information-matched rolling h=1, training-only MASE:",
      round(float(sarima_out["sarima_mase_s113"].mean()), 3))
print("Seasonal naive: same-weekday demand 7 days before TARGET date (corrected).")
assert float(table2.loc["SARIMA","Mean MAE"]) < 7, "Unexpected SARIMA row: static forecast may have been used"


### Mandatory post-run quality check: 181 test target dates, LASSO/MA/blends, LightGBM, SARIMA

In [ ]:
# Independent integrity audit: every primary target must cover exactly 181 outcome days.
from pathlib import Path
import pandas as pd, json
_audit_csv = Path("analysis_r2_s18s19/02_all_predictions_long.csv")
assert _audit_csv.exists(), f"No prediction CSV: {_audit_csv}"
_audit_preds = pd.read_csv(_audit_csv, parse_dates=["date"])
_audit_h1 = _audit_preds.loc[
    (_audit_preds["split"] == "test")
    & (_audit_preds["horizon"] == 1)
    & (_audit_preds["model"].isin([
        "lasso", "moving_average_7d", "moving_average_14d",
        "blend_lasso_ma7_50_50", "blend_lasso_ma14_50_50"
    ]))
].copy()
_audit_h1["target_date"] = _audit_h1["date"] + pd.Timedelta(days=1)
_expected = pd.date_range("2024-09-01", "2025-02-28", freq="D")
_check = _audit_h1.groupby(["target", "model"])["target_date"].agg(["count", "nunique", "min", "max"])
assert len(_check) == 8 * 5, f"Expected 40 target-model combinations, got {len(_check)}"
assert _check["count"].eq(181).all() and _check["nunique"].eq(181).all(), _check
assert (_check["min"] == _expected.min()).all(), _check
assert (_check["max"] == _expected.max()).all(), _check
for _name in ("SARIMA_S19_rolling_h1_matched_metrics_by_target.csv", "LightGBM_S18_metrics_by_target.csv"):
    _check_df = pd.read_csv(Path("analysis_r2_s18s19") / _name)
    assert len(_check_df) == 8 and _check_df["n_test_days"].eq(181).all(), _name
    print(_name, ": 8 targets × 181 test days confirmed")
_audit_obj = {
    "n_test_days_per_target": 181,
    "n_target_series": 8,
    "first_test_target_date": str(_expected.min().date()),
    "last_test_target_date": str(_expected.max().date()),
    "n_primary_model_target_combinations": int(len(_check)),
    "sarima_protocol": "rolling_h1_availability_matched_train_only",
    "lightgbm_hyperparameters_checked": True,
}
Path("analysis_r2_s18s19/R2_integrity_check.json").write_text(
    json.dumps(_audit_obj, indent=2), encoding="utf-8"
)
print("181-day target-date and benchmark integrity audit: PASS")
_sr = pd.read_csv("analysis_r2_s18s19/SARIMA_S19_rolling_h1_matched_predictions_181days.csv")
assert len(_sr)==8*181 and _sr["date"].nunique()==181 and _sr["target"].nunique()==8
assert _sr["forecast_horizon_from_origin"].eq(1).all() and _sr["protocol"].eq("rolling_h1_availability_matched").all()
print("SARIMA fair one-day horizon coverage: PASS")

# Seasonal naive implementation QA: target d must use y[d-7], not y[d-8].
_sn = _audit_preds.loc[(_audit_preds["split"] == "test") &
                       (_audit_preds["horizon"] == 1) &
                       (_audit_preds["model"] == "seasonal_naive")].copy()
assert len(_sn) == 8 * 181, f"Unexpected seasonal naive test rows: {len(_sn)}"
for _target, _group in _sn.groupby("target"):
    _origins = pd.to_datetime(_group["date"])
    _source_dates = _origins - pd.Timedelta(days=6)  # origin t; target t+1, 7-days back
    _expected_preds = bench_panel.loc[_source_dates, _target].to_numpy(dtype=float)
    assert np.allclose(_group["pred"].to_numpy(dtype=float), _expected_preds), _target
assert np.isclose(float(table2.loc["Seasonal naive", "Mean MAE"]), 7.623, atol=0.002)
print("Seasonal naive reference-day integrity: PASS (target d-7, 8 x 181 days)")


**Interpretation:** The manuscript's SARIMA row now uses the same one-day forecast-origin and last observed-demand cutoff (target date minus two days) as LASSO/MA/Blends. A separate SARIMA sensitivity using a more recent observation (target date minus one day) is reported but not used in Table 2. No existing LASSO/MA model or Table 3/4 inventory policy is retuned.

### Table 3. Product-level policies (unchanged), **now with Opening and Closing inventory**

The same integer age-structured engine, same fixed 2.5-day supply / 1.0 safety factor, same order timing and same source forecasts are preserved. Each policy now returns its **opening on-hand units on the first test day**, **closing on-hand units after the last test day**, and a day-by-day conservation audit. Orders placed on the final test date are included in `Procured` but not yet received: these are shown as `Outstanding orders at test end` in the audit CSV. No value is imputed to make the paper's published inventory numbers match.
**Comparison scope:** Table 3's Historical mean is calculated from **training plus validation**
(known by the test start), whereas Table 2's Historical mean is fitted on **training only**.
Each fixed-buffer policy begins with its own forecast-derived opening inventory (not a
common observed hospital stock), so policy comparisons reflect this assumption.
The separate core-pipeline `11/12_test_inventory_*` CSVs are exploratory and **must not**
be substituted for the manuscript's Table 3/4 outputs.


In [ ]:

# Table 3 compares DIFFERENT FORECASTING MODELS at one fixed, prespecified inventory
# buffer (2.5 days of supply, safety factor 1.0) -- this is a separate, simpler
# simulation than the validation-selected, per-target grid search used for Table 4
# (see Table 4 section below), so it is reproduced by hand here rather than taken
# directly from the pipeline's own (grid-searched) inventory outputs.
#
# IMPORTANT ALIGNMENT DETAIL: platelet_analysis_v2.py's predictions CSV indexes each
# row by the ORIGIN date (the day features were last available), not the day being
# predicted -- i.e. for horizon h, row "date" d's prediction is for demand at d + h.
# (This falls directly out of target_series_for_horizon() using panel[target].shift(-h).)
# To build a forecast series that is aligned the same way as `demand` (indexed by the
# calendar day it describes), horizon-1 predictions must be shifted forward by 1 day.

cfg = pv2.SparseHurdleConfig()
panel, meta = pv2.load_platelet_excel("platelet_data_english_260529.xlsx")
targets = pv2.get_target_columns(cfg, panel)

train_mask = pv2.in_window(panel.index, cfg.train_start, cfg.train_end)
train_for_test_mask = pv2.in_window(panel.index, cfg.train_start, cfg.val_end)  # train+validation
test_mask = pv2.in_window(panel.index, cfg.test_start, cfg.test_end)
train_df = panel.loc[train_mask]
train_for_test_df = panel.loc[train_for_test_mask]
test_df = panel.loc[test_mask]

PRODUCT_TARGETS = {
    "PC": ["plt_transf_a", "plt_transf_b", "plt_transf_o", "plt_transf_ab"],
    "APC": ["aph_transf_a", "aph_transf_b", "aph_transf_o", "aph_transf_ab"],
}

pred_long = pd.read_csv("analysis_r2_s18s19/02_all_predictions_long.csv", parse_dates=["date"])
pred_long_test = pred_long[(pred_long["split"] == "test") & (pred_long["horizon"] == 1)].copy()

def prediction_long_to_wide_target_date(pred_long, model_name, horizon=1):
    g = pred_long[pred_long["model"] == model_name].copy()
    g["target_date"] = g["date"] + pd.Timedelta(days=horizon)
    return g.pivot_table(index="target_date", columns="target", values="pred", aggfunc="first")

SHELF_LIFE_DAYS, DELIVERY_LAG_DAYS = 4, 1
SHORTAGE_COST, WASTAGE_COST = 10.0, 1.0

def _as_int_units(x):
    return int(math.ceil(x - 1e-9)) if x > 0 else 0

def simulate_inventory_policy(demand, forecast, target_days_supply=2.5, safety_factor=1.0,
                               shelf_life_days=SHELF_LIFE_DAYS,
                               delivery_lag_days=DELIVERY_LAG_DAYS,
                               shortage_cost=SHORTAGE_COST, wastage_cost=WASTAGE_COST,
                               return_trace=False):
    """Unchanged original order-up-to engine plus opening/closing inventory audit.

    Procurement = all orders *placed* during the 181-day horizon; last-day orders
    can remain in transit, and are explicitly tracked in terminal pending orders.
    """
    demand = pd.Series(demand).fillna(0).clip(lower=0)
    forecast = pd.Series(forecast).reindex(demand.index).clip(lower=0)
    if forecast.isna().any():
        raise ValueError("Missing inventory forecast: check target_date mapping")
    dates = demand.index
    n = len(dates)
    inventory = [0 for _ in range(shelf_life_days)]
    inventory[-1] = _as_int_units(forecast.iloc[0] * target_days_supply * safety_factor)
    initial_opening = int(sum(inventory))
    pending = {}
    total_demand = total_procured = total_fulfilled = total_unmet = total_wasted = 0
    total_received = 0
    trace_rows = []
    for i, date in enumerate(dates):
        opening = int(sum(inventory))
        received_today = int(pending.pop(i, 0))
        total_received += received_today
        inventory[-1] += received_today
        d = _as_int_units(demand.iloc[i])
        remaining = d
        fulfilled = 0
        for bucket in range(shelf_life_days):
            use = min(inventory[bucket], remaining)
            inventory[bucket] -= use
            remaining -= use
            fulfilled += use
            if remaining <= 0:
                break
        unmet = remaining
        wasted_today = int(inventory[0])
        inventory[:-1] = inventory[1:]
        inventory[-1] = 0
        future_i = min(i + delivery_lag_days, n - 1)
        future_forecast = float(forecast.iloc[future_i])
        target_inventory = target_days_supply * future_forecast * safety_factor
        on_hand = int(sum(inventory))
        pending_qty = int(sum(pending.values()))
        order_qty = max(0, math.ceil(target_inventory - on_hand - pending_qty))
        total_procured += order_qty  # preserve published definition: orders placed
        arrival_i = i + delivery_lag_days
        if arrival_i < n and order_qty > 0:
            pending[arrival_i] = pending.get(arrival_i, 0) + order_qty
        closing = int(sum(inventory))
        if opening + received_today != fulfilled + wasted_today + closing:
            raise AssertionError(f"Daily stock balance failed on {date}")
        total_demand += d
        total_fulfilled += fulfilled
        total_unmet += unmet
        total_wasted += wasted_today
        trace_rows.append({
            "date":date,"opening_on_hand":opening,"arrivals":received_today,
            "demand":d,"fulfilled":fulfilled,"unmet":unmet,
            "wasted":wasted_today,"closing_on_hand":closing,
            "ordered":order_qty,
            "orders_not_arrived_cumulative":total_procured-total_received,
            "daily_balance_residual":opening+received_today-fulfilled-wasted_today-closing
        })
    last_closing=int(sum(inventory))
    outstanding=int(total_procured-total_received)
    assert all(row["daily_balance_residual"] == 0 for row in trace_rows)
    # Financial/inventory mass balance uses ordered units separately from arrivals.
    assert initial_opening+total_received == total_fulfilled+total_wasted+last_closing
    assert initial_opening+total_procured == total_fulfilled+total_wasted+last_closing+outstanding
    assert total_demand==total_fulfilled+total_unmet
    service_level=total_fulfilled/total_demand*100 if total_demand else np.nan
    wastage_rate=total_wasted/total_procured*100 if total_procured else np.nan
    cost=shortage_cost*total_unmet+wastage_cost*total_wasted
    summary={
        "Demand":total_demand,"Procured":total_procured,
        "Service level (%)":service_level,"Unmet":total_unmet,
        "Wastage rate (%)":wastage_rate,"Wasted":total_wasted,"Cost":cost,
        "Opening inventory (units)":initial_opening,
        "Closing inventory (units)":last_closing,
        "Outstanding orders at test end (units)":outstanding,
        "Received during test (units)":total_received,
        "Inventory balance residual (units)":0,
    }
    if return_trace:
        return summary,pd.DataFrame(trace_rows)
    return summary

POLICIES = {"Historical mean": None, "MA7": "moving_average_7d", "LASSO": "lasso",
            "Actual-demand forecast": None, "MA14": "moving_average_14d",
            "LASSO-MA7 blend": "blend_lasso_ma7_50_50", "LASSO-MA14 blend": "blend_lasso_ma14_50_50"}

rows = []
table3_traces = []
for product, ptargets in PRODUCT_TARGETS.items():
    demand = test_df[ptargets].sum(axis=1)
    for policy, model_name in POLICIES.items():
        if policy == "Historical mean":
            forecast = pd.Series(train_for_test_df[ptargets].sum(axis=1).mean(), index=test_df.index)
        elif policy == "Actual-demand forecast":
            forecast = demand.copy()
        else:
            pred_wide = prediction_long_to_wide_target_date(pred_long_test, model_name)
            available = [t for t in ptargets if t in pred_wide.columns]
            forecast = pred_wide[available].sum(axis=1).reindex(test_df.index)
        metrics_row, daily_trace = simulate_inventory_policy(
            demand,forecast,return_trace=True)
        daily_trace.insert(0,"Ordering policy",policy)
        daily_trace.insert(0,"Product",product)
        table3_traces.append(daily_trace)
        rows.append({"Product": product, "Ordering policy": policy, **metrics_row})

table3 = pd.DataFrame(rows)
for c in table3.select_dtypes(include=[float]).columns:
    table3[c] = table3[c].round(2)
# Persist both the paper-ready summary and the independently auditable daily ledger.
Path("tables_r2_s18s19").mkdir(parents=True,exist_ok=True)
Path("analysis_r2_s18s19").mkdir(parents=True,exist_ok=True)
main_columns=["Product","Ordering policy","Demand","Procured",
    "Service level (%)","Unmet","Wastage rate (%)","Wasted",
    "Opening inventory (units)","Closing inventory (units)"]
table3[main_columns].to_csv(
    "tables_r2_s18s19/Table3_With_Opening_Closing.csv",index=False)
pd.concat(table3_traces,ignore_index=True).to_csv(
    "analysis_r2_s18s19/Table3_Daily_Inventory_Trace.csv",index=False)
table3.to_csv("analysis_r2_s18s19/Table3_Full_Inventory_Audit.csv",index=False)
assert len(pd.concat(table3_traces))==2*len(POLICIES)*181
assert table3["Inventory balance residual (units)"].eq(0).all()
print("Saved Table 3 with Opening/Closing columns and 2534 audited day-policy rows")
print("Reproduced Table 3:\n")
print(table3.to_string(index=False))

print("\nNOTE: This is the authoritative manuscript Table 3 engine and daily inventory ledger.")
print("Historical mean policy uses pre-test TRAIN+VALIDATION, unlike Table 2 training-only baseline.")
print("Each ordering policy has its own forecast-derived opening inventory; not a common observed stock.")



### Table 4. Independent-test performance of validation-selected forecasting and inventory
policies by product-ABO target

**This does *not* use the pipeline's own built-in `simulate_inventory_for_target()` /
`expected_demand_window()` grid search** (that function's multi-horizon engine, run with
its default candidate set and its default `target_days_supply_grid` of
`{2.0, 2.5, 3.0, 3.5}`, does not reproduce the published Table 4 at all — wrong models for
every target, because that default selection also pulls in sparse/intermittent "hurdle"
two-stage models that never appear in the published table).

The methodology that *does* reproduce Table 4 — reverse-engineered and validated while
building this notebook — turns out to be the **same simple, single-horizon inventory
engine as Table 3** (`simulate_inventory_policy`, defined above), with **both** the
forecasting model **and** the (days-of-supply, safety-factor) buffer jointly grid-searched
per target by minimizing validation-period cost (10 x unmet + 1 x wasted), restricted to
the same 7 non-hurdle candidates reported elsewhere in the manuscript (Historical mean,
MA7, MA14, LASSO, Ridge, and the two LASSO-moving-average blends), over a
**`target_days_supply` grid of `{2.0, 2.5, 3.0}`** (safety factor `{0.90, 1.00, 1.10,
1.20}` as elsewhere). Using the wider `{..., 3.5}` grid changes 4 of the 8 selected
buffers and no longer matches. With this grid, model + days-of-supply + safety-factor
match the published Table 4 exactly for all 8 targets, and Demand/Service level/Unmet
match exactly or near-exactly; Procured/Wastage-rate/Wasted match to within a handful of
units (the same order of magnitude as the "last day's order" boundary effect already
documented in Table 3's own footnote).

**Seven inventory-policy candidate forecasting models:** historical mean, MA7, MA14,
LASSO, Ridge, LASSO–MA7 and LASSO–MA14. SARIMA, LightGBM, Chronos and sparse-hurdle
models are NOT part of the manuscript Table 4 joint validation-cost grid search.


In [ ]:

CANDIDATES = ["historical_mean", "moving_average_7d", "moving_average_14d", "lasso", "ridge",
              "blend_lasso_ma7_50_50", "blend_lasso_ma14_50_50"]
MODEL_LABELS_T4 = {"historical_mean": "Historical mean", "moving_average_7d": "MA7",
                   "moving_average_14d": "MA14", "lasso": "LASSO", "ridge": "Ridge",
                   "blend_lasso_ma7_50_50": "LASSO-MA7 blend", "blend_lasso_ma14_50_50": "LASSO-MA14 blend"}
TDS_GRID_T4 = (2.0, 2.5, 3.0)   # NOTE: narrower than cfg.target_days_supply_grid -- see markdown above
SF_GRID_T4 = (0.90, 1.00, 1.10, 1.20)

pred_val = pred_long[(pred_long["split"] == "validation") & (pred_long["horizon"] == 1)].copy()
pred_test = pred_long[(pred_long["split"] == "test") & (pred_long["horizon"] == 1)].copy()
val_df = panel.loc[pv2.in_window(panel.index, cfg.val_start, cfg.val_end)]

def select_table4_policy(target):
    # Grid-search (model, days_supply, safety_factor) jointly on the validation
    # period, minimizing cost, restricted to the 7 non-hurdle candidates.
    demand_val = val_df[target]
    best = None
    for model_name in CANDIDATES:
        fwide_val = prediction_long_to_wide_target_date(pred_val, model_name)
        if target not in fwide_val.columns:
            continue
        forecast_val = fwide_val[target].reindex(demand_val.index)
        for days in TDS_GRID_T4:
            for sf in SF_GRID_T4:
                m = simulate_inventory_policy(demand_val, forecast_val, days, sf)
                cost = SHORTAGE_COST * m["Unmet"] + WASTAGE_COST * m["Wasted"]
                key = (cost, m["Unmet"], m["Wasted"])
                if best is None or key < best[0]:
                    best = (key, model_name, days, sf)
    return best[1], best[2], best[3]

t4_rows = []
for t in targets:
    label = TARGET_LABELS[t]
    model_name, days, sf = select_table4_policy(t)
    demand_test = test_df[t]
    fwide_test = prediction_long_to_wide_target_date(pred_test, model_name)
    forecast_test = fwide_test[t].reindex(demand_test.index)
    m = simulate_inventory_policy(demand_test, forecast_test, days, sf)
    t4_rows.append({"Target": label, "Selected model": MODEL_LABELS_T4[model_name],
                     "Days of supply": days, "Safety factor": sf, "Demand": m["Demand"],
                     "Procured": m["Procured"], "Service level": m["Service level (%)"],
                     "Unmet": m["Unmet"], "Wastage rate": m["Wastage rate (%)"], "Wasted": m["Wasted"]})

t4_display = pd.DataFrame(t4_rows)
print("Reproduced Table 4 (per target, jointly grid-searched model + buffer):\n")
print(t4_display.to_string(index=False))

print("\nManuscript Table 4 is generated by the seven-model JOINT validation-cost grid search; no old reference numbers are used.")
pc_rows = [r for r in t4_rows if r["Target"].startswith("PC-")]
apc_rows = [r for r in t4_rows if r["Target"].startswith("APC-")]

def aggregate(rows):
    demand = sum(r["Demand"] for r in rows)
    procured = sum(r["Procured"] for r in rows)
    unmet = sum(r["Unmet"] for r in rows)
    wasted = sum(r["Wasted"] for r in rows)
    return {"Demand": demand, "Procured": procured, "Unmet": unmet, "Wasted": wasted,
            "Service level": (demand - unmet) / demand * 100, "Wastage rate": wasted / procured * 100}

print("\nReproduced PC aggregate:", aggregate(pc_rows))
print("\nReproduced APC aggregate:", aggregate(apc_rows))



### Supplementary Table S6. ABO-compatible substitution sensitivity analysis

Per Supplementary Methods S1.16, the no-substitution scenario (S0) is exactly the four
target-level simulations above run independently and summed by product -- i.e. it is the
PC/APC aggregate just reproduced. The substitution scenario (S1) additionally allows,
for each day, AB-group recipients' unmet demand to be filled first from that **same
day's** surplus A-group units (about to be wasted, i.e. the oldest-age bucket emptied at
end of day) and then from surplus B-group units (never O-group). This needs the daily
trace, not just the aggregate summary, so a small variant of `simulate_inventory_policy`
below also returns the day-by-day demand/fulfilled/unmet/wasted series.


In [ ]:

def simulate_inventory_policy_trace(demand, forecast, target_days_supply=2.5, safety_factor=1.0):
    # Same engine as simulate_inventory_policy, but also returns the full daily
    # trace (date, demand, fulfilled, unmet, wasted) needed for the ABO-substitution rule.
    demand = pd.Series(demand).fillna(0).clip(lower=0)
    forecast = pd.Series(forecast).reindex(demand.index).clip(lower=0)
    if forecast.isna().any():
        raise ValueError("Missing inventory forecast: check target_date mapping; no bfill permitted")
    dates = demand.index
    n = len(dates)
    inventory = [0 for _ in range(SHELF_LIFE_DAYS)]
    inventory[-1] = _as_int_units(forecast.iloc[0] * target_days_supply * safety_factor)
    pending = {}
    rows = []
    for i, date in enumerate(dates):
        received_today = int(pending.pop(i, 0))
        inventory[-1] += received_today
        d = _as_int_units(demand.iloc[i])
        remaining = d
        fulfilled = 0
        for bucket in range(SHELF_LIFE_DAYS):
            use = min(inventory[bucket], remaining)
            inventory[bucket] -= use
            remaining -= use
            fulfilled += use
            if remaining <= 0:
                break
        unmet = remaining
        wasted_today = int(inventory[0])
        inventory[:-1] = inventory[1:]
        inventory[-1] = 0
        future_i = min(i + DELIVERY_LAG_DAYS, n - 1)
        future_forecast = float(forecast.iloc[future_i])
        target_inventory = target_days_supply * future_forecast * safety_factor
        on_hand = int(sum(inventory))
        pending_qty = int(sum(pending.values()))
        order_qty = max(0, math.ceil(target_inventory - on_hand - pending_qty))
        arrival_i = i + DELIVERY_LAG_DAYS
        if arrival_i < n and order_qty > 0:
            pending[arrival_i] = pending.get(arrival_i, 0) + order_qty
        rows.append({"date": date, "demand": d, "fulfilled": fulfilled, "unmet": unmet,
                      "wasted": wasted_today, "procured_ordered": order_qty})
    return pd.DataFrame(rows).set_index("date")

# Re-derive each target's trace using the SAME (model, days, safety) selections found
# for Table 4 above (t4_rows), restricted to raw target names rather than display labels.
TARGET_OF_LABEL = {v: k for k, v in TARGET_LABELS.items()}
traces = {}
t4_by_label = {r["Target"]: r for r in t4_rows}
MODEL_KEY_OF_LABEL = {v: k for k, v in MODEL_LABELS_T4.items()}
for label, r in t4_by_label.items():
    t = TARGET_OF_LABEL[label]
    model_key = MODEL_KEY_OF_LABEL[r["Selected model"]]
    demand_test = test_df[t]
    fwide_test = prediction_long_to_wide_target_date(pred_test, model_key)
    forecast_test = fwide_test[t].reindex(demand_test.index)
    traces[label] = simulate_inventory_policy_trace(demand_test, forecast_test, r["Days of supply"], r["Safety factor"])

def apply_substitution(ab_label, a_label, b_label):
    tr_ab, tr_a, tr_b = traces[ab_label], traces[a_label], traces[b_label]
    sub_from_a = np.minimum(tr_ab["unmet"].values, tr_a["wasted"].values)
    remaining = tr_ab["unmet"].values - sub_from_a
    sub_from_b = np.minimum(remaining, tr_b["wasted"].values)
    sub_days = [(str(d.date()), int(sa), int(sb)) for d, sa, sb in zip(tr_ab.index, sub_from_a, sub_from_b) if sa > 0 or sb > 0]
    return int(sub_from_a.sum()), int(sub_from_b.sum()), sub_days

def s0_s1_aggregate(labels, ab_label, a_label, b_label):
    s0 = aggregate([t4_by_label[l] for l in labels])
    sub_a, sub_b, sub_days = apply_substitution(ab_label, a_label, b_label)
    unmet_s1 = s0["Unmet"] - sub_a - sub_b
    wasted_s1 = s0["Wasted"] - sub_a - sub_b
    s1 = {"Demand": s0["Demand"], "Procured": s0["Procured"], "Unmet": unmet_s1, "Wasted": wasted_s1,
          "Service level": (s0["Demand"] - unmet_s1) / s0["Demand"] * 100,
          "Wastage rate": wasted_s1 / s0["Procured"] * 100}
    substitution_share = (sub_a + sub_b) / (s0["Demand"] - unmet_s1) * 100
    return s0, s1, sub_a, sub_b, sub_days, substitution_share

pc_s0, pc_s1, pc_suba, pc_subb, pc_subdays, pc_subshare = s0_s1_aggregate(
    ["PC-A", "PC-B", "PC-O", "PC-AB"], "PC-AB", "PC-A", "PC-B")
apc_s0, apc_s1, apc_suba, apc_subb, apc_subdays, apc_subshare = s0_s1_aggregate(
    ["APC-A", "APC-B", "APC-O", "APC-AB"], "APC-AB", "APC-A", "APC-B")

print("Reproduced Supplementary Table S6, Part A:\n")
print(f"  PC  No substitution (S0): {pc_s0}")
print(f"  PC  Substitution    (S1): {pc_s1}  substitution_share={pc_subshare:.2f}%")
print(f"  APC No substitution (S0): {apc_s0}")
print(f"  APC Substitution    (S1): {apc_s1}  substitution_share={apc_subshare:.2f}%")

print(f"\nSubstitution events -- PC: from A={pc_suba}, from B={pc_subb}, days={pc_subdays}")
print(f"Substitution events -- APC: from A={apc_suba}, from B={apc_subb}, days={apc_subdays}")

print("\nNOTE: S6 is an exploratory POST-HOC allocation against independently simulated ABO-group daily traces, not a coupled inventory recalculation.")



**Note:** Part B of Supplementary Table S6 (per-ABO-group identical-type-fulfilled /
substitution-fulfilled / unmet percentages) must be recomputed from the newest Table 4 trace and is not automatically unchanged by the correction above -- it
depends only on each group's own Demand and Unmet, neither of which changed -- and is not
separately reproduced here.



## 3. Figure 1 and Figure 2

Both figures are produced directly by the pipeline script's own plotting functions,
reading the raw panel and the validation-selected predictions it just wrote out.


In [ ]:

fig_cfg = pv2.FigureConfig(
    data_path="platelet_data_english_260529.xlsx",
    output_dir="figures_r2_s18s19",
    all_predictions_csv="analysis_r2_s18s19/02_all_predictions_long.csv",
    forecast_label="LASSO and MA7 forecasts",
    train_start=cfg.train_start, train_end=cfg.train_end,
    val_start=cfg.val_start, val_end=cfg.val_end,
    test_start=cfg.test_start, test_end=cfg.test_end,
)
panel_fig, meta_fig = pv2.load_daily_panel_and_metadata(fig_cfg.data_path)
pred_df_fig = pd.read_csv(fig_cfg.all_predictions_csv)

fig1_path = pv2.plot_figure1_temporal_patterns(panel=panel_fig, meta=meta_fig, cfg=fig_cfg)
print("Figure 1 saved to:", fig1_path)

from IPython.display import Image, display
display(Image(filename=str(fig1_path)))


In [ ]:
fig2_path = pv2.plot_figure2_product_abo_forecasts(
    panel=panel_fig, cfg=fig_cfg, pred_df=pred_df_fig
)
print("Figure 2 (8 product–ABO panels, LASSO + MA7) saved to:", fig2_path)
display(Image(filename=str(fig2_path)))


**Figure 2:** Eight product–ABO panels (PC-A, PC-B, PC-O, PC-AB,
APC-A, APC-B, APC-O, APC-AB) over the independent test period. Each panel
shows actual daily demand, the 30-day rolling mean, the training mean,
LASSO, and MA7. These two forecast series come from
`analysis/02_all_predictions_long.csv` (not the validation-selected model CSV).
The 30-day rolling mean is computed with pre-test demand history.


**Important date-alignment audit:** Forecast CSV dates are *origin* dates. Figure 2 shifts to `target_date = date + horizon` and includes the final validation-origin forecast (2024-08-31) to show the first test outcome (2024-09-01). **Methodological caveat:** Training/validation/test splits in the underlying pipeline are based on origin dates; the reported 181-day independent test and strict validation separation should be reevaluated using target-date splits. This figure-only repair does not revise the paper's model metrics or inventory calculations.


In [ ]:

pred_lasso_only = pd.read_csv("analysis_r2_s18s19/02_all_predictions_long.csv", parse_dates=["date"])
pred_lasso_only = pred_lasso_only[(pred_lasso_only["model"] == "lasso") & (pred_lasso_only["horizon"] == 1)]

for product in ["PC", "APC"]:
    path = pv2.plot_product_abo_trends_for_model(
        panel=panel_fig, pred_df_model=pred_lasso_only, model_name="lasso", product=product,
        output_dir="figures_r2_s18s19", train_start=cfg.train_start, train_end=cfg.train_end,
        test_start=cfg.test_start, test_end=cfg.test_end,
    )
    print(f"Per-target alternative figure ({product}) saved to:", path)
    display(Image(filename=str(path)))



## 4. Supplementary Table S4 — paired differences vs. MA14 (ΔMAE, bootstrap CI,
Diebold-Mariano) and mean MASE with bootstrap CI

Methodology (Supplementary Methods, Table S4 footnote): 7-day moving-block bootstrap
(5,000 replicates, seed 2026) for the 95% CI of mean ΔMAE; two-sided Diebold-Mariano test
with Newey-West/Bartlett-kernel HAC variance (max lag 7), referred to a Student-t(n-1)
distribution; Holm correction across the three comparisons against MA14 (a separate
family from the original LASSO-vs-blend comparisons).


In [ ]:

from scipy import stats

RNG_SEED, N_BOOT, BLOCK_LEN, MAX_LAG = 2026, 5000, 7, 7
MODEL_MAP_S4 = {"MA14": "moving_average_14d", "LASSO": "lasso",
                "LASSO-MA7 blend": "blend_lasso_ma7_50_50", "LASSO-MA14 blend": "blend_lasso_ma14_50_50"}

pred_long_all = pd.read_csv("analysis_r2_s18s19/02_all_predictions_long.csv", parse_dates=["date"])
pred_test_h1 = pred_long_all[(pred_long_all["split"] == "test") & (pred_long_all["horizon"] == 1)].copy()

def per_target_ae_wide(pl, model_name):
    g = pl[pl["model"] == model_name].copy()
    g["ae"] = (g["actual"] - g["pred"]).abs()
    return g.pivot_table(index="date", columns="target", values="ae", aggfunc="first")

def daily_mean_ae(pl, model_name, tlist):
    return per_target_ae_wide(pl, model_name).reindex(columns=tlist).mean(axis=1)

def moving_block_bootstrap_indices(n, block_len, n_boot, rng):
    n_blocks = int(np.ceil(n / block_len))
    out = np.empty((n_boot, n_blocks * block_len), dtype=int)
    for b in range(n_boot):
        starts = rng.integers(0, n, size=n_blocks)
        out[b] = np.concatenate([(s + np.arange(block_len)) % n for s in starts])
    return out[:, :n]

def bootstrap_ci_mean(vals, n_boot=N_BOOT, block_len=BLOCK_LEN, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    idx = moving_block_bootstrap_indices(len(vals), block_len, n_boot, rng)
    vals = np.asarray(vals)
    bm = vals[idx].mean(axis=1)
    lo, hi = np.percentile(bm, [2.5, 97.5])
    return float(np.mean(vals)), float(lo), float(hi)

def newey_west_bartlett_var(d, max_lag=MAX_LAG):
    n = len(d)
    x = np.asarray(d, dtype=float) - np.mean(d)
    var = np.dot(x, x) / n
    for k in range(1, max_lag + 1):
        w = 1 - k / (max_lag + 1)
        var += 2 * w * (np.dot(x[k:], x[:-k]) / n)
    return var / n

def dm_test(d, max_lag=MAX_LAG):
    n = len(d)
    dbar = np.mean(d)
    var_dbar = newey_west_bartlett_var(d, max_lag=max_lag)
    dm_stat = dbar / np.sqrt(var_dbar)
    pval = 2 * (1 - stats.t.cdf(np.abs(dm_stat), df=n - 1))
    return dm_stat, pval

def holm_correction(pvals):
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    m = len(pvals)
    adj = np.empty(m)
    running_max = 0
    for rank, idx in enumerate(order):
        val = (m - rank) * pvals[idx]
        running_max = max(running_max, val)
        adj[idx] = min(running_max, 1.0)
    return adj

daily_ae = {m: daily_mean_ae(pred_test_h1, mn, targets) for m, mn in MODEL_MAP_S4.items()}
dates = daily_ae["MA14"].dropna().index
for m in daily_ae:
    daily_ae[m] = daily_ae[m].reindex(dates)
print("N test days used:", len(dates))

rows, raw_p = [], []
for m in ["LASSO", "LASSO-MA7 blend", "LASSO-MA14 blend"]:
    d = (daily_ae[m] - daily_ae["MA14"]).values
    mean_d, lo, hi = bootstrap_ci_mean(d)
    dm_stat, pval = dm_test(d)
    rows.append({"Comparison": f"{m} vs MA14", "\u0394MAE": round(mean_d, 3),
                 "95% CI": f"{lo:.3f} to {hi:.3f}", "DM stat": round(dm_stat, 3), "raw P": round(pval, 4)})
    raw_p.append(pval)
holm_p = holm_correction(raw_p)
for i, r in enumerate(rows):
    r["Holm P"] = round(holm_p[i], 4)
s4a = pd.DataFrame(rows)
print("\nReproduced Table S4 Part A (new rows, vs MA14):\n")
print(s4a.to_string(index=False))

# Mean MASE with bootstrap CI (training-period denominator, horizon 1)
X_all_s4 = pv2.build_feature_frame(panel, targets, horizon=1, windows=cfg.moving_windows)
denoms = {}
for t in targets:
    y_all = pv2.target_series_for_horizon(panel, t, horizon=1)
    _, y_train = pv2.split_X_y_from_all(X_all_s4, y_all, cfg.train_start, cfg.train_end)
    yt = y_train.values
    denoms[t] = np.mean(np.abs(yt[7:] - yt[:-7]))

mase_rows = []
for m, mn in MODEL_MAP_S4.items():
    wide = per_target_ae_wide(pred_test_h1, mn).reindex(columns=targets)
    mase_per_t = {t: wide[t] / denoms[t] for t in targets}
    mase_daily_mean = pd.DataFrame(mase_per_t).mean(axis=1).reindex(dates)
    mean_mase, lo, hi = bootstrap_ci_mean(mase_daily_mean.values)
    mase_rows.append({"Model": m, "Mean MASE": round(mean_mase, 3), "95% CI": f"{lo:.3f}-{hi:.3f}"})
s4b = pd.DataFrame(mase_rows)
print("\nReproduced Table S4 Part B (mean MASE):\n")
print(s4b.to_string(index=False))


### Exploratory paired test: **information-matched** SARIMA vs LASSO–MA14

The two forecasts share all 181 target dates and 8 ABO series. The test is an *exploratory* paired loss comparison; do not claim a significant winner from mean MAE alone.

In [ ]:
# Exploratory, paired, NOT a new primary claim: SARIMA vs best LASSO-MA14 blend.
# Both models evaluated on the exact same 181 target dates and eight ABO targets.
_sar=pd.read_csv("analysis_r2_s18s19/SARIMA_S19_rolling_h1_matched_predictions_181days.csv",
                 parse_dates=["date"])
_main=pd.read_csv("analysis_r2_s18s19/02_all_predictions_long.csv",
                  parse_dates=["date"])
_blend=_main.loc[(_main["split"]=="test") & (_main["horizon"]==1) &
    (_main["model"]=="blend_lasso_ma14_50_50")].copy()
_blend["target_date"]=_blend["date"]+pd.Timedelta(days=1)
_blend["target_label"]=_blend["target"].map(TARGET_LABELS)
_sar["ae_sarima"]=( _sar["actual"]-_sar["pred"] ).abs()
_blend["ae_blend"]=( _blend["actual"]-_blend["pred"] ).abs()
_join=_sar.merge(_blend[["target_date","target_label","actual","ae_blend"]],
    left_on=["date","target"],right_on=["target_date","target_label"],
    validate="one_to_one",suffixes=("","_blend"))
assert len(_join)==181*8
assert np.allclose(_join["actual"],_join["actual_blend"])
_ae_daily=(_join.assign(diff=_join["ae_sarima"]-_join["ae_blend"])
                  .groupby("date")["diff"].mean())
assert len(_ae_daily)==181
_mean_delta,_lower,_upper=bootstrap_ci_mean(_ae_daily.to_numpy())
_dm_stat,_rawp=dm_test(_ae_daily.to_numpy())
comparison=pd.DataFrame([{
    "Comparison":"SARIMA rolling h=1 minus LASSO-MA14",
    "Delta MAE (units/day)":_mean_delta,
    "Bootstrap 95% CI lower":_lower,"Bootstrap 95% CI upper":_upper,
    "DM statistic":_dm_stat,"P (unadjusted, exploratory)":_rawp,
    "n_dates":181,"n_targets":8}])
comparison.to_csv("analysis_r2_s18s19/SARIMA_vs_LASSO_MA14_paired_audit.csv",index=False)
print(comparison.round(4).to_string(index=False))
print("This is exploratory; do not claim SARIMA superiority solely from numeric MAE.")



## 5. Supplementary Table S5 — rolling-origin sensitivity (Part A) and refitting-protocol
sensitivity (Part B)

**Part A:** six monthly folds (Sep 2024 - Feb 2025), each using an expanding training
window ending the day before that fold's evaluation month begins; LASSO is refit per
fold per target with the same fixed hyperparameters (alpha=0.05) as the primary
analysis, while MA7/MA14/the blends are computed causally without refitting.


In [ ]:

FOLDS = [("2024-09-01", "2024-09-30"), ("2024-10-01", "2024-10-31"), ("2024-11-01", "2024-11-30"),
         ("2024-12-01", "2024-12-31"), ("2025-01-01", "2025-01-31"), ("2025-02-01", "2025-02-28")]

X_all_fold = pv2.build_feature_frame(panel, targets, horizon=1, windows=cfg.moving_windows)
fold_results = []
for fold_start, fold_end in FOLDS:
    train_end_fold = (pd.Timestamp(fold_start) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    fold_ae = {"MA14": [], "LASSO": [], "LASSO-MA7 blend": [], "LASSO-MA14 blend": []}
    for t in targets:
        y_all = pv2.target_series_for_horizon(panel, t, horizon=1)
        X_train, y_train = pv2.split_X_y_from_all(X_all_fold, y_all, cfg.train_start, train_end_fold)
        X_eval, y_eval = pv2.split_X_y_from_all(X_all_fold, y_all, fold_start, fold_end)
        if len(y_train) == 0 or len(y_eval) == 0:
            continue
        lasso_m = pv2.make_lasso(alpha=0.05, random_state=cfg.random_state).fit(X_train, y_train)
        lasso_pred = pd.Series(np.maximum(lasso_m.predict(X_eval), 0.0), index=y_eval.index)
        y_full = panel[t].astype(float)
        ma7 = pv2.rolling_forecast_series(y_full, y_eval.index, horizon=1, window=7, kind="mean")
        ma14 = pv2.rolling_forecast_series(y_full, y_eval.index, horizon=1, window=14, kind="mean")
        fold_ae["MA14"].append((y_eval - ma14).abs())
        fold_ae["LASSO"].append((y_eval - lasso_pred).abs())
        fold_ae["LASSO-MA7 blend"].append((y_eval - (0.5 * lasso_pred + 0.5 * ma7)).abs())
        fold_ae["LASSO-MA14 blend"].append((y_eval - (0.5 * lasso_pred + 0.5 * ma14)).abs())
    row = {"Fold": fold_start}
    for mname in fold_ae:
        row[mname] = pd.concat(fold_ae[mname]).mean()
    fold_results.append(row)

s5a = pd.DataFrame(fold_results)
print("Reproduced Table S5 Part A (monthly-fold mean MAE):\n")
print(s5a.round(3).to_string(index=False))
print("\nMean (6 folds):")
print(s5a[["MA14", "LASSO", "LASSO-MA7 blend", "LASSO-MA14 blend"]].mean().round(3))



**Part B:** refitting-protocol sensitivity. Protocol A is the primary, train-only fit
(through Feb 2024); Protocol B refits the same models on the combined training-plus-
validation period (through Aug 2024) before the same test-period evaluation. This cell
re-runs the pipeline's own `run_analysis()` with the training window extended through
the validation period (val_start = val_end = the last validation day, so the
validation-dependent code paths still run on a minimal 1-day window without being
exercised for anything that matters here). This takes another 1-3 minutes.


In [ ]:
# Protocol B — same models and parameters, fit on train+validation only.
# Do not create a fictitious one-day validation set that overlaps model fitting.
from pathlib import Path
from sklearn.base import clone
X_all_B = pv2.build_feature_frame(panel, targets, horizon=1, windows=cfg.moving_windows)
protoB_rows = []
for t in targets:
    y_all_B = pv2.target_series_for_horizon(panel, t, horizon=1)
    X_fit_B, y_fit_B = pv2.split_X_y_from_all(
        X_all_B, y_all_B, cfg.train_start, cfg.val_end, horizon=1
    )
    X_eval_B, y_eval_B = pv2.split_X_y_from_all(
        X_all_B, y_all_B, cfg.test_start, cfg.test_end, horizon=1
    )
    lasso_B = pv2.make_lasso(alpha=0.05, random_state=cfg.random_state).fit(X_fit_B, y_fit_B)
    y_lasso = np.maximum(lasso_B.predict(X_eval_B), 0)
    ma7_B = pv2.rolling_forecast_series(panel[t].astype(float), y_eval_B.index, 1, 7)
    ma14_B = pv2.rolling_forecast_series(panel[t].astype(float), y_eval_B.index, 1, 14)
    model_preds = {
        "lasso": y_lasso,
        "moving_average_14d": ma14_B.to_numpy(),
        "blend_lasso_ma7_50_50": 0.5*y_lasso+0.5*ma7_B.to_numpy(),
        "blend_lasso_ma14_50_50": 0.5*y_lasso+0.5*ma14_B.to_numpy(),
    }
    for key, pred in model_preds.items():
        for d, actual, pred_val in zip(y_eval_B.index, y_eval_B.to_numpy(), pred):
            protoB_rows.append(dict(date=d, target=t, split='test', horizon=1,
                                    model=key, actual=float(actual), pred=float(pred_val)))
pred_protoB = pd.DataFrame(protoB_rows)
assert pred_protoB.groupby(['target','model']).size().eq(181).all()
print('Protocol B: train+validation fitting complete, without overlapping validation')


In [ ]:

pred_protoB_h1 = pred_protoB[(pred_protoB["split"] == "test") & (pred_protoB["horizon"] == 1)].copy()

rowsB = []
for m, mn in MODEL_MAP_S4.items():
    maeA = per_target_ae_wide(pred_test_h1, mn).reindex(columns=targets).mean(axis=0).mean()
    maeB = per_target_ae_wide(pred_protoB_h1, mn).reindex(columns=targets).mean(axis=0).mean()
    rowsB.append({"Model": m, "Protocol A (train-only) mean MAE": round(maeA, 3),
                  "Protocol B (train+val refit) mean MAE": round(maeB, 3), "\u0394 (B-A)": round(maeB - maeA, 3)})
s5b = pd.DataFrame(rowsB)
s5b["Rank A"] = s5b["Protocol A (train-only) mean MAE"].rank().astype(int)
s5b["Rank B"] = s5b["Protocol B (train+val refit) mean MAE"].rank().astype(int)
print("Reproduced Table S5 Part B:\n")
print(s5b.to_string(index=False))



## 6. Supplementary Table S8 — penalty-specification sensitivity (fixed vs.
cross-validated LASSO/Ridge alpha)

**SLOW (a few minutes):** 5-fold `TimeSeriesSplit` cross-validation, training period
only, over alpha grids {0.005, 0.01, 0.03, 0.05, 0.10, 0.30, 0.50} for LASSO and
{0.1, 0.5, 1, 5, 10, 50} for Ridge; the selected alpha is then refit on the full
training period and evaluated once on the test period.


In [ ]:

from sklearn.model_selection import TimeSeriesSplit

LASSO_GRID = [0.005, 0.01, 0.03, 0.05, 0.10, 0.30, 0.50]
RIDGE_GRID = [0.1, 0.5, 1, 5, 10, 50]
N_SPLITS = 5

X_all_s8 = pv2.build_feature_frame(panel, targets, horizon=1, windows=cfg.moving_windows)
rows_s8 = []
for t in targets:
    y_all = pv2.target_series_for_horizon(panel, t, horizon=1)
    X_train, y_train = pv2.split_X_y_from_all(X_all_s8, y_all, cfg.train_start, cfg.train_end)
    X_test, y_test = pv2.split_X_y_from_all(X_all_s8, y_all, cfg.test_start, cfg.test_end)
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)

    def cv_best_alpha(make_model_fn, grid):
        best_alpha, best_mae = None, np.inf
        for alpha in grid:
            fold_maes = []
            for tr_idx, va_idx in tscv.split(X_train):
                mdl = make_model_fn(alpha)
                mdl.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
                pred = np.maximum(mdl.predict(X_train.iloc[va_idx]), 0.0)
                fold_maes.append(np.mean(np.abs(y_train.iloc[va_idx].values - pred)))
            mean_mae = float(np.mean(fold_maes))
            if mean_mae < best_mae:
                best_mae, best_alpha = mean_mae, alpha
        return best_alpha

    lasso_alpha = cv_best_alpha(lambda a: pv2.make_lasso(alpha=a, random_state=cfg.random_state), LASSO_GRID)
    ridge_alpha = cv_best_alpha(lambda a: pv2.make_ridge(alpha=a), RIDGE_GRID)

    lasso_fixed = pv2.make_lasso(alpha=0.05, random_state=cfg.random_state).fit(X_train, y_train)
    lasso_cv = pv2.make_lasso(alpha=lasso_alpha, random_state=cfg.random_state).fit(X_train, y_train)
    ridge_fixed = pv2.make_ridge(alpha=5.0).fit(X_train, y_train)
    ridge_cv = pv2.make_ridge(alpha=ridge_alpha).fit(X_train, y_train)

    rows_s8.append({
        "Target": TARGET_LABELS[t],
        "LASSO fixed (a=0.05)": np.mean(np.abs(y_test.values - np.maximum(lasso_fixed.predict(X_test), 0))),
        "LASSO CV-a (best)": lasso_alpha,
        "LASSO CV MAE": np.mean(np.abs(y_test.values - np.maximum(lasso_cv.predict(X_test), 0))),
        "Ridge fixed (a=5.0)": np.mean(np.abs(y_test.values - np.maximum(ridge_fixed.predict(X_test), 0))),
        "Ridge CV-a (best)": ridge_alpha,
        "Ridge CV MAE": np.mean(np.abs(y_test.values - np.maximum(ridge_cv.predict(X_test), 0))),
    })

s8 = pd.DataFrame(rows_s8)  # Raw precision; use this for all summary calculations.
numeric_cols = [c for c in s8.columns if c not in ("Target", "LASSO CV-a (best)", "Ridge CV-a (best)")]
s8_display = s8.copy()
s8_display[numeric_cols] = s8_display[numeric_cols].round(3)
print("Reproduced Table S8 (individual cells rounded only for display):\n")
print(s8_display.to_string(index=False))
print("\nMean LASSO fixed -> CV:",
      round(float(s8["LASSO fixed (a=0.05)"].mean()),3), "->",
      round(float(s8["LASSO CV MAE"].mean()),3))
print("Mean Ridge fixed -> CV:",
      round(float(s8["Ridge fixed (a=5.0)"].mean()),3), "->",
      round(float(s8["Ridge CV MAE"].mean()),3))
Path("tables_r2_s18s19").mkdir(exist_ok=True)
s8.to_csv("tables_r2_s18s19/S8_penalty_sensitivity_full_precision.csv",index=False)
assert round(float(s8["Ridge fixed (a=5.0)"].mean()),3)==6.016


## 8. SARIMA fair 181-day **information-matched** execution record

SARIMA is fitted on training dates only. On every evaluation day d, its main forecast only uses observed data through d−2, matching the LASSO/MA history-lag policy; the more recent d−1 forecast is a sensitivity analysis. This cell only reads the metadata; no additional refitting is performed.

In [ ]:

RUN_CHRONOS = False  # set True to actually run this (slow) cell

if RUN_CHRONOS:
    import subprocess, importlib
    if importlib.util.find_spec("chronos") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting==2.3.2"])
    import torch
    from chronos import ChronosPipeline

    TEST_START, TEST_END = pd.Timestamp("2024-09-01"), pd.Timestamp("2025-02-28")
    TRAIN_START, TRAIN_END = pd.Timestamp("2019-03-01"), pd.Timestamp("2024-02-29")
    MODEL_ID, NUM_SAMPLES, SEED = "amazon/chronos-t5-small", 20, 2026

    df_chronos = panel[targets].rename(columns=TARGET_LABELS).sort_index()
    test_dates = df_chronos.loc[TEST_START:TEST_END].index
    train_chronos = df_chronos.loc[TRAIN_START:TRAIN_END]

    mase_denom = {}
    for col in df_chronos.columns:
        y = train_chronos[col].to_numpy(dtype=float)
        mase_denom[col] = float(np.mean(np.abs(y[7:] - y[:-7])))

    torch.manual_seed(SEED); np.random.seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32
    print("Loading", MODEL_ID, "on", device, "...")
    pipeline = ChronosPipeline.from_pretrained(MODEL_ID, device_map=device, torch_dtype=dtype)

    prediction_rows = []
    for i, forecast_date in enumerate(test_dates, start=1):
        hist = df_chronos.loc[df_chronos.index < forecast_date]
        contexts = [torch.tensor(hist[c].to_numpy(dtype=np.float32), dtype=torch.float32) for c in df_chronos.columns]
        forecast = pipeline.predict(inputs=contexts, prediction_length=1, num_samples=NUM_SAMPLES)
        if isinstance(forecast, list):
            forecast = torch.stack(forecast)
        arr = forecast.detach().cpu().numpy()
        point = np.clip(np.median(arr[:, :, 0], axis=1), 0.0, None)
        for j, col in enumerate(df_chronos.columns):
            prediction_rows.append({"date": forecast_date, "target": col,
                                     "y_true": float(df_chronos.loc[forecast_date, col]), "y_pred": float(point[j])})
        if i == 1 or i % 20 == 0 or i == len(test_dates):
            print(f"{i:3d}/{len(test_dates)} completed: {forecast_date.date()}")

    pred_chronos = pd.DataFrame(prediction_rows)
    metric_rows = []
    for col, g in pred_chronos.groupby("target", sort=False):
        mae = float(np.mean(np.abs(g["y_true"] - g["y_pred"])))
        metric_rows.append({"Target": col, "Chronos MAE": mae, "Chronos MASE": mae / mase_denom[col]})
    s7 = pd.DataFrame(metric_rows)
    print("\nReproduced Table S7:\n")
    print(s7.round(3).to_string(index=False))
    print("\nMean Chronos MAE / MASE:", s7["Chronos MAE"].mean().round(3), "/", s7["Chronos MASE"].mean().round(3))
    print("Published: mean MAE 5.872, mean MASE 1.367.")
else:
    print("RUN_CHRONOS is False -- set it to True above and re-run this cell to execute Chronos (slow).")


## 8. SARIMA information-matched sequential prediction verification

The SARIMA forecast was already generated in the benchmark section. This verification cell checks the **exact** metadata filename produced by the benchmark, its `rolling_h1_availability_matched` protocol, the convergence flags, and coverage of the 8 targets × 181 test dates. It does not re-fit or modify model outputs.


In [ ]:
# Corrected SARIMA audit: match the producer's actual metadata filename and protocol.
# Does not refit SARIMA; reads only files produced earlier in this notebook.
from pathlib import Path
import json
import pandas as pd
import numpy as np

sarima_dir = Path("analysis_r2_s18s19")
prefix = "SARIMA_S19_rolling_h1_matched"
sarima_report = sarima_dir / f"{prefix}_run_metadata.json"
sarima_metrics = sarima_dir / f"{prefix}_metrics_by_target.csv"
sarima_predictions = sarima_dir / f"{prefix}_predictions_181days.csv"

for required_file in (sarima_report, sarima_metrics, sarima_predictions):
    if not required_file.is_file():
        existing = sorted(p.name for p in sarima_dir.glob("SARIMA_S19_*"))
        raise FileNotFoundError(
            f"SARIMA producer output not found: {required_file.resolve()}\n"
            f"Available SARIMA outputs in {sarima_dir.resolve()}: {existing}\n"
            "Execute the SARIMA benchmark cell first (cell 14); do not reuse an old static forecast."
        )

report = json.loads(sarima_report.read_text(encoding="utf-8"))
assert report.get("protocol") == "rolling_h1_availability_matched", report
assert report.get("optimizer_all_converged") is True, report
assert int(report.get("n_test_days", 0)) == 181, report
assert int(report.get("n_targets", 0)) == 8, report

met = pd.read_csv(sarima_metrics)
pr = pd.read_csv(sarima_predictions)
assert len(met) == 8 and met["n_test_days"].eq(181).all(), met
assert len(pr) == 8 * 181, len(pr)
assert pr["target"].nunique() == 8 and pr["date"].nunique() == 181
assert pr["protocol"].eq("rolling_h1_availability_matched").all()
assert pr["forecast_horizon_from_origin"].eq(1).all()
assert np.isclose(float(met["sarima_mae"].mean()), float(report["mean_mae"]), atol=1e-10)
assert np.isclose(float(met["sarima_mase_s113"].mean()), float(report["mean_mase_s113"]), atol=1e-10)

print("SARIMA verification: PASS")
print("Protocol:", report["protocol"])
print("Coverage: 8 target series x 181 test days")
print("Mean MAE:", round(float(report["mean_mae"]), 3))
print("Mean MASE (S1.13):", round(float(report["mean_mase_s113"]), 3))
print("More recent-data sensitivity MAE:",
      round(float(report["supplementary_mae_latest_prior_day"]), 3))


## R2 minimal-change audit notes

**Retained:** primary model families, h=1, LASSO/Ridge fixed penalties, moving-average definitions, equal blend weights, 181-day test, Figures 1 and 2, Table 4 candidate model selection and policy parameter grids, and supplementary analysis cells.

**Changed here:** SARIMA Table 2 row is now strictly comparable **rolling one-day-ahead** (trained through 2024-02-29, observed validation used only to update state, test observations incorporated after each prediction). This replaces the former fixed 181-step SARIMA row; its MAE, MASE, mean rank and accompanying prose must be updated in all three revision documents. Table 3 now reports beginning and ending **on-hand** units plus daily inventory ledger and pending orders, without changing how orders or wastage are simulated.

**Scope caveat:** SARIMA is an independently audited **accuracy-only** comparator; it remains excluded from the validation-based Table 4 inventory-policy candidates. A new model ranking is not proof of statistically significant superiority; the paired S4-like exploratory test included below provides that context.


**R2 v3 correctness checks:**
- Seasonal naïve now uses target-date minus seven days; see the executable QA in cell 18.
- The *exploratory core inventory* uses only demand before validation/test for stock initialization
  (not the current test's actual mean); a scope file accompanies the core inventory CSVs.
- Manuscript Tables 3/4 continue using the original, separately audited daily engine.
- The Table 3 historical-mean ordering comparator uses training+validation history, while
  the Table 2 historical-mean prediction benchmark is training-only.
- Table 4 jointly selects forecasts and buffers among seven candidate model families.
- S8 means are computed before rounding. Chronos is optional (`RUN_CHRONOS=False`).


## 9. Archive the revised output without overwriting older revision results

In [ ]:
# Safe unique archive: never merge or overwrite previous run outputs.
from pathlib import Path
import hashlib, json, shutil, sys, importlib.metadata as md
SAVE_TO_DRIVE = True
REPORT_DIRS = ('analysis_r2_s18s19', 'tables_r2_s18s19', 'figures_r2_s18s19')
RUN_ROOT = WORKDIR
try:
    versions = {p: md.version(p) for p in
                ('numpy','pandas','scikit-learn','scipy','lightgbm','statsmodels','matplotlib')}
except Exception:
    versions = {}
manifest = {
    'run_id': RUN_ID,
    'notebook': 'Platelet_R2_Validated_Integrated_v3.ipynb',
    'python': sys.version.split()[0], 'packages': versions,
    'input_filename': DATA_FILENAME,
    'input_sha256': hashlib.sha256(Path(DATA_FILENAME).read_bytes()).hexdigest(),
    'n_test_target_days': 181, 'n_product_ABO_targets': 8,
    'seasonal_naive_source': 'target d minus 7 days',
    'core_inventory_initialization': 'pre-period historical mean; no test mean',
    'core_inventory_scope': 'exploratory, not manuscript Tables 3/4',
    'table3_inventory_scope': 'fixed 2.5-day buffer; train+validation historical comparator',
    'table4_inventory_scope': 'joint validation-cost optimization, 7 candidate forecasts',
    'chronos_executed': bool(globals().get('RUN_CHRONOS', False)),
}
(RUN_ROOT/'R2_reproducibility_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
assert all((RUN_ROOT/d).is_dir() for d in REPORT_DIRS)
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/Platelets/Reproducibility_Results') / 'R2_VALIDATED_V3_RUNS' / RUN_ID
    if drive_dir.exists():
        raise FileExistsError(f'Refusing to overwrite prior run: {drive_dir}')
    drive_dir.mkdir(parents=True)
    for name in REPORT_DIRS:
        shutil.copytree(RUN_ROOT/name, drive_dir/name)  # no dirs_exist_ok
    shutil.copy2(RUN_ROOT/'R2_reproducibility_manifest.json',drive_dir/'R2_reproducibility_manifest.json')
    print('Unique verified R2 run archive:',drive_dir)
else:
    print('Unique local R2 output directory:',RUN_ROOT)
print(json.dumps(manifest,indent=2,ensure_ascii=False))